In [ ]:
# configure iPython to automatically reload all external modules
# only needs to be executed once

%load_ext autoreload
%autoreload 2

## Imports and Init

In [ ]:
import os 
os.chdir("C:\\Users\\phys-pico-lab\\Documents\\Jupyter_Measurements")
print(os.getcwd())

In [1]:
%config IPCompleter.greedy=True

# convenience import for all QCCS software functionality
from laboneq.simple import *

# helper import - needed to extract qubit and readout parameters from measurement data
from lib.helpers.example_notebook_helper import *
from lib.helpers.meas_helper_mod import *
from lib.helpers.setup_helper import *
from lib.helpers.save_data_helper import *
from lib.helpers.create_meas_helper import *
#from helpers.pop_temp_helper import *
from lib.helpers.pop_temp_helper_v2 import *
from lib.helpers.fitting_helper import *
import lib.utils.calculator as calc

#import from python librarys
from scipy.io import savemat, loadmat
from scipy.stats import norm
from scipy.optimize import curve_fit
from scipy import fft
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os.path
import time
import json as js
# from scresonators.wrapper_functions import fit_single_STS_wrapper, plot_single_STS_wrapper
import pyvisa

#import instruments drivers
from blueftc.BlueforsController import BlueFTController
from lib.devices.bftc_credentials import BFTS_API_KEY, BFTS_IP, PORT_NUMBER, MXC_ID
from lib.devices.KeysightDMM34465A import KeysightDMM34465A
from lib.devices.SIM_wrapper import SIM900, SIM928
from lib.devices.YokoGS200_wrapper import YokoGS200

#imports for XLD server measurements
from lib.devices.XLD_Server_Client import XLDMeasClient
from time import sleep
from lib.devices.XLD_Server_Passkey import xld_ip

# from qmeas.qsample import QSample
# from qmeas.qparameters import QBaseParameters, QLinkedParameters

In [ ]:
from playsound import playsound

def meas_ready():
    if not mute:
        playsound(os.path.join('lib', 'utils', 'ding.mp3'))

mute = False

### Set up temperature controller

In [2]:
#Temperature controller parameters

import requests
from requests.packages.urllib3.exceptions import InsecureRequestWarning

requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

BFTCont = BlueFTController(ip=BFTS_IP, port=PORT_NUMBER, key=BFTS_API_KEY, 
                              mixing_chamber_channel_id=MXC_ID)

BFTCont.get_mxc_temperature()

2026-06-30 11:06:23,218 :-- ERROR --: Error: HTTPSConnectionPool(host='192.168.1.9', port=49098): Max retries exceeded with url: /values/mapper/heater_mappings_bftc/device/c6/temperature/?prettyprint=1&key=4f0283ae-3046-457a-bf87-3361f5e54833 (Caused by ConnectTimeoutError(<HTTPSConnection(host='192.168.1.9', port=49098) at 0x16591cad0>, 'Connection to 192.168.1.9 timed out. (connect timeout=None)'))
2026-06-30 11:06:23,223 :-- WARNING --: Could not verify synchronization status
2026-06-30 11:06:23,225 :-- WARNING --: Could not verify synchronization status


0.0

### Voltage source setup HERE FLUX FIXED AT -1.2V VIA 5.6kOhm

In [3]:
#Volt source parameters
Yoko_addr = 'TCPIP0::192.168.1.201::inst0::INSTR'
volt_source = YokoGS200(address=Yoko_addr)
R = 5.4e3 #Ohm

In [4]:
volt_source.set_range(10)

VisaIOError: VI_ERROR_RSRC_NFOUND (-1073807343): Insufficient location information or the requested device or resource is not present in the system.

In [ ]:
volt_source.get_range()

In [ ]:
# temp_dict = BFTCont.get_channel_temps_in_time(5, 57600)

# # print(temp_dict.keys())

# temps_ch_5 = pd.DataFrame({'time': temp_dict['timestamp'], 
#                            'temp': temp_dict['temperature']})
# temps_ch_5.to_csv('channel_5_temp_data_2024-07-11_23-00--2024-07-12_14-38.txt', index=False)

### Sample parameters

In [ ]:
cooldown_nr = 2

In [ ]:
#dictionary for sample parameters
sample_parameters = {
    'folder_name': r'N:\xld\Kubatkin\Data',
    'sample_name': r'S3',
    'structure_name': 'Q2'
}

qsample_params = QSample(directory=sample_parameters['folder_name'], 
                         sample=sample_parameters['sample_name'], 
                         structure=sample_parameters['structure_name'])

### Qubit and LO parameters

#### A: New parameters

In [5]:
# a collection of qubit control and readout parameters as a python dictionary
qubit_parameters = {
    'ro_freq' : 136e6, # 9.320e9,     # readout frequency of qubit 0 in [Hz] - relative to local oscillator for readout drive upconversion
    'ro_amp' : 0.5,              # readout amplitude
    'ro_amp_spec': 0.05,          # readout amplitude for spectroscopy 0.05
    'ro_len' : 2.0e-6,           # readout pulse length in [s]
    'ro_len_spec' : 4.0e-6,      # readout pulse length for resonator spectroscopy in [s]
    'ro_delay': 0.0,           # readout delay after last drive signal in [s]
    'ro_int_delay' : 330e-9,     # readout line offset calibration - delay between readout pulse and start of signal acquisition in [s]
    
    'th_res_freq': 0.0,
    
    'qb_freq': 50e6,            # qubit 0 drive frequency in [Hz] - relative to local oscillator for qubit drive upconversion
    'qb_anharm': 200e6,
    'qb_amp_spec': 0.0001,       # drive amplitude of qubit spectroscopy
    'qb_len_spec': 15e-6,        # drive pulse length for qubit spectroscopy in [s]
    'qb_len' : 6e-8,             # qubit drive pulse length in [s]
    'pi_amp' : 1.0,              # qubit drive amplitude for pi pulse
    'pi_half_amp' : 0.25,        # qubit drive amplitude for pi/2 pulse
    'qb_t1' : 6e-6,              # qubit T1 time
    'qb_t2_ramsey' : 1e-6,       # qubit T2 time
    'ramsey_det' : 2e6,       # qubit frequency detuning relative to qb_freq in [Hz]
    'qb_t2_echo'   : 1e-6,       # qubit T2 time
    'relax' : 500e-6             # delay time after each measurement for qubit reset in [s]
}

# qubit_parameters = QBaseParameters(sample=qsample_params,
#                                     name=f'qubit_parameters_cooldown_{cooldown_nr}',
#                                     parameters=qubit_parameters)

In [6]:
# up / downconversion settings - to convert between IF and RF frequencies 
# steps of 200Mhz
lo_settings = {
    'qb_lo': 3.8e9,              # qubit LO frequency in [Hz]
    'ro_lo': 6.2e9              # R1 readout LO frequency in [Hz]
}

# lo_settings = QBaseParameters(sample=qsample_params,
                            #  name=f'lo_settings_cooldown_{cooldown_nr}',
                            #  parameters=lo_settings)

#### B: Load parameters from file

In [ ]:
# #path to file with qubit parameters
# qp_file_path = r'N:\xld\Qubit Termometry\Data\S411-21\Q1\2025-04-07 Jupiter\Qubit_params_BACK15_41_28.txt'

# #path to file with lo settings
# lo_file_path = r'N:\xld\Qubit Termometry\Data\S351-23\QT\2023-12-14 Jupiter\LO_settings_dict15_19_08.txt'

# #qubit parameters loading
# with open(qp_file_path, 'r') as fp:
#     qubit_parameters = js.load(fp)
    
# #lo settings loading
# with open(lo_file_path, 'r') as fp:
#     lo_settings = js.load(fp)

previous_cooldown = 1

root_dir = os.path.join(sample_parameters['folder_name'], sample_parameters['sample_name'], sample_parameters['structure_name'], 'parameters')

with open(os.path.join(root_dir, f'lo_settings_cooldown_{previous_cooldown}.txt'), 'r') as f:
    pars = js.load(f)

    lo_settings = QBaseParameters(sample=qsample_params,
                            name=f'lo_settings_cooldown_{cooldown_nr}',
                            parameters=pars)

with open(os.path.join(root_dir, f'qubit_parameters_cooldown_{previous_cooldown}.txt'), 'r') as f:
    pars = js.load(f)

    qubit_parameters = QBaseParameters(sample=qsample_params,
                            name=f'qubit_parameters_cooldown_{cooldown_nr}',
                            parameters=pars)

# with open(os.path.join(root_dir, f'twpa_settings_cooldown_{previous_cooldown}.txt'), 'r') as f:
#     pars = js.load(f)

#     twpa_settings = QBaseParameters(sample=qsample_params,
#                             name=f'twpa_settings_cooldown_{this_cooldown}',
#                             parameters=pars)

### Zurich Instruments SFHQC Setup

In [7]:
# define the DeviceSetup from descriptor - additionally include information on the dataserver used to connect to the instruments 
my_setup = DeviceSetup.from_descriptor(
    my_descriptor,
    server_host="localhost",
    server_port="8004",
    setup_name="XLD",
) 

# define Calibration object based on qubit control and readout parameters
my_calibration = define_calibration(parameters=qubit_parameters, lo_settings=lo_settings)

# apply calibration to device setup
my_setup.set_calibration(my_calibration)

In [8]:
## define shortcut to logical signals for convenience
lsg_q0 = my_setup.logical_signal_groups["q0"].logical_signals
drive_Oscillator_q0 = lsg_q0["drive_line"].oscillator
drive_ef_Oscillator_q0 = lsg_q0["drive_ef_line"].oscillator
readout_Oscillator_q0 = lsg_q0["measure_line"].oscillator
acquire_Oscillator_q0 = lsg_q0["acquire_line"].oscillator

In [9]:
lsg_q0["acquire_line"].port_delay

3.3e-07

### Create and Connect to a LabOne Q Session

In [10]:
# perform experiments in emulation mode only? - if True, also generate dummy data for fitting
emulate = True

# create and connect to a session
my_session = Session(device_setup=my_setup)
my_session.connect(do_emulation=emulate, ignore_version_mismatch = True)

[2026.06.30 11:07:44.141] INFO    Logging initialized from [Default inline config in laboneq.laboneq_logging] logdir is /Users/khatran/Desktop/superconducting-qubit-thermometry/laboneq_output/log
[2026.06.30 11:07:44.145] INFO    VERSION: laboneq 26.4.0
[2026.06.30 11:07:44.146] INFO    Connecting to data server at localhost:8004
[2026.06.30 11:07:44.148] INFO    Connected to Zurich Instruments LabOne Data Server version 26.04.1.6 at localhost:8004
[2026.06.30 11:07:44.150] INFO    Configuring the device setup
[2026.06.30 11:07:44.157] INFO    The device setup is configured


In [11]:
lsg_q0["acquire_line"].range = -35

In [12]:
#Check ranges
print('Drive line range:', lsg_q0["drive_line"].range, 'dBm')
print('Measure line range:', lsg_q0["measure_line"].range, 'dBm')
print('Acquire line range:', lsg_q0["acquire_line"].range, 'dBm')

Drive line range: 5 dBm
Measure line range: -25 dBm
Acquire line range: -35 dBm


# Spectroscopy

## Pulsed Resonator Spectroscopy

In [13]:
# frequency range of spectroscopy scan - around expected centre frequency as defined in qubit parameters
spec_range = 10e6
# how many frequency points to measure
spec_num = 51

# define sweep parameters for two qubits
freq_sweep_ST = LinearSweepParameter(
    uid="res_freq",
    start=qubit_parameters["ro_freq"] - spec_range / 2,
    stop=qubit_parameters["ro_freq"] + spec_range / 2,
    count=spec_num,
)

# take how many averages per point: 2^n_average
n_average = 13

# spectroscopy excitation pulse
readout_pulse_spec = pulse_library.const(
    length=qubit_parameters["ro_len_spec"], amplitude=qubit_parameters["ro_amp_spec"]
)

In [ ]:
# define the experiment with the frequency sweep relevant for qubit 0
exp_spec = res_spectroscopy(freq_sweep_ST, readout_pulse_spec, n_average)

# set signal calibration and signal map for experiment to qubit 0
exp_spec.set_calibration(res_spec_calib(freq_sweep_ST))
exp_spec.set_signal_map(MA_map)

In [15]:
#compile experiment
exp_spec_comp = my_session.compile(exp_spec)

[2026.06.30 11:07:49.175] INFO    Starting LabOne Q Compiler run...
[2026.06.30 11:07:49.183] INFO    Schedule completed. [0.001 s]
[2026.06.30 11:07:49.198] INFO    Code generation completed for all AWGs. [0.015 s]
[2026.06.30 11:07:49.199] INFO    Completed compilation step 1 of 51. [0.017 s]
[2026.06.30 11:07:49.199] INFO    Skipping compilation for next step(s)...
[2026.06.30 11:07:49.208] INFO     ────────────────────────────────────────────────────────────────── 
[2026.06.30 11:07:49.208] INFO      Device         AWG   SeqC LOC   CT entries   Waveforms   Samples  
[2026.06.30 11:07:49.208] INFO     ────────────────────────────────────────────────────────────────── 
[2026.06.30 11:07:49.209] INFO      device_shfqc     0          8            0           1     16000  
[2026.06.30 11:07:49.209] INFO     ────────────────────────────────────────────────────────────────── 
[2026.06.30 11:07:49.209] INFO      TOTAL                       8            0                 16000  
[2026.06.30

In [16]:
# run the experiment on the open instrument session
ST_results = my_session.run(exp_spec_comp)

# meas_ready()

[2026.06.30 11:07:50.340] INFO    Starting near-time execution...
[2026.06.30 11:07:50.479] INFO    Finished near-time execution.


In [17]:
# get the measurement data returned by the instruments from the LabOne Q session
spec_res = ST_results.get_data("res_spec")

# define the frequency axis from the qubit parameters
spec_freq = lo_settings["ro_lo"] + ST_results.get_axis("res_spec")[0]

In [18]:
# # fit to asymmetric Lorentzian model 
full_result, quick_result, sweep_manager = fit_single_STS_wrapper(feqs=spec_freq, y_data=spec_res)
print(quick_result)

fig, ax = plot_single_STS_wrapper(sweep_man=sweep_manager)

#Change RO freq
qubit_parameters.update_parameter("ro_freq", quick_result['f0'] - lo_settings["ro_lo"])
# qubit_parameters["ro_freq"] = quick_res
print('RO Frequency:', qubit_parameters["ro_freq"]*1e-6, 'MHz')

NameError: name 'fit_single_STS_wrapper' is not defined

In [ ]:
print('Readout frequency from fit: ', qubit_parameters["ro_freq"]*1e-6, 'MHz')

## Pulsed Qubit Spectroscopy

### Pulsed measurements presettings

In [ ]:
#USUALLY DON'T NEED TO BE USED!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
#define shortcuts for lo settings

#lsg_q0 = my_setup.logical_signal_groups["q0"].logical_signals
#drive_Oscillator_q1 = lsg_q0["drive_line"].local_oscillator

In [ ]:
#manual settings of the readout and exitation pulse properties

#exitation:
qubit_parameters.update_parameter("qb_len_spec", 20e-6)
qubit_parameters.update_parameter("qb_amp_spec", 0.6)#<----------- Most important parameter for good Qubit Spectroscopy!!!

print('Qubit spectroscopy length:', qubit_parameters["qb_len_spec"]*1e6, 'mks')
print('Qubit spectroscopy amplitude:', qubit_parameters["qb_amp_spec"])

#readout
#qubit_parameters["ro_len"]=2e-6 #Not bigger that 2e-6!
#qubit_parameters["ro_amp"] = 0.2

print('Readout length:', qubit_parameters["ro_len"]*1e6, 'mks')
print('Readout amplitude:', qubit_parameters["ro_amp"])

#relaxation
# qubit_parameters.update_parameter('relax', 75e-6)
print('Relaxation time:', qubit_parameters['relax']*1e6, 'mks')

In [ ]:
#USUALLY DON'T NEED TO BE USED!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
#manual preset of the qubit frequency
#qubit_parameters["qb_freq"] = -100e6

In [ ]:
#PULSE SETTINGS
# square pulse to excite the qubit
# square_pulse = pulse_library.const(
#     uid="const_iq",
#     length=qubit_parameters["qb_len_spec"],
#     amplitude=qubit_parameters["qb_amp_spec"],
# )

square_pulse = pulse_library.gaussian(
    uid="const_gauss",
    length=qubit_parameters["qb_len_spec"],
    amplitude=qubit_parameters["qb_amp_spec"],
)

# qubit readout pulse - here simple constant pulse
readout_pulse = pulse_library.const(
    uid="readout_pulse",
    length=qubit_parameters["ro_len"],
    amplitude=qubit_parameters["ro_amp"],
)
# integration weights for qubit measurement - here simple constant weights, i.e. all parts of the return signal are weighted equally
readout_weighting_function = pulse_library.const(
    uid="readout_weighting_function", length=qubit_parameters["ro_len"], amplitude=1.0
)

In [ ]:
readout_spec = {'readout_type': 'spectroscopy',
                'readout_pulse': readout_pulse,
                'readout_weighting_function': readout_weighting_function,
                'relax_time': qubit_parameters["relax"],
                'measure_freq': qubit_parameters["ro_freq"],
                'acquire_freq': qubit_parameters["ro_freq"],
                'readout_range': -25,
                'readout_delay': qubit_parameters['ro_int_delay']
               }

In [ ]:
print('Qubit frequency before spectroscopy: ', qubit_parameters["qb_freq"]*1e-6, 'MHz')

In [ ]:
# frequency range of spectroscopy scan - defined around expected qubit frequency as defined in qubit parameters
qspec_range = 10e6
# how many frequency points to measure
qspec_num = 101

#Set sweep with span around qubit frequency
freq_sweep_TT = LinearSweepParameter(
    uid="freq-qubit",
    start=qubit_parameters["qb_freq"] - qspec_range / 2,
    stop=qubit_parameters["qb_freq"] +  qspec_range / 2,
    count=qspec_num,
)

# #Set sweep in fixed range
# freq_sweep_TT = LinearSweepParameter(
#    uid="freq-qubit",
#    start=0e6,
#    stop=200e6,
#    count=qspec_num,
# )

# how many averages per point: 2^n_average
n_average = 12

In [ ]:
# qubit_parameters.update_parameter("qb_freq", 4.327e9 - lo_settings["qb_lo"]) 132.81175270395565
# qubit_parameters.update_parameter("qb_freq", 132.81175270395565e6)

In [ ]:
## configure oscillator settings for readout and acquisition - defined as calibration on device setup as persistent baseline settings

# readout pulse frequency resonant with the readout resonator
readout_Oscillator_q0.frequency = qubit_parameters["ro_freq"]
# change oscillator type to software to enable multiplexed qubit readout
readout_Oscillator_q0.modulation_type = ModulationType.SOFTWARE
# demodulation frequency same as readout pulse frequency
acquire_Oscillator_q0.frequency = qubit_parameters["ro_freq"]
acquire_Oscillator_q0.modulation_type = ModulationType.SOFTWARE

print('Readout frequency set to: ', qubit_parameters["ro_freq"]*1e-6, 'MHz')

In [ ]:
#Uncalibrated spectroscopy readout
exp_qspec = create_qubit_spec(freq_sweep_TT, square_pulse, readout_spec, n_average)

#Calibrated low-power readout
#exp_qspec = create_qubit_spec(freq_sweep_TT, square_pulse, readout_low, n_average)

In [ ]:
#compile experiment

exp_qspec_comp = my_session.compile(exp_qspec)
#show_pulse_sheet("Q spec", my_session.compiled_experiment)

In [ ]:
# run the experiment
qspec_results = my_session.run(exp_qspec_comp)

#meas_ready()

In [ ]:
# get measurement data returned by the instruments
qspec_res = qspec_results.get_data("q0_spec")

# define a frequency axis from the parameters
qspec_freq = lo_settings["qb_lo"] + qspec_results.get_axis("q0_spec")[0]
#qspec_freq = drive_Oscillator_q1.frequency + my_results.get_axis("q0_spec")[0]

In [ ]:
# plot measurement data
fig, (ax1, ax2) = plt.subplots(2, 1, sharex = True, figsize=(12,8))
ax1.plot(qspec_freq / 1e9, abs(qspec_res), ".k", label = 'amp'),
ax2.plot(qspec_freq / 1e9, np.unwrap(np.angle(qspec_res)), ".k", label = 'pha')
ax1.set_ylabel("Amplitude, a.u.", fontsize = 14)
ax2.set_ylabel("Phase, rad", fontsize = 14)
ax2.set_xlabel("Frequency, GHz", fontsize = 14)
plt.suptitle('Pulse qubit spectroscopy', y = 0.92, fontsize = 14)

# increase number of plot points for smooth plotting of fit reults
freq_plot = np.linspace(qspec_freq[0], qspec_freq[-1], 5 * len(qspec_freq))

guess_freq = 3.8478e9

manual_freq = 3.8478e9
# ax1.vlines([manual_freq / 1e9], abs(qspec_res).min(), abs(qspec_res).max())
# ax2.vlines([manual_freq / 1e9], np.unwrap(np.angle(qspec_res)).min(), np.unwrap(np.angle(qspec_res)).max())

# fit measurement data - here assuming an inverted Lorentzian response
popt_pha, pcov_pha = fit_3DSpec(
   qspec_freq,
   #abs(qspec_res),
   np.unwrap(np.angle(qspec_res)),
   1e6,
   guess_freq,#qubit_parameters["qb0_freq"] + lo_settings["qb_lo"],
   10e3,
   off=2.0,
   plot=False,
   #bounds=[[5e5, 5.24e9, 0, 0.02], [3e6, 5.280e9, 3.0e5, 0.06]],
   #bounds=[[10e4, 5.210e9, 0, 0], [10e6, 5.240e9, 1e8, 2]],
)

# fit measurement data - here assuming an inverted Lorentzian response
popt_amp, pcov_amp = fit_3DSpec(
    qspec_freq,
    abs(qspec_res),
    #np.unwrap(np.angle(qspec_res)),
    5e5,
    guess_freq,#qubit_parameters["qb0_freq"] + lo_settings["qb_lo"],
    10.0e3,
    off=0.28,
    plot=False,
    #bounds=[[5e5, 5.24e9, 0, 0.02], [3e6, 5.280e9, 3.0e5, 0.06]],
    #bounds=[[10e4, 5.210e9, 0, 0], [10e6, 5.240e9, 1e8, 2]],
)

# plot fit results together with measurement data
ax1.plot(freq_plot / 1e9, func_lorentz(freq_plot, *popt_amp), "-r")
ax2.plot(freq_plot / 1e9, func_lorentz(freq_plot, *popt_pha), "-r")

plt.legend()

#save figure
# figname = 'Pulsed_TTS_2fig_'
# file_path = get_path_to_file(figname, '.png', sample_parameters)
# plt.savefig(file_path, dpi=600, format='png', metadata=None,
#         bbox_inches=None, pad_inches=0.1,
#         facecolor='auto', edgecolor='auto',
#         backend=None,
#        )

#print('Qubit frequency from fit:', round(popt[1]*1e-9, 3), 'GHz')

# update qubit parameters
qubit_parameters.update_parameter("qb_freq", popt_pha[1] - lo_settings["qb_lo"])
# qubit_parameters.update_parameter("qb_freq", popt_amp[1] - lo_settings["qb_lo"])
#qubit_parameters["qb1_freq"] = popt[1] - drive_Oscillator_q1.frequency
# qubit_parameters.update_parameter("qb_freq", manual_freq - lo_settings["qb_lo"])
print('Qubit parameter qb_freq:', qubit_parameters["qb_freq"]*1e-6, 'MHz')

In [ ]:
popt_pha[1]

In [ ]:
popt_amp[1]-4.2e9

In [ ]:
#qubit_parameters['qb_anharm'] = -(popt_amp[1] - lo_settings["qb_lo"]-qubit_parameters["qb_freq"])*2
#print('Qubit anharmonism: ', qubit_parameters['qb_anharm']*1e-6, 'MHz')

In [ ]:
Data_TT = {'qspec_freq': qspec_freq,
            'qspec_res': qspec_res
            }

Data_TT.update(qubit_parameters._params)
Data_TT.update(lo_settings._params)

file_name = 'TTS_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_TT)

In [ ]:
qubit_parameters

### TTS with flux

In [ ]:
#Qubit spectroscopy
qspec_res_list = []
TTS_popt_list = []

In [ ]:
qubit_parameters['flux_lower_ss'] = -1.9e-05

In [ ]:
print(qubit_parameters['flux_lower_ss'])

In [ ]:
flux_sweep = np.linspace(-0.002, 0.002, 25)*1e-3
flux_points = qubit_parameters['flux_lower_ss']+flux_sweep
print(flux_points)

In [ ]:
for i in range(len(flux_sweep)):
    print('Measurement number:', i)
    volt_source.ramp(flux_points[i]*R, 10)
    print('Flux:', flux_points[i])
    
    # #Make the TTS
    my_results_TTS = my_session.run(exp_qspec_comp)
        
    qspec_res = my_results_TTS.get_data("q0_spec")
    qspec_freq = lo_settings["qb_lo"] + my_results_TTS.get_axis("q0_spec")[0]
        
    qspec_res_list.append(qspec_res)

    popt_pha, pcov_pha = fit_3DSpec(
           qspec_freq,
           #abs(qspec_res),
           np.unwrap(np.angle(qspec_res)),
           1e6,
           4.6e9,#qubit_parameters["qb0_freq"] + lo_settings["qb_lo"],
           10e3,
           off=1.84,
           plot=True,
           #bounds=[[5e5, 5.24e9, 0, 0.02], [3e6, 5.280e9, 3.0e5, 0.06]],
           #bounds=[[10e4, 5.210e9, 0, 0], [10e6, 5.240e9, 1e8, 2]],
    )

    TTS_popt_list.append(popt_pha)

In [ ]:
qspec_res_arr = np.array(qspec_res_list)
TTS_popt_arr = np.array(TTS_popt_list)

In [ ]:
TTS_popt_arr.shape

In [ ]:
plt.plot(flux_points, TTS_popt_arr[:,1])

In [ ]:
qubit_parameters['flux_lower_ss'] = -1.85e-05
volt_source.ramp(qubit_parameters['flux_lower_ss']*R, 10)

### ef-transition spectroscopy

In [ ]:
# qubit_parameters.update_parameter('qb_anharm', 220e6)
#qubit_parameters["ro_len"]=2e-6
#qubit_parameters['qb_ef_amp_spec']=0.2
# qubit_parameters.update_parameter('qb_ef_amp_spec', 0.2)
#qubit_parameters['qb4_freq']=22e6

In [ ]:
#manual settings of the readout and exitation pulse properties

#exitation:
qubit_parameters.update_parameter("qb_len_spec", 20e-6)
qubit_parameters.update_parameter("qb_ef_amp_spec", 0.8)#<----------- Most important parameter for good Qubit Spectroscopy!!!

print('Qubit spectroscopy length:', qubit_parameters["qb_len_spec"]*1e6, 'mks')
print('Qubit spectroscopy ef amplitude:', qubit_parameters["qb_ef_amp_spec"])

In [ ]:
#qubit_parameters["qb_freq"]*1e-6


In [ ]:
qubit_parameters['qb_anharm']

In [ ]:
#qubit_parameters["qb_freq"]+3.8e9-3.6344e9

In [ ]:
#qubit_parameters.update_parameter('qb_anharm', 213307499.87887526)

In [ ]:
# frequency range of spectroscopy scan - defined around expected qubit frequency as defined in qubit parameters
qspec_range = 20e6
# how many frequency points to measure
qspec_num = 501

# set up sweep parameters - qubit drive frequency
freq_sweep_ef = LinearSweepParameter(
    uid="freq-ef-qubit",
    start=qubit_parameters["qb_freq"] - qubit_parameters['qb_anharm'] - qspec_range / 2,
    stop=qubit_parameters["qb_freq"]  - qubit_parameters['qb_anharm'] +  qspec_range / 2,
    count=qspec_num,
)

# how many averages per point: 2^n_average
n_average = 14

# square pulse to excite the qubit
square_pulse = pulse_library.const(
    uid="const_iq",
    length=qubit_parameters["qb_len_spec"],
    amplitude=qubit_parameters["qb_ef_amp_spec"],
)

In [ ]:
# exp_ef_qspec = create_qubit_ef_spec(freq_sweep_ef, 
#                                     square_pulse, 
#                                     readout_spec, 
#                                     n_average)

exp_ef_qspec = create_qubit_ef_spec_prep(freq_sweep_ef, 
                                    x180,
                                    square_pulse, 
                                    readout_low, 
                                    n_average)

In [ ]:
#compile the experiment
exp_ef_qspec_comp = my_session.compile(exp_ef_qspec)
#show_pulse_sheet("Q spec", my_session.compiled_experiment)

In [ ]:
# run the experiment on qubit 0
ef_qspec_results = my_session.run(exp_ef_qspec_comp)

In [ ]:
# get measurement data returned by the instruments
ef_qspec_res = ef_qspec_results.get_data("q0_ef_spec")

# define a frequency axis from the parameters
#ef_qspec_freq = drive_Oscillator_q1.frequency + my_results.get_axis("q0_ef_spec")[0]
ef_qspec_freq = lo_settings["qb_lo"] + ef_qspec_results.get_axis("q0_ef_spec")[0]

In [ ]:
# plot measurement data
#%matplotlib inline
fig = plt.figure()
# plt.plot(ef_qspec_freq / 1e9, abs(ef_qspec_res), ".k")
plt.plot(ef_qspec_freq / 1e9, np.unwrap(np.angle(ef_qspec_res)), ".k")
plt.ylabel("A (a.u.)")
plt.xlabel("Frequency (GHz)")
plt.title('Pulse qubit ef-spectroscopy')

#xlim = [4.997, 5.003]
#plt.xlim(xlim)
#ylim = [0.032, 0.034]
#plt.ylim(ylim)

# increase number of plot points for smooth plotting of fit reults
freq_plot = np.linspace(ef_qspec_freq[0], ef_qspec_freq[-1], 5 * len(ef_qspec_freq))

# fit measurement data - here assuming an inverted Lorentzian response
popt, pcov = fit_3DSpec(
    ef_qspec_freq,
    #abs(ef_qspec_res),
    np.unwrap(np.angle(ef_qspec_res)),
    1e6,
    3.6345e9,#qubit_parameters["qb0_freq"] + lo_settings["qb_lo"],
    10,
    off=2.0,
    plot=False,
    #bounds=[[5e3, 6.2e9, 0, 0], [5e6, 6.3e9, 5.0e4, 0.06]],
    #bounds=[[10e4, 5.210e9, 0, 0], [10e6, 5.240e9, 1e8, 2]],
)
print(popt)
#%matplotlib QT
# plot fit results together with measurement data
plt.plot(freq_plot / 1e9, func_lorentz(freq_plot, *popt), "-r")
# update qubit parameters (uncomment only if fit converged!)
# qubit_parameters.update_parameter('qb_anharm', -(popt[1] - lo_settings["qb_lo"]-qubit_parameters["qb_freq"]))
#qubit_parameters['qb_anharm'] = -(popt[1] - lo_settings["qb_lo"]-qubit_parameters["qb_freq"])*2
# qubit_parameters.update_parameter('qb_anharm', -(3.635e9 - lo_settings["qb_lo"] - qubit_parameters["qb_freq"]))

#qubit_parameters['qb_anharm'] = popt[1] - qubit_parameters["qb1_freq"]
# qubit_parameters["qb4_freq"] = 3.636e9 - lo_settings["qb_lo"]
# print(qubit_parameters["qb4_freq"])
print('Qubit anharmonism: ', qubit_parameters['qb_anharm']*1e-6, 'MHz')

In [ ]:
#For manual setting of anharmonicity
#qubit_parameters['qb_anharm'] = 100e6

In [ ]:
#qubit_parameters["qb4_freq"] = popt[1] - lo_settings["qb_lo"]
qubit_parameters["qb4_freq"]-qubit_parameters["qb_freq"]

In [ ]:
qubit_parameters["qb4_freq"]

In [ ]:
3.8-3.63438


In [ ]:
Data_TT = {'qspec_freq': ef_qspec_freq,
            'qspec_res': ef_qspec_res
            }

Data_TT.update(qubit_parameters._params)
Data_TT.update(lo_settings._params)

file_name = 'ef_TTS_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_TT)

# Control Pulse Setup

## First Transition: g-e

### Amplitude Rabi Experiment

In [ ]:
# qubit drive frequency - defined in calibration on device setup as baseline reference
drive_Oscillator_q0.frequency = qubit_parameters["qb_freq"]
# set oscillator type to hardware to ensure optimal use of the instrument functionality
drive_Oscillator_q0.modulation_type = ModulationType.HARDWARE

In [ ]:
#Set power ranges for drive, measure and aquire
###USUALLY DON'T NEED TO BE USED!!!!!!

####Drive#### Usualy 5 dBm
lsg_q0["drive_line"].range = 10

####Measure###
#lsg_q0["measure_line"].range = -25
# lsg_q0["measure_line"].range = -10 #High Power
lsg_q0["measure_line"].range = -20 #Low Power

####Acquire### Usualy -30
lsg_q0["acquire_line"].range = -35

In [ ]:
#Check ranges
print('Drive line range:', lsg_q0["drive_line"].range, 'dBm')
print('Measure line range:', lsg_q0["measure_line"].range, 'dBm')
print('Acquire line range:', lsg_q0["acquire_line"].range, 'dBm')

In [ ]:
range_params = QBaseParameters(sample=qsample_params,
                                    name=f'range_paramaters_cooldown_{cooldown_nr}',
                                    parameters={'drive': lsg_q0["drive_line"].range,
                                                'drive_ef': lsg_q0["drive_ef_line"].range,
                                                'measure': lsg_q0["measure_line"].range,
                                                'acquire': lsg_q0["acquire_line"].range})

range_params

In [ ]:
#Check frequencies
print('Readout frequency:', qubit_parameters["ro_freq"]*1e-6, 'MHz')
print('Qubit frequency:', qubit_parameters["qb_freq"]*1e-6, 'MHz')
# qubit_parameters.update_parameter("qb_freq", 599.5593508373827 * 1e6)

In [ ]:
#Set readout frequency

# readout pulse frequency resonant with the readout resonator
readout_Oscillator_q0.frequency = qubit_parameters["ro_freq"]
#readout_Oscillator_q0.frequency = qubit_parameters["ro_bare"]
# change oscillator type to software to enable multiplexed qubit readout
readout_Oscillator_q0.modulation_type = ModulationType.SOFTWARE

# demodulation frequency same as readout pulse frequency
acquire_Oscillator_q0.frequency = qubit_parameters["ro_freq"]
#acquire_Oscillator_q0.frequency = qubit_parameters["ro_bare"]
acquire_Oscillator_q0.modulation_type = ModulationType.SOFTWARE

In [ ]:
#change lenght of the qubit exitation pulse
qubit_parameters.update_parameter('qb_len', 300e-9)
#qubit_parameters.add_parameter('qb._len', 50e-9)

print('Qubit exitation pulse length: ', qubit_parameters["qb_len"]*1e9, 'ns')

In [ ]:
# qubit_parameters["relax"] = 7e-5
# qubit_parameters.update_parameter('relax',70e-6)

In [ ]:
readout_low = {'readout_type': 'pulse',
                'readout_pulse': readout_pulse,
                'readout_weighting_function': readout_weighting_function,
                'relax_time': qubit_parameters["relax"],
                'measure_freq': qubit_parameters["ro_freq"],
                'acquire_freq': qubit_parameters["ro_freq"],
                'readout_range': lsg_q0["measure_line"].range,
                'readout_delay': qubit_parameters['ro_int_delay']
               }

In [ ]:
readout_low

In [ ]:
# range of pulse amplitude scan
amp_min = 0
amp_max = 1.0
amp_num = 31

# set up sweep parameter - qubit drive pulse amplitude
rabi_sweep = LinearSweepParameter(
    uid="rabi_amp", start=amp_min, stop=amp_max, count=amp_num
)

# how many averages per point: 2^n_average
n_average = 11

# Rabi excitation pulse - gaussian of unit amplitude - amplitude will be scaled with sweep parameter in experiment
gaussian_pulse = pulse_library.gaussian(
    uid="gaussian_pulse", length=qubit_parameters["qb_len"], amplitude=1.0
)

In [ ]:
exp_rabi = create_rabi(rabi_sweep, 
                       gaussian_pulse, 
                       readout_low, #readout_opt, #readout_low,
                       n_average)

exp_rabi_comp = my_session.compile(exp_rabi)

In [ ]:
# run the experiment on qubit 0
my_results = my_session.run(exp_rabi_comp)

meas_ready()

In [ ]:
# get measurement data returned by the instruments
rabi_res = my_results.get_data("q0_rabi")

# define amplitude axis from qubit parameters
rabi_amp = my_results.get_axis("q0_rabi")[0]

In [ ]:
signal = rabi_res

fig, ax = plt.subplots(2, 2, sharex = True, figsize = (10, 8))

fig.suptitle('Rabi oscillations vs amplitude, $\Delta t_q$ = ' + '{:.0f}'.format(qubit_parameters["qb_len"]*1e9) + 'ns, '+'$N_{av}$ = ' + '{:.0f}'.format(n_average), fontsize=16)

fig.supxlabel("Amplitude (a.u.)")
fig.supylabel("Amplitude (a.u.)")

ax[0,0].plot(rabi_amp, abs(signal), ".k", label = 'amp')
ax[1,0].plot(rabi_amp, np.unwrap(np.angle(signal)), ".k", label = 'phase')
ax[0,1].plot(rabi_amp, np.real(signal), ".k", label = 'real')
ax[1,1].plot(rabi_amp, np.imag(signal), ".k", label = 'imag')

ax[0,0].set_title('amplitude')
ax[1,0].set_title('phase')
ax[0,1].set_title('real part')
ax[1,1].set_title('imaginary part')

amp_plot = np.linspace(rabi_amp[0], rabi_amp[-1], 5 * len(rabi_amp))

popt_amp, pcov_amp = fit_Rabi(rabi_amp, abs(signal), 10, 1, 1, 0.048, plot=False)
popt_pha, pcov_pha = fit_Rabi(rabi_amp, np.unwrap(np.angle(signal)), 10, 1, 1, 0.048, plot=False)
popt_real, pcov_real = fit_Rabi(rabi_amp, np.real(signal), 10, 1, 1, 0.048, plot=False)
popt_imag, pcov_imag = fit_Rabi(rabi_amp, np.imag(signal), 10, 1, 1, 0.048, plot=False)

ax[0,0].plot(amp_plot, func_osc(amp_plot, *popt_amp), "-r")
ax[1,0].plot(amp_plot, func_osc(amp_plot, *popt_pha), "-r")
ax[0,1].plot(amp_plot, func_osc(amp_plot, *popt_real), "-r")
ax[1,1].plot(amp_plot, func_osc(amp_plot, *popt_imag), "-r")

print('fitting results for x180-pulse amplitude:')
print('amp   --- {:4.4f} +/- {:4.4f}'.format(np.pi/popt_amp[0], np.sqrt(np.diag(pcov_amp))[0]))
print('pha   --- {:4.4f} +/- {:4.4f}'.format(np.pi/popt_pha[0], np.sqrt(np.diag(pcov_pha))[0]))
print('real  --- {:4.4f} +/- {:4.4f}'.format(np.pi/popt_real[0], np.sqrt(np.diag(pcov_real))[0]))
print('imag  --- {:4.4f} +/- {:4.4f}'.format(np.pi/popt_imag[0], np.sqrt(np.diag(pcov_imag))[0]))

#qubit_parameters["pi_amp"] = np.pi/popt_pha[0]
#qubit_parameters["pihalf_amp"] = np.pi / 2 / (popt_pha[0])

#save figure
figname = 'Rabi_osc_'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

In [ ]:
signal = rabi_res

fig, ax = plt.subplots(3, 1, sharex = True, figsize = (10, 8))

I_zero = np.real(signal)[0]
Q_zero = np.imag(signal)[0]

new_signal = np.sqrt((np.real(signal) - I_zero)**2 + (np.imag(signal) - Q_zero)**2)

fig.suptitle(r'Rabi oscillations vs amplitude, $\Delta t_q$ = ' + '{:.0f}'.format(qubit_parameters["qb_len"]*1e9) + 'ns, '+'$N_{av}$ = ' + '{:.0f}'.format(n_average), fontsize=16)

fig.supxlabel("Amplitude (a.u.)")
fig.supylabel("Amplitude (a.u.)")

ax[0].plot(rabi_amp, np.real(signal), ".k")
ax[1].plot(rabi_amp, np.imag(signal), ".k")
ax[2].plot(rabi_amp, new_signal, ".k")

ax[0].set_title('real')
ax[1].set_title('imag')
ax[2].set_title('norm')

amp_plot = np.linspace(rabi_amp[0], rabi_amp[-1], 5 * len(rabi_amp))

popt_real, pcov_real = fit_Rabi(rabi_amp, np.real(signal), 10, 1, 1, 0.048, plot=False)
popt_imag, pcov_imag = fit_Rabi(rabi_amp, np.imag(signal), 10, 1, 1, 0.048, plot=False)
popt_norm, pcov_norm = fit_Rabi(rabi_amp, new_signal, 10, 1, 1, 0.01, plot=False)

ax[0].plot(amp_plot, func_osc(amp_plot, *popt_real), "-r")
ax[1].plot(amp_plot, func_osc(amp_plot, *popt_imag), "-r")
ax[2].plot(amp_plot, func_osc(amp_plot, *popt_norm), "-r")

print(f"Current pi pulse amplitude: \n{qubit_parameters['pi_amp']}")
print('fitting results for x180-pulse amplitude:')
print('real  --- {:4.4f} +/- {:4.4f}'.format(np.pi/popt_real[0], np.sqrt(np.diag(pcov_real))[0]))
print('imag  --- {:4.4f} +/- {:4.4f}'.format(np.pi/popt_imag[0], np.sqrt(np.diag(pcov_imag))[0]))
print('norm  --- {:4.4f} +/- {:4.4f}'.format(np.pi/popt_norm[0], np.sqrt(np.diag(pcov_norm))[0]))

#save figure
figname = 'Rabi_osc_norm_'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

In [ ]:
qubit_parameters.update_parameter("pi_amp", np.pi / (popt_norm[0]))
try:
    qubit_parameters.update_parameter("pihalf_amp", np.pi / 2 / (popt_norm[0]))
except Exception as ex:
    print(ex)
    qubit_parameters.add_parameter("pihalf_amp", np.pi / 2 / (popt_norm[0]))
    print('Added new parameter')

In [ ]:
#plot data on IQ plane
plt.title(r'Rabi oscillations on IQ, $\Delta t_q$ = ' + '{:.0f}'.format(qubit_parameters["qb_len"]*1e9) + 'ns, '+'$N_{av}$ = ' + '{:.0f}'.format(n_average), fontsize=16)

plt.plot(np.real(signal), np.imag(signal), ".k")

popt, pcov = fit_linear(np.real(signal), np.imag(signal), plot=False)

plt.plot(np.real(signal), func_lin(np.real(signal),*popt), 'r')

plt.xlabel('Real, a.u')
plt.ylabel('Imag, a.u')

In [ ]:
Data_rabi = {'rabi_res': rabi_res,
            'rabi_amp': rabi_amp
            }

Data_rabi.update(qubit_parameters._params)
Data_rabi.update(lo_settings._params)

file_name = 'Rabi_ampl_rough_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_rabi)

### Set/Update x90 and x180 pulses

In [ ]:
print('Pi amplitude:', qubit_parameters["pi_amp"])
print('Pi length:', qubit_parameters["qb_len"])

In [ ]:
x180 = pulse_library.gaussian(
    uid="x180", length=qubit_parameters["qb_len"], amplitude=qubit_parameters["pi_amp"]
)

x180_drag = pulse_library.drag(
    uid="x180_drag", length=qubit_parameters["qb_len"], amplitude=qubit_parameters["pi_amp"], sigma=0.3, beta=0.2
)

x90 = pulse_library.gaussian(
    uid="x90",
    length=qubit_parameters["qb_len"],
    amplitude=qubit_parameters["pihalf_amp"],
)

### Rabi Shevron

In [ ]:
results_list = []

detuning_shev = np.linspace(-5, 5, 41)*1e6

In [ ]:
for i in range(len(detuning_shev)):
    print('Measurement number:', i)
    
    # qubit drive frequency - defined in calibration on device setup as baseline reference
    drive_Oscillator_q0.frequency = qubit_parameters["qb_freq"]+detuning_shev[i]
    # set oscillator type to hardware to ensure optimal use of the instrument functionality
    drive_Oscillator_q0.modulation_type = ModulationType.HARDWARE
    
    my_results = my_session.run(exp_rabi)
    results_list.append(my_results.get_data("q0_rabi"))

In [ ]:
rabi_shev_arr = np.array(results_list)

In [ ]:
X = rabi_amp
Y = detuning_shev*1e-6
#Z = np.unwrap(np.angle(rabi_shev_arr))
Z = np.absolute(rabi_shev_arr)
calc.plot_2d(Y, X,Z, flip = False, cmap = 'Reds')
plt.title('Rabi shevron, ABS, $\Delta t_q$ = ' + '{:.0f}'.format(qubit_parameters["qb_len"]*1e9) + 'ns, '+'$N_{av}$ = ' + '{:.0f}'.format(n_average), fontsize=16)
plt.xlabel('Detuning, MHz')
plt.ylabel('Rabi Amplitude')

#save figure
figname = 'Rabi_shevron_'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

In [ ]:
Data_rabi = {'rabi_shev': rabi_shev_arr,
               'detuning': detuning_shev,
               'rabi_amp': rabi_amp,
               }

Data_rabi.update(qubit_parameters._params)
Data_rabi.update(lo_settings._params)

file_name = 'Rabi_shev_ge_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_rabi)

In [ ]:
#Restore proper qubit frequency!

drive_Oscillator_q0.frequency = qubit_parameters["qb_freq"]
# set oscillator type to hardware to ensure optimal use of the instrument functionality
drive_Oscillator_q0.modulation_type = ModulationType.HARDWARE

### Rabi Frequency Calibration

In [ ]:
#Settings for rabi frequency calibration
# range of pulse amplitude scan
amp_min = 0
amp_max = 1.0 #min([qubit_parameters["pi_amp"] * 2.2, 1.0])
# how many amplitude points to measure
amp_num = 101

# set up sweep parameter - qubit drive pulse amplitude
rabi_sweep = LinearSweepParameter(
    uid="rabi_amp", start=amp_min, stop=amp_max, count=amp_num
)

# how many averages per point: 2^n_average
n_average = 12

# # Rabi excitation pulse - gaussian of unit amplitude - amplitude will be scaled with sweep parameter in experiment
# gaussian_pulse = pulse_library.gaussian(
#     uid="gaussian_pulse", length=20e-9, amplitude=1.0
# )

In [ ]:
rabi_length_arr = np.linspace(50, 400, 36)*1e-9
print(rabi_length_arr)

In [ ]:
freq_test_list= [10]

rabi_list = []
rabi_popt_list = []
rabi_pcov_list = []

for i in range(len(rabi_length_arr)):
    print('Length: ', rabi_length_arr[i]*1e9, 'ns')
    gaussian_pulse_sw = pulse_library.gaussian(
        uid="gaussian_pulse", length=rabi_length_arr[i], amplitude=1.0
    )
    
    exp_rabi_sw = make_rabi(rabi_sweep, 
                         gaussian_pulse_sw, 
                         readout_pulse, 
                         readout_weighting_function, 
                         qubit_parameters["relax"], 
                         n_average)
    
    exp_rabi_sw.set_signal_map(qubit_meas_map)
    # compile
    exp_rabi_sw_comp = my_session.compile(exp_rabi_sw)
    
    rabi_sw_results = my_session.run(exp_rabi_sw_comp)
    
    # get measurement data returned by the instruments
    rabi_res = rabi_sw_results.get_data("q0_rabi")
    rabi_amp = rabi_sw_results.get_axis("q0_rabi")[0]
    
    I_zero = np.real(rabi_res)[0]
    Q_zero = np.imag(rabi_res)[0]
    
    norm_signal = np.sqrt((np.real(rabi_res) - I_zero)**2 + (np.imag(rabi_res) - Q_zero)**2)
    

    popt_norm, pcov_norm = fit_Rabi(rabi_amp, norm_signal, freq_test_list[-1], 1.0, 0.1, 0.048, plot=True)
    freq_test_list.append(popt_norm[0])
    
    rabi_list.append(rabi_res)
    rabi_popt_list.append(popt_norm)
    rabi_pcov_list.append(pcov_norm)
    
rabi_arr = np.array(rabi_list)
rabi_popt_arr = np.array(rabi_popt_list)
rabi_pcov_arr = np.array(rabi_pcov_list)

In [ ]:
X = rabi_amp
Y = rabi_length_arr
#Z = np.unwrap(np.angle(rabi_shev_arr))
Z = np.absolute(rabi_arr)
calc.plot_2d(Y, X,Z, flip = False, cmap = 'Reds')
plt.xlabel('Pulse_length')
plt.ylabel('Rabi_amplitude')

In [ ]:
#x = 1e-6*np.pi / rabi_length_arr
x = 1e-6/ rabi_length_arr
y = np.pi / (rabi_popt_arr[:,0])

popt, pcov = fit_linear(x, y, plot=False)

plt.plot(x, y, '.k')
plt.plot(x, func_lin(x,*popt), 'r')

plt.xlabel('Rabi frequency, MHz')
plt.ylabel('Rabi amplitude')

print('Slope: ', popt[0])
print('Intercept: ', popt[1])

qubit_parameters['rabi_slope'] = popt[0]
qubit_parameters['rabi_intercept'] = popt[1]

In [ ]:
Data_rabi_calib = {'rabi_length':rabi_length_arr,
                   'rabi_data': rabi_arr, 
                   'rabi_amp': rabi_amp,
                   'rabi_popt': rabi_popt_arr,
                   'rabi_pcov': rabi_pcov_arr,
                  }

Data_rabi_calib.update(qubit_parameters)
Data_rabi_calib.update(lo_settings)

file_name = 'Rabi_calib_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_rabi_calib)

### T1 Experiment

In [ ]:
# qubit_parameters.update_parameter('relax', 500e-6)

In [ ]:
# delay range for ramsey pulses
t1_min = 0.0
t1_max = 100e-6 #5 * qubit_parameters["qb_t1"]
# how many delay points to sweep
t1_num = 21

# set up sweep parameter
#t1_sweep = LinearSweepParameter(uid="t1_delay", start=t1_min, stop=t1_max, count=t1_num)

#log sweep
def log_sweep_help(t1_min, t1_max, t1_num):
    t1_log_sweep = np.logspace(np.log10(t1_max/t1_num/10), np.log10(t1_max), t1_num-1)
    t1_log_sweep_arr = np.append([0.0], t1_log_sweep)
    return SweepParameter(values=t1_log_sweep_arr)

t1_sweep = log_sweep_help(t1_min, t1_max, t1_num)

# how many averages per point: 2^n_average
n_average = 12

In [ ]:
exp_t1 = create_t1(t1_sweep, 
                   x180, 
                   readout_low,#readout_opt
                   n_average)

exp_t1_comp = my_session.compile(exp_t1)

In [ ]:
# run the experiment on qubit 0
t1_results = my_session.run(exp_t1_comp)

meas_ready()

In [ ]:
# get measurement data returned by the instruments
t1_res = t1_results.get_data("q0_t1")

# define time axis from qubit parameters
t1_delay = t1_results.get_axis("q0_t1")[0]

In [ ]:
#fit the data
popt_sl, pcov_sl = auto_T1_fit(t1_delay, t1_res, data_type = 'rot', plot = True)

#Transform fit parameters to proper formatting for one slice measurements
popt = popt_sl[0,:]
pcov = pcov_sl[0,:,:]

plt.title('Decay, rotated data')
plt.figtext(0.6, 0.5, r'$T_1$ = '+'{:.2f}'.format(1/popt[0]*1e6)+r'$\mu$s')

# update qubit parameters - here relaxation time / qubit lifetime
qubit_parameters.update_parameter("qb_t1", 1 / popt[0])

# T1 error in us
err = np.sqrt(np.diag(pcov))
print(f'T1 = {qubit_parameters["qb_t1"]*1e6:.3f} +- {(err[0]/(popt[0]*popt[0])*1e6):.3f} us')

#save figure
figname = 'T1'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

In [ ]:
Data_T1 = {'t1_res': t1_res,
            't1_delay': t1_delay
            }

Data_T1.update(qubit_parameters._params)
Data_T1.update(lo_settings._params)

file_name = 'T1_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_T1)

In [ ]:
#file_name = 'Qubit_params_dict_lower_ss_after_T1'
#file_path = get_path_to_file(file_name, '.txt', sample_parameters = sample_parameters)
#with open(file_path, 'w') as fp:
    #js.dump(qubit_parameters,fp)

### Ramsey Experiments

#### One-slice experiment

In [ ]:
#Set Ramsey detuning for single sweep
qubit_parameters.update_parameter("ramsey_det", 0.5e6)
qubit_parameters["ramsey_det"]

# qubit_parameters.update_parameter("relax", 100e-6)
#qubit_parameters.update_parameter('qb_t2_ramsey', 10e-6)

In [ ]:
qubit_parameters["ramsey_det"]


In [ ]:
# delay range for ramsey pulses
ramsey_min = 0.0

#ramsey_max = 5.0e-6 
ramsey_max = 15e-6 

# how many delay points to sweep
ramsey_num = 101
#ramsey_num = 151

# set up delay sweep parameter
ramsey_sweep = LinearSweepParameter(
    uid="ramsey_delay", start=ramsey_min, stop=ramsey_max, count=ramsey_num
)

# how many averages per point: 2^n_average
n_average = 12

In [ ]:
exp_ramsey = create_ramsey(ramsey_sweep, 
                           x90, 
                           readout_low, #readout_low,
                           n_average, 
                           qubit_parameters)

exp_ramsey_comp = my_session.compile(exp_ramsey)

In [ ]:
#my_results = my_session.run(exp_ramsey)
ramsey_results = my_session.run(exp_ramsey_comp)

meas_ready()

In [ ]:
# get measurement data returned by the instruments
ramsey_res = ramsey_results.get_data("q0_ramsey")

# define time axis from qubit parameters
ramsey_delay = ramsey_results.get_axis("q0_ramsey")[0]

In [ ]:
# popt[0]/np.pi/2*1e-6 = np.float64(0.5165350234216134) old

In [ ]:
# plot measurement results
fig = plt.figure(figsize=(12,8))

#data = ramsey_res.real
data = ramsey_res.imag
# data = abs(ramsey_res)
# data = np.unwrap(np.angle(ramsey_res))

plt.plot(ramsey_delay, data, ".k") #np.unwrap(np.angle(
#plt.plot(ramsey_delay, np.unwrap(np.angle(ramsey_res)), ".k")
plt.ylabel("A (a.u.)")
plt.xlabel("delay (s)")
plt.title('Ramsey oscillations')

# increase number of plot points for smooth plotting of fit results
delay_plot = np.linspace(ramsey_delay[0], ramsey_delay[-1], 5 * len(ramsey_delay))

## fit measurement data to decaying sinusoidal oscillatio
popt, pcov = fit_Ramsey(
    x=ramsey_delay,
    y=data,
    #np.unwrap(np.angle(ramsey_res)),
    freq=5* qubit_parameters["ramsey_det"],
    phase=0,
    #2 / qubit_parameters["qb_t2_ramsey"],
    rate=1 / 10e-6,
    amp=0.02,
    off=0.129,
    plot=False,
     bounds=[
         [0.1e6, -np.pi, 0.1 / qubit_parameters["qb_t2_ramsey"], 0.0001, -2],
         [50e6, np.pi, 10 / qubit_parameters["qb_t2_ramsey"], 2, 2],
    ],
#    bounds=[
#        [0.01e6, -1.5*np.pi, 0.05 / 6e-5, 0.0001, -4],
#        [150e6, 1.5*np.pi, 30 / 6e-5, 10, 10],
#    ],
)
#print(popt)

# plot fit results together with experimental data
plt.plot(delay_plot, func_decayOsc(delay_plot, *popt), "-r")
plt.figtext(0.6, 0.8, 'T2 = '+'{:.3f}'.format(1/popt[2]*1e6)+'us')
plt.figtext(0.6, 0.75, r'$\Delta$ = '+'{:.3f}'.format(popt[0]/(2*np.pi)*1e-6)+'MHz')

# update qubit parameters - here qubit dephasing time and qubit frequency
#qubit_parameters["qb_t2_ramsey"] = 1 / popt[2]
# qubit_parameters.update_parameter('qb_t2_ramsey', 1 / popt[2])
# qubit_parameters.update_parameter('qb_freq', qubit_parameters['qb_freq'] + qubit_parameters["ramsey_det"] - popt[0]/np.pi/2) #for positive detuning
# qubit_parameters.update_parameter('qb_freq', qubit_parameters['qb_freq'] + qubit_parameters["ramsey_det"] + popt[0]/np.pi/2) #for negative detuning

# T2 Ramsey error in us
err = np.sqrt(np.diag(pcov))
print(f'T2* = {1/popt[2]*1e6:.3f} +- {err[2]/popt[2]*qubit_parameters["qb_t2_ramsey"]*1e6:.3f} us')
print(f'Detuning is {popt[0]/np.pi/2*1e-6:.4} +- {err[0]/np.pi/2*1e-6:.2} MHz')

#save figure
figname = 'Ramsey_slice_'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

In [ ]:
qb_f_neg=qubit_parameters['qb_freq'] + qubit_parameters["ramsey_det"] - popt[0]/2/np.pi
qb_f_pos=qubit_parameters['qb_freq'] - qubit_parameters["ramsey_det"] + popt[0]/2/np.pi

In [ ]:
#try pos
qubit_parameters.update_parameter('qb_freq',qb_f_pos)

In [ ]:
#qubit_parameters.update_parameter('qb_freq',47.70797100737143*1e6) #default

In [ ]:
qubit_parameters['qb_freq']

In [ ]:
#try neg
qubit_parameters.update_parameter('qb_freq',qb_f_neg)

#### Ramsey fringes

In [ ]:
#set ramsey detuning limitst and point number

ramsey_min_freq = -2
ramsey_max_freq = 2
N_sweep = 21

ramsey_shev_det = np.linspace(ramsey_min_freq, ramsey_max_freq, N_sweep)*1e6

In [ ]:
ramsey_sh_list = []

for i in range(N_sweep):
    ramsey_cal_sh = make_ramsey_calib(qubit_parameters["qb_freq"], ramsey_shev_det[i])
    
    exp_ramsey.set_calibration(ramsey_cal_sh)
    shev_results = my_session.run(exp_ramsey)
    
    ramsey_sh_list.append(shev_results.get_data("q0_ramsey"))

ramsey_fringes_arr = np.array(ramsey_sh_list)

#restore proper calibration
exp_ramsey.set_calibration(ramsey_cal)

In [ ]:
plt.plot(ramsey_delay, np.unwrap(np.angle(ramsey_fringes_arr[25,:])))

In [ ]:
ramsey_fringes_arr.shape

In [ ]:
X = ramsey_shev_det
Y = ramsey_delay
Z = np.absolute(ramsey_fringes_arr)
#Z = np.unwrap(np.angle(ramsey_fringes_arr))
calc.plot_2d(X, Y, Z, flip = False)

# figname = 'Ramsey_fringes_'
# file_path = get_path_to_file(figname, '.png')
# plt.savefig(file_path, dpi=600, format='png', metadata=None,
#         bbox_inches=None, pad_inches=0.1,
#         facecolor='auto', edgecolor='auto',
#         backend=None,
#        )

In [ ]:
Data_ramsey_shev = {'ramsey_fr': ramsey_fringes_arr,
                   'ramsey_delay': ramsey_delay, 
                    'ramsey_fr_det': ramsey_shev_det
               }

Data_ramsey_shev.update(qubit_parameters)
Data_ramsey_shev.update(lo_settings)

file_name = 'Ramsey_fringes_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_ramsey_shev)

### Hahn Echo Experiment

In [ ]:
# delay range for echo pulses
echo_min = 0.0
echo_max = 50e-6 # 5.0 * qubit_parameters["qb_t2_echo"]
# how many delay points to sweep
echo_num = 151

# # set up delay sweep parameter - max of sweep parameter is half of max delay since here are two delays in sequence
# echo_sweep = LinearSweepParameter(
#     uid="echo_delay", start=echo_min, stop=echo_max / 2, count=echo_num
# )

echo_sweep = log_sweep_help(echo_min, echo_max/2, echo_num)

# how many averages per point: 2^n_average
n_average = 12

In [ ]:
echo_sweep.stop = echo_max

In [ ]:
exp_echo = create_echo(echo_sweep, 
                       x90, 
                       x180, 
                       readout_low, 
                       n_average)

exp_echo_comp = my_session.compile(exp_echo)

In [ ]:
echo_results = my_session.run(exp_echo_comp)

In [ ]:
# get measurement data returned by the instruments
echo_res = echo_results.get_data("q0_echo")

# define time axis from qubit parameters
echo_delay = echo_results.get_axis("q0_echo")[0]

In [ ]:
# plot measurement results - multiply delay by factor of two to get full sequence length
fig = plt.figure()
#plt.plot(2 * echo_delay, abs(echo_res), ".k") #np.unwrap(np.angle(
plt.plot(2 * echo_delay, np.unwrap(np.angle(echo_res)), ".k")
plt.ylabel("A (a.u.)")
plt.xlabel("delay (s)")
plt.title('Hahn echo')

# increase number of plot points for smooth plotting of fit results
delay_plot = np.linspace(echo_delay[0], 2 * echo_delay[-1], 5 * len(echo_delay))

## fit measurement data to decaying sinusoidal oscillatio
#popt, pcov = fit_T1(2 * echo_delay, abs(echo_res), 1e6, 0, -1, plot=False)
popt, pcov = fit_T1(2 * echo_delay, np.unwrap(np.angle(echo_res)), 1e6, 0, -1, plot=False)
print(popt)

# plot fit results together with experimental data
plt.plot(delay_plot, func_exp(delay_plot, *popt), "-r")

# update qubit parameters - here qubit dephasing time
# qubit_parameters["qb_t2_echo"] = 1 / popt[0]
print(1 / popt[0] * 1e6, ' us')

# T2 Echo error in us
err = np.sqrt(np.diag(pcov))
#print(f'T2E = {qubit_parameters["qb_t2_echo"]*1e6:.3f} +- {err[2]/popt[2]*qubit_parameters["qb_t2_echo"]*1e6:.3f} us')

#Save figure
figname = 'Hahn_echo'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

In [ ]:
Data_hahn_echo = {'echo_res': echo_res,
               'echo_delay': echo_delay
               }

Data_hahn_echo.update(qubit_parameters)
Data_hahn_echo.update(lo_settings)

file_name = 'Hahn_echo_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_hahn_echo)

In [ ]:
file_name = 'Qubit_params_dict'
file_path = get_path_to_file(file_name, '.txt', sample_parameters)
with open(file_path, 'w') as fp:
    js.dump(qubit_parameters,fp)

### Rabi Error Amplification

#### x90 pulse

In [ ]:
max_pulses = 15
n_average = 12

rabi_err_amp_flips = np.arange(1, max_pulses+1, 2)
flips_sweep = SweepParameter(values=rabi_err_amp_flips)

print(f'Repition sequence: {rabi_err_amp_flips}')
print(f'Approximate length of longest pulse sequence: {rabi_err_amp_flips.max() * x90.length * 1e6} us')

# qubit_parameters.update_parameter('relax',  200e-6)
# readout_opt['relax_time'] = qubit_parameters['relax']

In [ ]:
exp_rabi_error_amp_eg_x90 = create_rabi_error_amp(flips_sweep, x180, x90, 
                                              readout_low,
                                              n_average,
                                             target_pulse='x90eg')

exp_rabi_error_amp_eg_comp_x90 = my_session.compile(exp_rabi_error_amp_eg_x90)

# show_pulse_sheet("Rabi Error Amplification", exp_rabi_error_amp_eg_comp)

In [ ]:
rabi_err_amp_results_x90 = my_session.run(exp_rabi_error_amp_eg_comp_x90)

In [ ]:
# get measurement data returned by the instruments
rabi_err_amp_res = rabi_err_amp_results_x90.get_data("q0_rabi")
rabi_err_amp_res_0 = rabi_err_amp_results_x90.get_data("calib_0")
rabi_err_amp_res_1 = rabi_err_amp_results_x90.get_data("calib_1")

In [ ]:
fig, ax = plt.subplots()
amp = np.abs(rabi_err_amp_res)

ax.scatter(rabi_err_amp_flips-1, amp)

ax.set_ylabel('Amplitude (a.u.)')
ax.set_xlabel('Number of applied pi/2 pulses')

axa = ax.twiny()
axa.set_xlabel('Approximate Pulse Sequence Duration (us)')
axa.plot(rabi_err_amp_flips * x90.length * 1e6, np.abs(rabi_err_amp_res), alpha=0)

plot_flips = np.linspace(1, max_pulses + 1, 501)
print(rabi_err_amp_flips)

calib_amp = np.abs(rabi_err_amp_res_0 - rabi_err_amp_res_1) / 2

try:
    popt_amp, pcov_amp = fit_osc_fixed_amp_phase(rabi_err_amp_flips - 1, amp, 
                                  freq=1.5, 
                                  fixed_phase=np.pi / 2, 
                                  fixed_amp=calib_amp, 
                                  off=np.mean(amp))
    popt_amp = list(popt_amp)
    popt_amp.insert(1, np.pi / 2)
    popt_amp.insert(2, calib_amp)
    ax.plot(plot_flips - 1, func_osc(plot_flips - 1 , *popt_amp), "-r")
    # print(popt_amp)

except RuntimeError as ex:
    print('ERROR!')
    print(ex, end='\n\n')
    
ax.hlines([np.abs(rabi_err_amp_res_0)], plot_flips.min(), plot_flips.max(), color='gray', linestyle='dashed')
ax.hlines([np.abs(rabi_err_amp_res_1)], plot_flips.min(), plot_flips.max(), color='gray', linestyle='dotted')

amp_corr = 0.5 * np.pi / popt_amp[0]
print(f'Current x90 amplitude: {qubit_parameters["pihalf_amp"]}')
print(f'Amplitude correction factor: {amp_corr}')

#### Update x90 pulse

In [ ]:
print(f'Old x90 amplitude: {qubit_parameters["pihalf_amp"]}')
qubit_parameters.update_parameter("pihalf_amp", qubit_parameters["pihalf_amp"] * amp_corr)
print(f'New x90 amplitude: {qubit_parameters["pihalf_amp"]}')

In [ ]:
amp_corr

In [ ]:
x90 = pulse_library.gaussian(
    uid="x90", length=qubit_parameters["qb_len"], amplitude=qubit_parameters["pihalf_amp"]
)

#### x180 pulse

In [ ]:
#qubit_parameters.update_parameter("pihalf_amp", qubit_parameters["pi_amp"]/2)

In [ ]:
max_pulses = 15
n_average = 12

rabi_err_amp_flips = np.arange(0, max_pulses+1)
flips_sweep = SweepParameter(values=rabi_err_amp_flips)

print(f'Repition sequence: {rabi_err_amp_flips}')
print(f'Approximate length of longest pulse sequence: {(rabi_err_amp_flips.max() * x180.length + x90.length) * 1e6} us')

# qubit_parameters.update_parameter('relax',  150e-6)
# readout_opt['relax_time'] = qubit_parameters['relax']

In [ ]:
#qubit_parameters.update_parameter('relax',  400e-6)

In [ ]:
exp_rabi_error_amp_eg = create_rabi_error_amp(flips_sweep, x180, x90, 
                                              readout_low,
                                              n_average,
                                             target_pulse='x180eg')

exp_rabi_error_amp_eg_comp = my_session.compile(exp_rabi_error_amp_eg)

# show_pulse_sheet("Rabi Error Amplification", exp_rabi_error_amp_eg_comp)

In [ ]:
rabi_err_amp_results = my_session.run(exp_rabi_error_amp_eg_comp)

In [ ]:
# get measurement data returned by the instruments
rabi_err_amp_res = rabi_err_amp_results.get_data("q0_rabi")
rabi_err_amp_res_0 = rabi_err_amp_results.get_data("calib_0")
rabi_err_amp_res_1 = rabi_err_amp_results.get_data("calib_1")

In [ ]:
fig, ax = plt.subplots()

ax.scatter(rabi_err_amp_flips, np.abs(rabi_err_amp_res))

ax.set_ylabel('Amplitude (a.u.)')
ax.set_xlabel('Number of applied pi pulses')

axa = ax.twiny()
axa.set_xlabel('Approximate Pulse Sequence Duration (us)')
axa.plot((rabi_err_amp_flips * x180.length + x90.length) * 1e6, np.abs(rabi_err_amp_res), alpha=0)

plot_flips = np.linspace(0, max_pulses + 1, 501)

amp = np.abs(rabi_err_amp_res)
calib_amp = np.abs(rabi_err_amp_res_0 - rabi_err_amp_res_1) / 2

try:
    popt_amp, pcov_amp = fit_osc_fixed_amp_phase(rabi_err_amp_flips, amp, 
                                  freq=np.pi, 
                                  fixed_phase=np.pi / 2, 
                                  fixed_amp=calib_amp, 
                                  off=np.mean(amp))
    popt_amp = list(popt_amp)
    popt_amp.insert(1, np.pi / 2)
    popt_amp.insert(2, calib_amp)
    ax.plot(plot_flips, func_osc(plot_flips, *popt_amp), "-r")

except RuntimeError as ex:
    print('ERROR!')
    print(ex, end='\n\n')
    
ax.hlines([np.abs(rabi_err_amp_res_0)], plot_flips.min(), plot_flips.max(), color='gray', linestyle='dashed')
ax.hlines([np.abs(rabi_err_amp_res_1)], plot_flips.min(), plot_flips.max(), color='gray', linestyle='dotted')

amp_corr = np.pi / popt_amp[0]
print(f'Current x180 amplitude: {qubit_parameters["pi_amp"]}')
print(f'Amplitude correction factor: {amp_corr}')

#### Update x180 pulse

In [ ]:
print(f'Old x180 amplitude: {qubit_parameters["pi_amp"]}')
qubit_parameters.update_parameter("pi_amp", qubit_parameters["pi_amp"] * amp_corr)
print(f'New x180 amplitude: {qubit_parameters["pi_amp"]}')

In [ ]:
x180 = pulse_library.gaussian(
    uid="x180", length=qubit_parameters["qb_len"], amplitude=qubit_parameters["pi_amp"]
)

## Second Transition: e-f

### Amplitude Rabi Experiment e-f

In [ ]:
# range of pulse amplitude scan
amp_min = 0
amp_max = 1.0 #min([qubit_parameters["pi_amp"] * 2.2, 1.0])
# how many amplitude points to measure
amp_num = 51

# set up sweep parameter - qubit drive pulse amplitude
rabi_ef_sweep = LinearSweepParameter(
    uid="rabi_ef_amp", start=amp_min, stop=amp_max, count=amp_num
)

# qubit_parameters.add_parameter("qb_ef_len", 50e-9)
qubit_parameters.update_parameter("qb_ef_len", 300e-9)
# how many averages per point: 2^n_average
n_average = 12

In [ ]:
#qubit_parameters.update_parameter("qb_ef_freq", -0.16561999999999966*1e6)

In [ ]:
qubit_parameters["qb_ef_len"]

In [ ]:
gaussian_ef = pulse_library.gaussian(
    uid="gaussian_ef", length=qubit_parameters["qb_ef_len"], amplitude=1.0
)

#qubit_parameters["qb_len"]

In [ ]:
# qubit_parameters['qb_ef_freq'] = 1
#qubit_parameters.update_parameter('qb_ef_freq', qubit_parameters["qb_freq"] - qubit_parameters['qb_anharm'])
print('Qubit ef transition frequency: ', qubit_parameters['qb_ef_freq']*1e-6, 'MHz')

In [ ]:
detuning_ef = 0 #e6

# qubit drive frequency - defined in calibration on device setup as baseline reference
drive_ef_Oscillator_q0.frequency = qubit_parameters["qb_ef_freq"] + detuning_ef
# set oscillator type to hardware to ensure optimal use of the instrument functionality
drive_ef_Oscillator_q0.modulation_type = ModulationType.HARDWARE

In [ ]:
exp_rabi_ef = create_rabi_ef(rabi_ef_sweep, 
                             gaussian_ef, 
                             readout_opt,
                             #readout_low,
                             n_average, 
                             x180, 
                             ge_amp = 1.0, 
                             n_ge_pulses = 2)

exp_rabi_ef_comp = my_session.compile(exp_rabi_ef)

show_pulse_sheet("Rabi-ef", exp_rabi_ef_comp)

In [ ]:
#my_results = my_session.run(exp_rabi_ef_comp)

rabi_ef_results = my_session.run(exp_rabi_ef_comp)

In [ ]:
# get measurement data returned by the instruments
rabi_ef_res = rabi_ef_results.get_data("q0_rabi_ef")

# define amplitude axis from qubit parameters
rabi_ef_amp = rabi_ef_results.get_axis("q0_rabi_ef")[0]

In [ ]:
# plot measurement data
fig = plt.figure()
plt.plot(rabi_ef_amp, abs(rabi_ef_res), ".k")
plt.ylabel("A (a.u.)")
# plt.plot(rabi_ef_amp, np.unwrap(np.angle(rabi_ef_res)), ".k")
# plt.ylabel("phase (deg)")
plt.xlabel("amplitude (a.u.)")
plt.title('Rabi oscillations for ef-transition')
#ylim = [0.0370, 0.0400]
#plt.ylim(ylim)

# increase number of plot points for smooth plotting of fit results
ef_amp_plot = np.linspace(rabi_ef_amp[0], rabi_ef_amp[-1], 5 * len(rabi_ef_amp))

# fit measurement results - assume sinusoidal oscillation with drive amplitude
popt, pcov = fit_Rabi(rabi_ef_amp, abs(rabi_ef_res), 20, 1.0, 1.0, 0.1626, plot=False)
# popt, pcov = fit_Rabi(rabi_ef_amp, np.unwrap(np.angle(rabi_ef_res)), 10, 1.0, 0.1, -1.9, plot=False)

# plot fit results together with measurement data
plt.plot(ef_amp_plot, func_osc(ef_amp_plot, *popt), "-r")
#plt.plot(ef_amp_plot, func_decayOsc(ef_amp_plot, *popt), "-r")
# update qubit parameters - pulse amplitdues for pi and pi/2 pulses
# qubit_parameters.update_parameter("pi_ef_amp", np.pi / (popt[0]))
print(qubit_parameters["pi_ef_amp"])

#qubit_parameters["pihalf_ef_amp"] = np.pi / 2 / (popt[0])
#print(qubit_parameters["pihalf_ef_amp"])

print(np.pi / (popt[0]))

#Nmin = np.argmax(func_decayOsc(ef_amp_plot, *popt))
#print(ef_amp_plot[Nmin])

# qubit_parameters["pi_ef_amp"] = ef_amp_plot[Nmin]
# print(qubit_parameters["pi_ef_amp"])

# qubit_parameters["pihalf_ef_amp"] = ef_amp_plot[Nmin] / 2
# print(qubit_parameters["pihalf_ef_amp"])

In [ ]:
signal = rabi_ef_res

fig, ax = plt.subplots(3, 1, sharex = True, figsize = (10, 8))

I_zero = np.real(signal)[0]
Q_zero = np.imag(signal)[0]

new_signal = np.sqrt((np.real(signal) - I_zero)**2 + (np.imag(signal) - Q_zero)**2)

fig.suptitle('Rabi oscillations vs pulse amplitude', fontsize=16)

fig.supxlabel("Amplitude (a.u.)")
fig.supylabel("Amplitude (a.u.)")

ax[0].plot(rabi_ef_amp, np.real(signal), ".k")
ax[1].plot(rabi_ef_amp, np.imag(signal), ".k")
ax[2].plot(rabi_ef_amp, new_signal, ".k")

ax[0].set_title('real part')
ax[1].set_title('imaginary part')
ax[2].set_title('norm')

amp_ef_plot = np.linspace(rabi_ef_amp[0], rabi_ef_amp[-1], 5 * len(rabi_ef_amp))

popt_real, pcov_real = fit_Ramsey(rabi_ef_amp, np.real(signal),
    30, 0, 2,
    amp=0.01, off=2.16, plot=False)
popt_imag, pcov_imag = fit_Ramsey(rabi_ef_amp, np.imag(signal),
    30, 0, 2,
    amp=0.001, off=0.45, plot=False)
popt_norm, pcov_norm = fit_Ramsey(rabi_ef_amp[:], new_signal[:],
    30, 0, 2,
    amp=0.001, off=2e-3, plot=False)

ax[0].plot(amp_ef_plot, func_decayOsc(amp_ef_plot, *popt_real), "-r")
ax[1].plot(amp_ef_plot, func_decayOsc(amp_ef_plot, *popt_imag), "-r")
ax[2].plot(amp_ef_plot, func_decayOsc(amp_ef_plot, *popt_norm), "-r")


print('fitting results for x180-pulse amplitude:')
print('real  --- {:4.4f} +/- {:4.4f}'.format(np.pi/popt_real[0], np.pi*np.sqrt(np.diag(pcov_real))[0]/(popt_real[0])**2))
print('imag  --- {:4.4f} +/- {:4.4f}'.format(np.pi/popt_imag[0], np.pi*np.sqrt(np.diag(pcov_imag))[0]/(popt_imag[0])**2))
print('norm  --- {:4.4f} +/- {:4.4f}'.format(np.pi/popt_norm[0], np.pi*np.sqrt(np.diag(pcov_norm))[0]/(popt_norm[0])**2))

Nmax = np.argmax(func_decayOsc(amp_ef_plot, *popt_norm))
print(amp_ef_plot[Nmax])

# qubit_parameters["pi_ef_amp"] = amp_ef_plot[Nmax]
# qubit_parameters["pihalf_ef_amp"] = amp_ef_plot[Nmax] / 2
# print(qubit_parameters["pi_ef_amp"])
# print(qubit_parameters["pihalf_ef_amp"])

#qubit_parameters.update_parameter("pi_ef_amp", np.pi/popt_norm[0])
print(qubit_parameters["pi_ef_amp"])

#qubit_parameters.update_parameter("pihalf_ef_amp", np.pi / 2 / (popt_norm[0]))
print(qubit_parameters["pihalf_ef_amp"])

# qubit_parameters["pi_ef_amp"] = np.pi/popt_real[0]
#print(qubit_parameters["pi_ef_amp"])

# qubit_parameters["pihalf_ef_amp"] = np.pi / 2 / (popt_real[0])
# print(qubit_parameters["pihalf_ef_amp"])

In [ ]:
#qubit_parameters.add_parameter("pi_ef_amp", np.pi/popt_norm[0])
#qubit_parameters.add_parameter("pihalf_ef_amp", np.pi / 2 / (popt_norm[0]))


In [ ]:
#set a pi_ef pulse
x180_ef = pulse_library.gaussian(
    uid="x180_ef", length=qubit_parameters["qb_ef_len"], amplitude=qubit_parameters["pi_ef_amp"]
)

#set a pi_ef pulse
x90_ef = pulse_library.gaussian(
    uid="x90_ef", length=qubit_parameters["qb_ef_len"], amplitude=qubit_parameters["pi_ef_amp"]/2
)

### Rabi Error Amplification

#### x90 pulse

In [ ]:
max_pulses = 19
n_average = 13

rabi_err_amp_flips = np.arange(1, max_pulses+1, 2)
flips_sweep = SweepParameter(values=rabi_err_amp_flips)

print(f'Repition sequence: {rabi_err_amp_flips}')
print(f'Approximate length of longest pulse sequence: {(rabi_err_amp_flips.max() * x90_ef.length + 2 * x180.length) * 1e6} us')

# qubit_parameters.update_parameter('relax',  200e-6)
# readout_opt['relax_time'] = qubit_parameters['relax']

In [ ]:
exp_rabi_error_amp_ef_x90 = create_rabi_error_amp_ef(flips_sweep, x180, x90, x180_ef, x90_ef,
                                              readout_opt,
                                              n_average,
                                             target_pulse='x90ef')

exp_rabi_error_amp_ef_comp_x90 = my_session.compile(exp_rabi_error_amp_ef_x90)

# show_pulse_sheet("Rabi Error Amplification ef", exp_rabi_error_amp_ef_comp_x90)

In [ ]:
rabi_err_amp_results_x90_ef = my_session.run(exp_rabi_error_amp_ef_comp_x90)

In [ ]:
# get measurement data returned by the instruments
rabi_err_amp_res = rabi_err_amp_results_x90_ef.get_data("q0_rabi")
rabi_err_amp_res_0 = rabi_err_amp_results_x90_ef.get_data("calib_0")
rabi_err_amp_res_1 = rabi_err_amp_results_x90_ef.get_data("calib_1")

In [ ]:
fig, ax = plt.subplots()
amp = np.abs(rabi_err_amp_res)

ax.scatter(rabi_err_amp_flips-1, amp)

ax.set_ylabel('Amplitude (a.u.)')
ax.set_xlabel('Number of applied pi/2 pulses')

axa = ax.twiny()
axa.set_xlabel('Approximate Pulse Sequence Duration (us)')
axa.plot((rabi_err_amp_flips * x90_ef.length + 2 * x180.length) * 1e6, np.abs(rabi_err_amp_res), alpha=0)

plot_flips = np.linspace(1, max_pulses + 1, 501)
print(rabi_err_amp_flips)

calib_amp = np.abs(rabi_err_amp_res_0 - rabi_err_amp_res_1) / 2

try:
    popt_amp, pcov_amp = fit_osc_fixed_amp_phase(rabi_err_amp_flips - 1, amp, 
                                  freq=1.5, 
                                  fixed_phase=np.pi / 2, 
                                  fixed_amp=calib_amp, 
                                  off=np.mean(amp))
    popt_amp = list(popt_amp)
    popt_amp.insert(1, np.pi / 2)
    popt_amp.insert(2, calib_amp)
    ax.plot(plot_flips - 1, func_osc(plot_flips - 1 , *popt_amp), "-r")
    # print(popt_amp)

except RuntimeError as ex:
    print('ERROR!')
    print(ex, end='\n\n')
    
ax.hlines([np.abs(rabi_err_amp_res_0)], plot_flips.min(), plot_flips.max(), color='gray', linestyle='dashed')
ax.hlines([np.abs(rabi_err_amp_res_1)], plot_flips.min(), plot_flips.max(), color='gray', linestyle='dotted')

amp_corr = 0.5 * np.pi / popt_amp[0]
print(f'Current x90 ef amplitude: {qubit_parameters["pihalf_ef_amp"]}')
print(f'Amplitude correction factor: {amp_corr}')

#### Update x90 pulse

In [ ]:
print(f'Old x90 ef amplitude: {qubit_parameters["pihalf_ef_amp"]}')
qubit_parameters.update_parameter("pihalf_ef_amp", qubit_parameters["pihalf_ef_amp"] * amp_corr)
print(f'New x90 ef amplitude: {qubit_parameters["pihalf_ef_amp"]}')

In [ ]:
x90_ef = pulse_library.gaussian(
    uid="x90_ef", length=qubit_parameters["qb_ef_len"], amplitude=qubit_parameters["pihalf_ef_amp"]
)

#### x180 pulse

In [ ]:
max_pulses = 25
n_average = 12

rabi_err_amp_flips = np.arange(0, max_pulses+1)
flips_sweep = SweepParameter(values=rabi_err_amp_flips)

print(f'Repition sequence: {rabi_err_amp_flips}')
print(f'Approximate length of longest pulse sequence: {(rabi_err_amp_flips.max() * x180_ef.length + 2 * x180.length + x90_ef.length) * 1e6} us')

# qubit_parameters.update_parameter('relax',  150e-6)
# readout_opt['relax_time'] = qubit_parameters['relax']

In [ ]:
exp_rabi_error_amp_ef = create_rabi_error_amp_ef(flips_sweep, x180, x90, x180_ef, x90_ef,
                                              readout_opt,
                                              n_average,
                                             target_pulse='x180ef')

exp_rabi_error_amp_ef_comp = my_session.compile(exp_rabi_error_amp_ef)

# show_pulse_sheet("Rabi Error Amplification", exp_rabi_error_amp_eg_comp)

In [ ]:
rabi_err_amp_results = my_session.run(exp_rabi_error_amp_ef_comp)

In [ ]:
# get measurement data returned by the instruments
rabi_err_amp_res = rabi_err_amp_results.get_data("q0_rabi")
rabi_err_amp_res_0 = rabi_err_amp_results.get_data("calib_0")
rabi_err_amp_res_1 = rabi_err_amp_results.get_data("calib_1")

In [ ]:
fig, ax = plt.subplots()

ax.scatter(rabi_err_amp_flips, np.abs(rabi_err_amp_res))

ax.set_ylabel('Amplitude (a.u.)')
ax.set_xlabel('Number of applied pi pulses')

axa = ax.twiny()
axa.set_xlabel('Approximate Pulse Sequence Duration (us)')
axa.plot((rabi_err_amp_flips * x180_ef.length + 2 * x180.length + x90_ef.length) * 1e6, np.abs(rabi_err_amp_res), alpha=0)

plot_flips = np.linspace(0, max_pulses + 1, 501)

amp = np.abs(rabi_err_amp_res)
calib_amp = np.abs(rabi_err_amp_res_0 - rabi_err_amp_res_1) / 2

try:
    popt_amp, pcov_amp = fit_osc_fixed_amp_phase(rabi_err_amp_flips, amp, 
                                  freq=np.pi, 
                                  fixed_phase=np.pi / 2, 
                                  fixed_amp=calib_amp, 
                                  off=np.mean(amp))
    popt_amp = list(popt_amp)
    popt_amp.insert(1, np.pi / 2)
    popt_amp.insert(2, calib_amp)
    ax.plot(plot_flips, func_osc(plot_flips, *popt_amp), "-r")

except RuntimeError as ex:
    print('ERROR!')
    print(ex, end='\n\n')
    
ax.hlines([np.abs(rabi_err_amp_res_0)], plot_flips.min(), plot_flips.max(), color='gray', linestyle='dashed')
ax.hlines([np.abs(rabi_err_amp_res_1)], plot_flips.min(), plot_flips.max(), color='gray', linestyle='dotted')

amp_corr = np.pi / popt_amp[0]
print(f'Current x180 ef amplitude: {qubit_parameters["pi_ef_amp"]}')
print(f'Amplitude correction factor: {amp_corr}')

#### Update x180 pulse

In [ ]:
print(f'Old x180 ef amplitude: {qubit_parameters["pi_ef_amp"]}')
qubit_parameters.update_parameter("pi_ef_amp", qubit_parameters["pi_ef_amp"] * amp_corr)
print(f'New x180 ef amplitude: {qubit_parameters["pi_ef_amp"]}')

In [ ]:
x180_ef = pulse_library.gaussian(
    uid="x180_ef", length=qubit_parameters["qb_ef_len"], amplitude=qubit_parameters["pi_ef_amp"]
)

### Rabi shevron e-f

In [ ]:
results_list = []

detuning_ef = np.linspace(-20.0, 20.0, 31)*1e6

In [ ]:
(qubit_parameters["qb_ef_freq"] + lo_settings['qb_lo']) * 1e-9

In [ ]:
for i in range(len(detuning_ef)):
    print('Measurement number:', i)
    # qubit drive frequency - defined in calibration on device setup as baseline reference
    drive_ef_Oscillator_q0.frequency = qubit_parameters["qb_ef_freq"] + detuning_ef[i]
    # set oscillator type to hardware to ensure optimal use of the instrument functionality
    drive_ef_Oscillator_q0.modulation_type = ModulationType.HARDWARE
    
    my_results = my_session.run(exp_rabi_ef)
    results_list.append(my_results.get_data("q0_rabi_ef"))
    
results_arr = np.array(results_list)

In [ ]:
i = 15
#plt.plot(rabi_ef_amp, np.unwrap(np.angle(results_list[11])), "-o")
plt.plot(rabi_ef_amp, results_list[i].imag, "-o")
#plt.plot(rabi_ef_amp, np.unwrap(np.angle(results_list[23])), "-o")
print(detuning_ef[i]*1e-6, 'MHz')

In [ ]:
X = rabi_ef_amp
Y = detuning_ef
# Z = np.unwrap(np.angle(results_arr))
#Z = np.absolute(results_arr)
Z = np.imag(results_arr)
calc.plot_2d(Y, X,Z, flip = False, cmap = 'Reds')

In [ ]:
# qubit_parameters.update_parameter("qb_ef_freq", qubit_parameters["qb_ef_freq"] + 15e6)

In [ ]:
Data_ef_rabi = {'rabi_shev_ef': results_arr,
               'detuning_ef': detuning_ef,
               'rabi_ef_amp': rabi_ef_amp,
               }

Data_ef_rabi.update(qubit_parameters._params)
Data_ef_rabi.update(lo_settings._params)

file_name = 'Rabi_shev_ef'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_ef_rabi)

In [ ]:
file_name = 'Qubit_params'
file_path = get_path_to_file(file_name, '.txt', sample_parameters)
with open(file_path, 'w') as fp:
    js.dump(qubit_parameters,fp)

### T1 Experiement e-f

In [ ]:
# delay range for ramsey pulses
t1_min = 0.0
t1_max = 5e-6 #5 * qubit_parameters["qb_t1"]
# how many delay points to sweep
t1_num = 31

# set up sweep parameter
t1_ef_sweep = LinearSweepParameter(uid="t1_delay", start=t1_min, stop=t1_max, count=t1_num)

# how many averages per point: 2^n_average
n_average = 13

In [ ]:
exp_T1_ef = create_T1_ef_1(t1_ef_sweep, 
                           x180_ef, 
                           readout_low, 
                           n_average, 
                           x180, 
                           ge_amp = 1.0)

exp_T1_ef_comp = my_session.compile(exp_T1_ef)

#show_pulse_sheet("Decay-ef", exp_T1_ef_comp)

In [ ]:
T1_ef_results = my_session.run(exp_T1_ef_comp)

In [ ]:
# get measurement data returned by the instruments
t1_ef_res = T1_ef_results.get_data("q0_t1_ef")

# define amplitude axis from qubit parameters
t1_ef_delay = T1_ef_results.get_axis("q0_t1_ef")[0]

In [ ]:
# plot measurement results
fig = plt.figure()
plt.plot(t1_ef_delay, abs(t1_ef_res), ".k")
# plt.plot(t1_ef_delay, np.unwrap(np.angle(t1_ef_res)), ".k")
plt.ylabel("A (a.u.)")
plt.xlabel("delay (s)")
plt.title('Decay on ef-transition')

# increase number of plot points for smooth plotting of fit results
delay_plot = np.linspace(t1_ef_delay[0], t1_ef_delay[-1], 5 * len(t1_ef_delay))

# fit measurement results to decaying exponential curve
popt, pcov = fit_T1_ef(x = t1_ef_delay, 
                       # y = np.unwrap(np.angle(t1_ef_res)),
                       y = abs(t1_ef_res),
                       rate1 = 1/4e-6, 
                       rate2 = 1/16e-6,
                       off = 1.54, 
                       A = 0.1,
                       B = 0.1,
                       plot=False)


# plot fit results together with measurement data
plt.plot(delay_plot, func_two_exp(delay_plot, *popt), "-r")
plt.figtext(0.6, 0.8, r'$T_1^{ef}$ = '+'{:.1f}'.format(1/popt[0]*1e6)+r'$\mu$s')
plt.figtext(0.6, 0.7, r'$T_1^{ge}$ = '+'{:.1f}'.format(1/popt[1]*1e6)+r'$\mu$s')

# update qubit parameters - here relaxation time / qubit lifetime
qubit_parameters["qb_ef_t1"] = 1 / popt[0]


#gamma_fe = (popt[0]+popt[1])
#gamma_ge = popt[0]*popt[1]/gamma_fe

print('T1_fe:', 1/popt[0])
print('T1_ge:', 1/popt[1])

#save figure
figname = 'T1_ef_'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

In [ ]:
Data_T1_ef = {'t1_ef_res': t1_res,
            't1_ef_delay': t1_delay
            }

Data_T1_ef.update(qubit_parameters)
Data_T1_ef.update(lo_settings)

file_name = 'T1_ef_onege_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_T1_ef)

### Ramsey Experiement e-f

In [ ]:
# qubit_parameters['ramsey_det_ef'] = 2e6
qubit_parameters.update_parameter('ramsey_det_ef', 0.5e6)

In [ ]:
# delay range for ramsey pulses
ramsey_min = 0.0
#ramsey_max = 4.0 * qubit_parameters["ramsey_det"]
ramsey_max = 10e-6 #3.5 * qubit_parameters["qb_t2_ramsey"]
#ramsey_max = 1.5e-6 
# how many delay points to sweep
#ramsey_num = 151
ramsey_num = 401

# set up delay sweep parameter
ramsey_ef_sweep = LinearSweepParameter(
    uid="ramsey_delay", start=ramsey_min, stop=ramsey_max, count=ramsey_num
)

# how many averages per point: 2^n_average
n_average = 11

In [ ]:
#make experiment
exp_ramsey_ef_2 = make_ramsey_ef_2(ramsey_ef_sweep, 
                 x90_ef, 
                 readout_pulse, 
                 readout_weighting_function, 
                 qubit_parameters["relax"], 
                 n_average, 
                 x180, 
                 ge_amp = 1.0)

#set signal map
exp_ramsey_ef_2.set_signal_map(qubit_ef_map)

#make and set calibration
ramsey_ef_cal = make_ramsey_ef_calib(qubit_parameters["qb_ef_freq"], qubit_parameters['ramsey_det_ef'])
exp_ramsey_ef_2.set_calibration(ramsey_ef_cal) 

#compile the experiment
exp_ramsey_ef_2_comp = my_session.compile(exp_ramsey_ef_2)

#show_pulse_sheet("Ramsey-ef", exp_ramsey_ef_2_comp)

In [ ]:
#run the experiment
ramsey_ef_results = my_session.run(exp_ramsey_ef_2_comp)

In [ ]:
# get measurement data returned by the instruments
ramsey_ef_res = ramsey_ef_results.get_data("q0_ramsey_ef")

# define time axis from qubit parameters
ramsey_ef_delay = ramsey_ef_results.get_axis("q0_ramsey_ef")[0]

In [ ]:
# plot measurement results
fig = plt.figure(figsize=(12,8))

data_ramsey_ef = ramsey_ef_res.real
# data_ramsey_ef = np.unwrap(np.angle(ramsey_ef_res))

#plt.plot(ramsey_delay, abs(ramsey_res), ".k") #np.unwrap(np.angle(
plt.plot(ramsey_ef_delay, data_ramsey_ef, ".k")
plt.ylabel("A (a.u.)")
plt.xlabel("delay (s)")
plt.title('Ramsey oscillations')

# increase number of plot points for smooth plotting of fit results
delay_plot = np.linspace(ramsey_ef_delay[0], ramsey_ef_delay[-1], 5 * len(ramsey_ef_delay))

## fit measurement data to decaying sinusoidal oscillatio
popt, pcov = fit_Ramsey(
    ramsey_ef_delay,
    data_ramsey_ef,
    1e6, #qubit_parameters["ramsey_det"],
    0,
    #2 / qubit_parameters["qb_t2_ramsey"],
    2 / 6e-7,
    amp=0.01,
    off=-0.08,
    plot=False,
#     bounds=[
#         [0.1e6, -np.pi, 0.05 / qubit_parameters["qb_t2_ramsey"], 0.0001, -2],
#         [50e6, np.pi, 10 / qubit_parameters["qb_t2_ramsey"], 2, 2],
#     ],
    bounds=[
        [0.01e6, -np.pi, 0.05 / 6e-7, 0.0001, -4],
        [150e6, np.pi, 30 / 6e-7, 2, 4],
    ],
)
#print(popt)

# plot fit results together with experimental data
plt.plot(delay_plot, func_decayOsc(delay_plot, *popt), "-r")
plt.figtext(0.6, 0.8, 'T2 = '+'{:.3f}'.format(1/popt[2]*1e6)+'us')
plt.figtext(0.6, 0.75, r'$\Delta$ = '+'{:.3f}'.format(popt[0]/(2*np.pi)*1e-6)+'MHz')

# update qubit parameters - here qubit dephasing time and qubit frequency
#qubit_parameters["qb_t2_ramsey"] = 1 / popt[2]

# qubit_parameters.update_parameter('qb_ef_freq', qubit_parameters['qb_ef_freq'] + qubit_parameters["ramsey_det_ef"] - popt[0]/np.pi/2)

# T2 Ramsey error in us
err = np.sqrt(np.diag(pcov))
#print(f'T2* = {1/popt[2]*1e6:.3f} +- {err[2]/popt[2]*qubit_parameters["qb_t2_ramsey"]*1e6:.3f} us')
print(f'Detuning is {popt[0]/np.pi/2*1e-6:.4} +- {err[0]/np.pi/2*1e-6:.2} MHz')

#save figure
figname = 'Ramsey_ef_slice_'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

In [ ]:
Data_Ramsey_ef = {'ramsey_ef_res': ramsey_ef_res,
            'ramsey_ef_delay': ramsey_ef_delay
            }

Data_Ramsey_ef.update(qubit_parameters._params)
Data_Ramsey_ef.update(lo_settings._params)

file_name = 'Ramsey_ef_onege_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_Ramsey_ef)

# Readout optimization

In [ ]:
n_average = 12

ro_amp_min = 0
ro_amp_max = 1
ro_amp_points = 11

ro_amp_arr = np.linspace(ro_amp_min, ro_amp_max, ro_amp_points)

In [ ]:
lsg_q0["measure_line"].range = -20

pulse_length = 2e-6

readout_weighting_function_sw = pulse_library.const(
            uid="readout_weighting_function_sw", length=pulse_length, amplitude=1.0
        )

In [ ]:
rabi_sw_res = []
popt_rabi_sw = []
pcov_rabi_sw = []

In [ ]:
for ro_amp in ro_amp_arr:
    print('Measure at amplitude:', round(ro_amp, 5))
    
    # #Set readout pulse
    # readout_pulse_sw = pulse_library.gaussian_square(
    #     uid="readout_pulse",
    #     length=pulse_length, #qubit_parameters["ro_len"],
    #     amplitude=round(ro_amp, 5),
    # )

    readout_pulse_sw = pulse_library.gaussian_square(
        uid="readout_pulse",
        length=qubit_parameters["ro_len"],
        amplitude=round(ro_amp, 5),
        width = qubit_parameters["ro_len"]*0.95,
    )
    
    #experiment definition
    exp_rabi_sw = make_rabi(rabi_sweep, 
                     gaussian_pulse, 
                     readout_pulse_sw, 
                     readout_weighting_function_sw, 
                     qubit_parameters["relax"], 
                     n_average)

    # set signal map for rabi experiment no experimental calibration necessary, 
    #calibration taken from DeviceSetup, i.e. baseline
    exp_rabi_sw.set_signal_map(qubit_meas_map)
    # compile
    exp_rabi_sw_comp = my_session.compile(exp_rabi_sw) 
    rabi_sw_results = my_session.run(exp_rabi_sw_comp)
    
    # get measurement data returned by the instruments
    rabi_res = rabi_sw_results.get_data("q0_rabi")
    rabi_amp = rabi_sw_results.get_axis("q0_rabi")[0]
    
    rabi_sw_res.append(rabi_res)
    
    I_zero = np.real(rabi_res)[0]
    Q_zero = np.imag(rabi_res)[0]
    
    norm_signal = np.sqrt((np.real(rabi_res) - I_zero)**2 + (np.imag(rabi_res) - Q_zero)**2)
    
    popt_norm, pcov_norm = fit_Rabi(rabi_amp, norm_signal, 10, 1, 1, 0.048, plot=True)
    
    popt_rabi_sw.append(popt_norm)
    pcov_rabi_sw.append(pcov_norm)
    
popt_rabi_arr = np.array(popt_rabi_sw)
pcov_rabi_arr = np.array(pcov_rabi_sw)

In [ ]:
#get error from covariance matrix
err = []

for pcov in pcov_rabi_sw:
    err.append(np.sqrt(np.diag(pcov)))
    
err_arr = np.array(err)

In [ ]:
fig, ax = plt.subplots(2, 2, sharex = True, figsize = (10, 8))

fig.suptitle('Relative Errors for Rabi measurements', fontsize=16)

fig.supxlabel("Readout amplitude (a.u.)")
fig.supylabel("Error (a.u.)")

ll = 1
ul = -1

# ax[0,0].plot(ro_amp_arr, err_arr[:,0], ".k", label = 'freq')
# ax[1,0].plot(ro_amp_arr, err_arr[:,1], ".k", label = 'phase')
# ax[0,1].plot(ro_amp_arr, err_arr[:,2], ".k", label = 'amp')
# ax[1,1].plot(ro_amp_arr, err_arr[:,3], ".k", label = 'off')

ax[0,0].plot(ro_amp_arr[ll:ul], err_arr[ll:ul,0]/popt_rabi_arr[ll:ul,0], ".k", label = 'freq')
ax[1,0].plot(ro_amp_arr[ll:ul], err_arr[ll:ul,1]/popt_rabi_arr[ll:ul,1], ".k", label = 'phase')
ax[0,1].plot(ro_amp_arr[ll:ul], -err_arr[ll:ul,2]/popt_rabi_arr[ll:ul,2], ".k", label = 'amp')
ax[1,1].plot(ro_amp_arr[ll:ul], err_arr[ll:ul,3]/popt_rabi_arr[ll:ul,3], ".k", label = 'off')

ax[0,0].set_title('Frequency')
ax[1,0].set_title('Phase')
ax[0,1].set_title('Amplitude')
ax[1,1].set_title('Offset')

#save figure
figname = 'Rabi_optimiz_20dBm_'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

### Readout pulse settings

In [ ]:
#Set the parameters
qubit_parameters.update_parameter("ro_amp", 0.5)

qubit_parameters.update_parameter("ro_len", 2e-6)

In [ ]:
print('Readout length: ', qubit_parameters["ro_len"])
print('Readout amplitude: ', qubit_parameters["ro_amp"])
print('Readout Wfunc length: ', readout_weighting_function.length)

In [ ]:
print(readout_pulse)

In [ ]:
# qubit readout pulse - here simple constant pulse
# readout_pulse = pulse_library.const(
#     uid="readout_pulse",
#     length=qubit_parameters["ro_len"],
#     amplitude=qubit_parameters["ro_amp"],
# )

# qubit readout pulse - here with gaussian rise and fall
readout_pulse = pulse_library.gaussian_square(
    uid="readout_pulse_g",
    length=qubit_parameters["ro_len"],
    amplitude=qubit_parameters["ro_amp"],
    width = qubit_parameters["ro_len"]*0.95,
)

In [ ]:
# Set the weightning function
readout_weighting_function = pulse_library.const(
            uid="readout_weighting_function", length=qubit_parameters["ro_len"], amplitude=1.0
        )

In [ ]:
#Set the readout
readout_opt = {'readout_type': 'pulse',
                'readout_pulse': readout_pulse,
                'readout_weighting_function': readout_weighting_function,
                'relax_time': qubit_parameters["relax"],
                'measure_freq': qubit_parameters["ro_freq"],
                'acquire_freq': qubit_parameters["ro_freq"],
                'readout_range': -25,
                'readout_delay': qubit_parameters['ro_int_delay']
               }

readout_opt = QBaseParameters(sample=qsample_params,
                             name='readout_opt',
                             parameters=readout_opt)

## Spectroscopy of resonator in different states

In [ ]:
# frequency range of spectroscopy scan - around expected centre frequency as defined in qubit parameters
spec_range = 2e6
# how many frequency points to measure
spec_num = 101

# define sweep parameters for two qubits
freq_sweep_ST = LinearSweepParameter(
    uid="res_freq",
    start=qubit_parameters["ro_freq"] - spec_range / 2,
    stop=qubit_parameters["ro_freq"] + spec_range / 2,
    count=spec_num,
)

# freq_sweep_ST = LinearSweepParameter(
#     uid="res_freq",
#     start=91e6,
#     stop=99e6,
#     count=spec_num,
# )

# take how many averages per point: 2^n_average
n_average = 12

In [ ]:
exp_spec_g = create_res_spec_gef(freq_sweep_ST, 
                                 x180, 
                                 None, 
                                 readout_low,#readout_low,
                                 n_average, 
                                 level = 0)

exp_spec_e = create_res_spec_gef(freq_sweep_ST, 
                                 x180, 
                                 None, 
                                 readout_low,#readout_low,
                                 n_average, 
                                 level = 1)

exp_spec_f = create_res_spec_gef(freq_sweep_ST, 
                                 x180, 
                                 x180_ef, 
                                 readout_low,#readout_low,
                                 n_average, 
                                 level = 2)

In [ ]:
exp_spec_g_comp = my_session.compile(exp_spec_g)

exp_spec_e_comp = my_session.compile(exp_spec_e)

exp_spec_f_comp = my_session.compile(exp_spec_f)

In [ ]:
print('Spectroscopy for qubit in g-state:')
spec_g_res = my_session.run(exp_spec_g_comp)

print('Spectroscopy for qubit in e-state:')
spec_e_res = my_session.run(exp_spec_e_comp)

print('Spectroscopy for qubit in f-state:')
spec_f_res = my_session.run(exp_spec_f_comp)

meas_ready()

In [ ]:
# get measurement data returned by the instruments
res_0_res = spec_g_res.get_data("q0_res_spec_e")
# define a frequency axis from the parameters
res_0_freq = lo_settings["ro_lo"] + spec_g_res.get_axis("q0_res_spec_e")[0]

# get measurement data returned by the instruments
res_1_res = spec_e_res.get_data("q0_res_spec_e")
# define a frequency axis from the parameters
res_1_freq = lo_settings["ro_lo"] + spec_e_res.get_axis("q0_res_spec_e")[0]


# # get measurement data returned by the instruments
res_2_res = spec_f_res.get_data("q0_res_spec_f")
# define a frequency axis from the parameters
res_2_freq = lo_settings["ro_lo"] + spec_f_res.get_axis("q0_res_spec_f")[0]

In [ ]:
fig, ax = plt.subplots(2, 1, sharex = True, figsize = (10, 8))

fig.suptitle('Spectroscopy of readout resonator for three qubit states', fontsize=16)
fig.supxlabel('Readout frequency, GHz')

ax[0].set_title('Amplitude')
ax[1].set_title('Phase')

ax[0].plot(res_0_freq*1e-9, np.abs(res_0_res), 'b', label = 'ground')
ax[0].plot(res_1_freq*1e-9, np.abs(res_1_res), 'r', label = 'first exited')
ax[0].plot(res_2_freq*1e-9, np.abs(res_2_res), 'g', label = 'second exited')
ax[0].axvline(x = (lo_settings["ro_lo"] + qubit_parameters["ro_freq"])*1e-9)
ax[0].legend()
ax[0].set_ylabel('Amplitude, a.u.')

ax[1].plot(res_0_freq*1e-9, np.unwrap(np.angle(res_0_res)), 'b', label = 'ground')
ax[1].plot(res_1_freq*1e-9, np.unwrap(np.angle(res_1_res)), 'r', label = 'first exited')
ax[1].plot(res_2_freq*1e-9, np.unwrap(np.angle(res_2_res)), 'g', label = 'second exited')
ax[1].axvline(x = (lo_settings["ro_lo"] + qubit_parameters["ro_freq"])*1e-9)
ax[1].legend()
ax[1].set_ylabel('Phase, a.u.')

#save figure
figname = 'Readout_spec_for_g_e'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

In [ ]:
plt.title('Spectroscopy of readout resonator for three qubit states')

plt.plot(res_0_res.real, res_0_res.imag, 'b', label = 'ground')
plt.plot(res_1_res.real, res_1_res.imag, 'r', label = 'first exited')
plt.plot(res_2_res.real, res_2_res.imag, 'g', label = 'second exited')

plt.xlabel('Real part, a.u.')
plt.ylabel('Imaginary part, a.u.')

plt.legend()

In [ ]:
plt.title('Spectroscopy of readout resonator for three qubit states: distance')

plt.plot(res_1_freq*1e-9, np.abs(res_0_res-res_1_res), 'r', label = '0-1')
plt.plot(res_1_freq*1e-9, np.abs(res_1_res-res_2_res), 'b', label = '1-2')
plt.plot(res_1_freq*1e-9, np.abs(res_0_res-res_2_res), 'g', label = '0-2')
plt.axvline(x = (lo_settings["ro_lo"] + qubit_parameters["ro_freq"])*1e-9)
plt.axvline(x = (lo_settings["ro_lo"] + qubit_parameters["ro_freq"])*1e-9, ls = '--', label = 'readout')


max_dist_arg = np.argmax(np.abs(res_0_res-res_1_res))
print('Optimat readout frequency:', res_1_freq[max_dist_arg]*1e-6, 'MHz')
plt.axvline(x = res_1_freq[max_dist_arg]*1e-9, ls = '-.', label = 'optimal')

max_1_arg = np.argmax(np.abs(res_1_res))
plt.axvline(x = res_1_freq[max_1_arg]*1e-9, ls = '-.', label = 'optimal')

# plt.axvline(x = 6.893, ls = ':', label = 'manual optimal')

plt.legend()

figname = 'Readout_spec_for_g_e_'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

In [ ]:
#Define optimal readout parameter
#preset optimal frequency for ge
try:
    qubit_parameters.update_parameter("ro_freq_opt", res_1_freq[max_dist_arg]-lo_settings["ro_lo"])
except Exception as ex:
    qubit_parameters["ro_freq_opt"] = res_1_freq[max_dist_arg]-lo_settings["ro_lo"]
# qubit_parameters.update_parameter("ro_freq_opt", 6.893e9 - lo_settings["ro_lo"])

print(f'ro freq opt: {(qubit_parameters["ro_freq_opt"] + lo_settings["ro_lo"]) * 1e-9:.6f} GHz')

In [ ]:
Data_gef_spec = {'ground_res': res_0_res,
                'ground_freq': res_0_freq,
                'fst_ex_res': res_1_res,
                'fst_ex_freq': res_1_freq,
                # 'scnd_ex_res': res_2_res,
                # 'scnd_ex_freq': res_2_freq,
            }

Data_gef_spec.update(qubit_parameters._params)
Data_gef_spec.update(lo_settings._params)

file_name = 'ge_spec_readout_test_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_gef_spec)

In [ ]:
qubit_parameters.update_parameter("relax", 120e-6)

In [ ]:
#define optimal readout

readout_opt = {'readout_type': 'pulse',
                'readout_pulse': readout_pulse,
                'readout_weighting_function': readout_weighting_function,
                'relax_time': qubit_parameters["relax"],
                'measure_freq': qubit_parameters["ro_freq_opt"],
                'acquire_freq': qubit_parameters["ro_freq_opt"],
                'readout_range': -20,
                'readout_delay': qubit_parameters['ro_int_delay']
               }

# Single Shot 0 and 1 measurements

In [ ]:
print('Measure lenght', qubit_parameters["ro_len"])
print('Aq port delay', lsg_q0["acquire_line"].port_delay)

In [ ]:
#lsg_q0["acquire_line"].port_delay = 0.5e-07

In [ ]:
ro_len_test = 1e-6
ro_amp_test = 0.55

In [ ]:
readout_low['readout_pulse']

In [ ]:
readout_pulse_test = pulse_library.const(
    uid="readout_pulse",
    length=ro_len_test,
    amplitude=ro_amp_test,
)

readout_weighting_function_test = pulse_library.const(
            uid="readout_weighting_function", length=ro_len_test, amplitude=1.0
        )

readout_test = {'readout_type': 'pulses',
                'readout_pulse': readout_pulse_test,
                'readout_weighting_function': readout_weighting_function_test,
                'relax_time': qubit_parameters["relax"],
                'measure_freq': qubit_parameters["ro_freq_opt"],
                'acquire_freq': qubit_parameters["ro_freq_opt"],
                'readout_range': -20,
                'readout_delay': qubit_parameters['ro_int_delay']
               }

## One Single Shot measurement

In [ ]:
# qubit_parameters.update_parameter("relax", 100e-3)
# readout_long_wait = readout_low.copy()
readout_long_wait['relax_time'] = 500e-3

In [ ]:
# how many averages per point: 2^n_average
n_average = 17

#make two-point sweep
sweep_2 = make_two_point_sweep()

exp_rabi_SS = create_rabi_SS(sweep_2, 
               x180, 
               readout_opt,#readout_low,
               n_average)

# compile
exp_rabi_SS_comp = my_session.compile(exp_rabi_SS)

# show_pulse_sheet("Single Shot Test", exp_rabi_SS_comp)

In [ ]:
#run the experiment
SS_results = my_session.run(exp_rabi_SS_comp)

meas_ready()

In [ ]:
# get measurement data returned by the instruments
SS_res = SS_results.get_data("SS_rabi")

# define amplitude axis from qubit parameters
SS_it = SS_results.get_axis("SS_rabi")[0]
SS_amp = SS_results.get_axis("SS_rabi")[1]

In [ ]:
Data_SS = {'rabi_SS_res': SS_res,
           'rabi_SS_amp': SS_amp,
           'rabi_SS_it': SS_it
            }

Data_SS.update(qubit_parameters._params)
Data_SS.update(lo_settings._params)

file_name = 'Single_shots_all_unsh_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_SS)

In [ ]:
# how many averages per point: 2^n_average
n_average = 17

states = ['g', 'e', 'f']

exp_SS_comp_list = []

for state in states:
    exp_SS = create_SS(state, 
                            x180, 
                            x180_ef,
                            readout_opt,#readout_opt,#readout_low,
                            n_average)

    exp_SS_comp_list.append(my_session.compile(exp_SS))

In [ ]:
SS_results_states = []

for exp_comp in exp_SS_comp_list:
    SS_results_states.append(my_session.run(exp_comp))  

In [ ]:
result_states = []

for result in SS_results_states:
    result_states.append(result.get_data("shots"))

In [ ]:
result_arr = np.array(result_states)
result_arr.shape

In [ ]:
#for old way

# zero_data = SS_res[:,0]
# one_data = SS_res[:,1]

#for new way

zero_data = result_arr[0,:]
one_data = result_arr[1,:]
two_data = result_arr[2,:]

In [ ]:
# zero_data = SS_res[:,0]
# one_data = SS_res[:,1]

# plot measurement data on IQ
fig = plt.figure()
#plt.plot(rabi_amp, abs(rabi_res), ".k")
plt.plot(zero_data.real, zero_data.imag, '.', alpha=0.1)
#plt.plot(rabi_SS_res[:,15].real, rabi_SS_res[:,15].imag, '.')
plt.plot(one_data.real, one_data.imag, '.', alpha=0.1)
plt.plot(two_data.real, two_data.imag, '.', alpha=0.01)
plt.ylabel("Imaginary part")
plt.xlabel("Real part")

plt.xlim(-0.1, 0.12)
plt.ylim(-0.25, 0.05)

plt.plot(np.mean(zero_data.real), np.mean(zero_data.imag), 'bo')
plt.plot(np.mean(one_data.real), np.mean(one_data.imag), 'ro')
plt.plot(np.mean(two_data.real), np.mean(two_data.imag), 'yo')

# figname = 'Single_shot_points_'
# file_path = get_path_to_file(figname, '.png')
# plt.savefig(file_path, dpi=600, format='png', metadata=None,
#         bbox_inches=None, pad_inches=0.1,
#         facecolor='auto', edgecolor='auto',
#         backend=None,
#        )

In [ ]:
def analize_Single_Shots(SS_results, plot = False, n_bins = 50):
    SS_res = SS_results.get_data("SS_rabi")

    print(SS_res)

    Data_0 = SS_res[:,0]
    Data_1 = SS_res[:,1]

    zero_mean = np.mean(Data_0)
    one_mean = np.mean(Data_1)

    mid_point = 0.5*(zero_mean+one_mean)

    Data_0_corr = Data_0 - mid_point
    Data_1_corr = Data_1 - mid_point

    zero_mean_corr = zero_mean - mid_point

    r = np.abs(zero_mean-one_mean)/2

    phi = np.angle(zero_mean_corr)

    Data_0_rot = Data_0_corr*np.exp(1j*(np.pi-phi))
    Data_1_rot = Data_1_corr*np.exp(1j*(np.pi-phi))

    M_z = np.mean(Data_0_rot)
    M_o = np.mean(Data_1_rot)

    STD_x_0 = np.sqrt(np.var(Data_0_rot.real, ddof=1))
    STD_y_0 = np.sqrt(np.var(Data_0_rot.imag, ddof=1))
    
    STD_x_1 = np.sqrt(np.var(Data_1_rot.real, ddof=1))
    STD_y_1 = np.sqrt(np.var(Data_1_rot.imag, ddof=1))

    print('Number of posive elements:', np.sum(np.array(Data_0_rot.real) > 0))
    print('Number of negative elements:', np.sum(np.array(Data_0_rot.real) <= 0))

    Results = {'distance': r,
               'mean_0': M_z,
               'mean_1': M_o,
               'STD_x_0': STD_x_0,
               'STD_y_0': STD_y_0,
               'STD_x_1': STD_x_1,
               'STD_y_1': STD_y_1,
               'Rel_STD_0': STD_x_0/r,
               'Rel_STD_1': STD_x_1/r
              }
    if plot:
        n_bins = 50

        fig, axs = plt.subplots(1, 2, sharey=True, tight_layout=True)

        # We can set the number of bins with the *bins* keyword argument.
        axs[0].title.set_text('Real')
        axs[0].hist(Data_0_rot.real, bins=n_bins, alpha=0.5)
        axs[0].hist(Data_1_rot.real, bins=n_bins, alpha=0.5)
        axs[0].set_yscale('log')

        axs[1].title.set_text('Imag')
        axs[1].hist(Data_0_rot.imag, bins=n_bins, alpha=0.5)
        axs[1].hist(Data_1_rot.imag, bins=n_bins, alpha=0.5)
        axs[1].set_yscale('log')
    
    return Results

In [ ]:
res = analize_Single_Shots(SS_results, plot = True)

In [ ]:
res2 = analize_Single_Shots(SS_results, plot = True)

In [ ]:
res

In [ ]:
#plot distributions

n_bins = 50

fig, axs = plt.subplots(1, 2, sharey=True, tight_layout=True)

# We can set the number of bins with the *bins* keyword argument.
axs[0].title.set_text('Real')
axs[0].hist(zero_data.real, bins=n_bins)
axs[0].hist(one_data.real, bins=n_bins)

axs[1].title.set_text('Imag')
axs[1].hist(zero_data.imag, bins=n_bins)
axs[1].hist(one_data.imag, bins=n_bins)

# figname = 'Single_shot_hist_'
# file_path = get_path_to_file(figname, '.png')
# plt.savefig(file_path, dpi=600, format='png', metadata=None,
#         bbox_inches=None, pad_inches=0.1,
#         facecolor='auto', edgecolor='auto',
#         backend=None,
#        )

In [ ]:
n_bins =200
plt.hist(zero_data_proj/r, bins=n_bins, alpha=0.5)
plt.hist(one_data_proj/r, bins=n_bins, alpha=0.5)
plt.axvline(1)
plt.axvline(-1)
plt.yscale('log')
plt.ylabel('N points')
plt.xlabel('Relative distance')

In [ ]:
data = zero_data_proj/r

n, bins, patches = plt.hist(data, bins=n_bins, alpha=0.5, density=True)

(mu, sigma) = norm.fit(data)

# add a 'best fit' line
y = norm.pdf(bins, mu, sigma)
l = plt.plot(bins, y, 'r--', linewidth=2)
plt.yscale('log')

In [ ]:
data = one_data_proj/r

y, x, patches = plt.hist(data, bins=n_bins, color='red', alpha=0.25)
x=(x[1:]+x[:-1])/2

expected = (-1, .2, 1050, 1, .2, 125)
params, cov = curve_fit(bimodal, x, y, expected)
sigma=np.sqrt(np.diag(cov))
x_fit = np.linspace(x.min(), x.max(), 500)
#plot combined...
plt.plot(x_fit, bimodal(x_fit, *params), color='green', lw=3, label='One+Zero')
#...and individual Gauss curves
plt.plot(x_fit, gauss(x_fit, *params[:3]), color='red', lw=2, ls="--", label='One')
plt.plot(x_fit, gauss(x_fit, *params[3:]), color='b', lw=2, ls=":", label='Zero')
#and the original data points if no histogram has been created before
#plt.scatter(x, y, marker="X", color="black", label="original data")
plt.yscale('log')
plt.ylabel('N points')
plt.xlabel('Relative distance')
plt.ylim(1e-1, 5e3)
plt.legend()
print(pd.DataFrame(data={'params': params, 'sigma': sigma}, index=bimodal.__code__.co_varnames[1:]))
plt.show() 

print('Area ratio:', params[4]*params[5]/(params[1]*params[2]))

In [ ]:
data = zero_data_proj/r

y, x, patches = plt.hist(data, bins=n_bins, color='blue', alpha=0.25)
x=(x[1:]+x[:-1])/2

expected = (-1.5, 1.0, 250, 1, 1.0, 2025)
params, cov = curve_fit(bimodal, x, y, expected)
sigma=np.sqrt(np.diag(cov))
x_fit = np.linspace(x.min(), x.max(), 500)
#plot combined...
plt.plot(x_fit, bimodal(x_fit, *params), color='green', lw=3, label='One+Zero')
#...and individual Gauss curves
plt.plot(x_fit, gauss(x_fit, *params[:3]), color='red', lw=2, ls="--", label='One')
plt.plot(x_fit, gauss(x_fit, *params[3:]), color='blue', lw=2, ls=":", label='Zero')
#and the original data points if no histogram has been created before
#plt.scatter(x, y, marker="X", color="black", label="original data")
plt.yscale('log')
plt.ylabel('N points')
plt.xlabel('Relative distance')
plt.ylim(1e-1, 5e3)
plt.legend()
print(pd.DataFrame(data={'params': params, 'sigma': sigma}, index=bimodal.__code__.co_varnames[1:]))
plt.show() 

print('Area ratio:', params[1]*params[2]/(params[4]*params[5]))

In [ ]:
compute_pca(zero_data.real, zero_data.imag, plot=True, scaling_fac=0.1)

pca_ss_dict = compute_pca(one_data.real, one_data.imag, plot=True, scaling_fac=0.1)

In [ ]:
pca_ss_dict

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis




In [ ]:
n_bins = 50

fig, ax = plt.subplots(tight_layout=True)
hist = ax.hist2d(zero_data.real, zero_data.imag, n_bins)
hist = ax.hist2d(one_data.real, one_data.imag, n_bins)

## Single Shot measurement for different Rabi frequences

In [ ]:
#Sweep for different pi-pulses
n_average = 17

rabi_SS_sweep_list = []

pulse_length = np.linspace(60e-9, 400e-9, 36)
rabi_freq = 1/pulse_length
pi_amp = qubit_parameters['rabi_slope']*rabi_freq*1e-6+qubit_parameters['rabi_intercept']

for i in range(len(pulse_length)):
    
    sweep_2 = make_two_point_sweep(stop=pi_amp[i])

    gaussian_pulse = pulse_library.gaussian(
        uid="gaussian_pulse", length=pulse_length[i], amplitude=1.0
    )
    
    exp_rabi_SS = make_rabi_SS(sweep_2, 
                     gaussian_pulse, 
                     readout_pulse, 
                     readout_weighting_function, 
                     qubit_parameters["relax"], 
                     n_average)
    
    exp_rabi_SS.set_signal_map(qubit_meas_map)
    exp_rabi_SS_comp = my_session.compile(exp_rabi_SS)
    results_SS = my_session.run(exp_rabi_SS_comp)
    
    rabi_SS_sweep_list.append(results_SS.get_data("SS_rabi"))
    
rabi_SS_sweep_arr = np.array(rabi_SS_sweep_list)

In [ ]:
pi_amp

In [ ]:
rabi_freq*1e-6

In [ ]:
Data_SS = {'rabi_SS_sweep_arr': rabi_SS_sweep_arr,
           'rabi_SS_amp': SS_amp,
           'rabi_SS_it': SS_it,
           'pulse_length': pulse_length,
           'rabi_freq': rabi_freq,
           'pi_amp': pi_amp
            }

Data_SS.update(qubit_parameters)
Data_SS.update(lo_settings)

file_name = 'Single_shots_0_pi_diff_pipulses_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_SS)

## Single Shot measurement for different aqusition lengths and delays

In [ ]:
print('Measure lenght', readout_pulse.length)
print('Aq port delay', lsg_q0["acquire_line"].port_delay)

In [ ]:
# Here the readout pulse amplitude and length are fixed and we vary the delay and window length

In [ ]:
#Sweep for different readout length
n_average = 17

rabi_SS_sweep_list = []

#ro_len = np.linspace(100, 2000, 20)*1e-9

port_delay_sw = np.linspace(380, 400, 5)*1e-9

for k in range(len(port_delay_sw)):
    print('Delay: ', port_delay_sw[k]) 
    
    rabi_SS_sweep_list_int = []
    
    lsg_q0["acquire_line"].port_delay = port_delay_sw[k]
    
    #ro_len = np.arange(100, 2100 - port_delay_sw[k], 100)*1e-9
    ro_len = [readout_pulse.length]

    for i in range(len(ro_len)):

        print('Ro_len: ', ro_len[i])
    
        readout_weighting_function_i = pulse_library.const(
            uid="readout_weighting_function", length=ro_len[i], amplitude=1.0
        )

        exp_rabi_SS = make_rabi_SS(sweep_2, 
                         x180, 
                         readout_pulse, 
                         readout_weighting_function_i, 
                         qubit_parameters["relax"], 
                         n_average)
    
        exp_rabi_SS.set_signal_map(qubit_meas_map)
        exp_rabi_SS_comp = my_session.compile(exp_rabi_SS)
        results_SS = my_session.run(exp_rabi_SS_comp)

        res = analize_Single_Shots(results_SS, plot = False)
    
        rabi_SS_sweep_list_int.append(res)
    
    rabi_SS_sweep_list.append(rabi_SS_sweep_list_int)
    

In [ ]:
readout_pulse.amplitude

In [ ]:
ro_len

In [ ]:
min_val = []
argmin_val = []

for i in range(len(rabi_SS_sweep_list)):
    l = [d['Rel_STD_0'] for d in rabi_SS_sweep_list[i]]
    min_val.append(np.min(np.array(l)))
    argmin_val.append(np.argmin(np.array(l)))

MM = np.argmin(np.array(min_val))
#ro_len = np.arange(100, 2100 - port_delay_sw[MM], 100)*1e-9
ro_len = [readout_pulse.length]

print('Readout length is', readout_pulse.length)
print('Minimal relative error value is ', np.round(min_val[MM], 3), 'for delay', np.round(port_delay_sw[MM]*1e9,3), 'ns (index', MM, ') and length', ro_len[argmin_val[MM]], '(index', argmin_val[MM], ').')
print('Optimal port delay is between ', np.round(port_delay_sw[MM-1]*1e9,3), ' and ', np.round(port_delay_sw[MM+1]*1e9,3), 'ns')

In [ ]:
print('Optimal port delay is between ', np.round(port_delay_sw[1]*1e9,3), ' and ', np.round(port_delay_sw[2]*1e9,3), 'ns')

In [ ]:
lsg_q0["acquire_line"].port_delay = 3.8e-7

In [ ]:
#Sweep power and length of the readout pulse. Here delay is fixed and window is equal to pulse length

#Sweep for different readout length
n_average = 17

rabi_SS_sweep_list = []

#ro_len = np.linspace(100, 2000, 20)*1e-9

ro_amp_sw = np.linspace(0.05, 0.8, 21)
ro_len_sw = np.linspace(100, 2000, 10)*1e-9

for k in range(len(ro_amp_sw)):
    print('RO Amplitude: ', ro_amp_sw[k]) 
    
    rabi_SS_sweep_list_int = []
    
    readout_pulse.amplitude = ro_amp_sw[k]

    for i in range(len(ro_len_sw)):

        print('Ro_len: ', ro_len_sw[i])

        readout_pulse.length = ro_len_sw[i]
    
        readout_weighting_function_i = pulse_library.const(
            uid="readout_weighting_function", length=ro_len_sw[i], amplitude=1.0
        )

        exp_rabi_SS = make_rabi_SS(sweep_2, 
                         x180, 
                         readout_pulse, 
                         readout_weighting_function_i, 
                         qubit_parameters["relax"], 
                         n_average)
    
        exp_rabi_SS.set_signal_map(qubit_meas_map)
        exp_rabi_SS_comp = my_session.compile(exp_rabi_SS)
        results_SS = my_session.run(exp_rabi_SS_comp)

        res = analize_Single_Shots(results_SS, plot = False)
    
        rabi_SS_sweep_list_int.append(res)
    
    rabi_SS_sweep_list.append(rabi_SS_sweep_list_int)

In [ ]:
min_val = []
argmin_val = []

for i in range(len(rabi_SS_sweep_list)):
    l = [d['Rel_STD_0'] for d in rabi_SS_sweep_list[i]]
    min_val.append(np.min(np.array(l)))
    argmin_val.append(np.argmin(np.array(l)))

MM = np.argmin(np.array(min_val))
#ro_len = np.arange(100, 2100 - port_delay_sw[MM], 100)*1e-9
#ro_len = [readout_pulse.length]

#print('Readout length is', readout_pulse.length)
print('Minimal relative error value is ', np.round(min_val[MM], 3), 'for amplitude', np.round(ro_amp_sw[MM],3), ' (index', MM, ') and length', ro_len_sw[argmin_val[MM]], '(index', argmin_val[MM], ').')
#print('Optimal port delay is between ', np.round(port_delay_sw[MM-1]*1e9,3), ' and ', np.round(port_delay_sw[MM+1]*1e9,3), 'ns')

In [ ]:
for i in range(len(rabi_SS_sweep_list)):
    l = [d['Rel_STD_0'] for d in rabi_SS_sweep_list[:][i]]
    plt.plot(ro_amp_sw, np.array(i))

In [ ]:
STD_list = []
for l in rabi_SS_sweep_list:
    STD_list.append([d['Rel_STD_0'] for d in l])

STD_array = np.array(STD_list)

for k in range(len(ro_len_sw)):
    plt.plot(ro_amp_sw, STD_array[:,k])

plt.yscale('log')

In [ ]:
len(rabi_SS_sweep_list[6])

In [ ]:
[d['Rel_STD_0'] for d in rabi_SS_sweep_list[17]]

In [ ]:
# get measurement data returned by the instruments
rabi_SS_res = rabi_SS_sweep_arr[19,19, :, :]

In [ ]:
Data_SS = {'rabi_SS_sweep_arr': rabi_SS_sweep_arr,
           'rabi_SS_amp': SS_amp,
           'rabi_SS_it': SS_it,
           'port_delay_sw': port_delay_sw,
           'ro_len': ro_len,
           'n_av': n_average,
           'comment': 'Sweep delay and ro len, array dim [delay, ro_len, n_av, 0 or 1]'
            }

Data_SS.update(qubit_parameters)
Data_SS.update(lo_settings)

file_name = 'Single_shots_0_pi_sweep_delay_and_ro_len_'
file_path = get_path_to_file(file_name, '.mat')
savemat(file_path, Data_SS)

# Population measurements

## Rabi Population Measurements (RPM)

### Setup

#### Oscillations Check

In [ ]:
# range of ef pulse amplitude scan
amp_min = 0
amp_max = 1
# how many amplitude points to measure
amp_num = 51

# set up sweep parameter - qubit drive pulse amplitude
theta_ef_sweep = LinearSweepParameter(
    uid="rabi_ef_amp", start=amp_min, stop=amp_max, count=amp_num
)

qubit_parameters["qb_ef_len_qrpm"] = qubit_parameters["qb_ef_len"]

# how many averages per point: 2^n_average
n_average = 16

In [ ]:
gaussian_pulse_qrpm = pulse_library.gaussian(
    uid="gaussian_pulse", length=qubit_parameters["qb_ef_len_qrpm"], amplitude=1.0
)

In [ ]:
detuning_ef = 0

In [ ]:
# qubit drive frequency - defined in calibration on device setup as baseline reference
drive_ef_Oscillator_q0.frequency = qubit_parameters["qb_ef_freq"] + detuning_ef
# set oscillator type to hardware to ensure optimal use of the instrument functionality
drive_ef_Oscillator_q0.modulation_type = ModulationType.HARDWARE

In [ ]:
rpm_exp = get_rabi_population_calibration_measurement(theta_ef_sweep=theta_ef_sweep, gaussian_pulse=gaussian_pulse_qrpm, 
                                          readout_pulse=readout_pulse, 
                                          readout_weighting_function=readout_weighting_function,
                                          relax_time=qubit_parameters["relax"], n_average=n_average, x180=x180)

In [ ]:
# signal map for qubit 0
rpm_map = {
    "drive": "/logical_signal_groups/q0/drive_line",
    "drive_ef": "/logical_signal_groups/q0/drive_ef_line",
    "measure": "/logical_signal_groups/q0/measure_line",
    "acquire": "/logical_signal_groups/q0/acquire_line",
}

In [ ]:
rpm_exp.set_signal_map(rpm_map)

In [ ]:
rpm_exp_comp = my_session.compile(rpm_exp)

In [ ]:
my_results = my_session.run(rpm_exp_comp)
# rpm_res_loop = []
# for i in range(25):
#     my_results = my_session.run(rpm_exp_comp)
#     rpm_res = {}
#     for key in ['with', 'wo']:
#         rpm_res[key] = {'amp': my_results.get_axis(f"rpm_{key}")[0],
#                         'mag': np.abs(my_results.get_data(f"rpm_{key}")),
#                         'pha': np.unwrap(np.angle(my_results.get_data(f"rpm_{key}")))}
#     rpm_res_loop.append(rpm_res)

In [ ]:
# get measurement data returned by the instruments
rpm_res = {}
for key in ['with', 'wo']:
    rpm_res[key] = {'amp': my_results.get_axis(f"rpm_{key}")[0],
                    'mag': np.abs(my_results.get_data(f"rpm_{key}")),
                    'pha': np.unwrap(np.angle(my_results.get_data(f"rpm_{key}")))}

# rpm_res = rpm_res_loop[0]

In [ ]:
rpm_popt, rpm_pcov = {'with': {}, 'wo': {}}, {'with': {}, 'wo': {}}

freq_estimate = 8
 
rpm_popt['with']['mag'], rpm_pcov['with']['mag'] = fit_Rabi(x=rpm_res['with']['amp'], y=rpm_res['with']['mag'], 
                                                            freq=freq_estimate, phase=1.0, amp=1.0, off=0.018, plot=False)
rpm_popt['with']['pha'], rpm_pcov['with']['mag'] = fit_Rabi(x=rpm_res['with']['amp'], y=rpm_res['with']['pha'], 
                                                            freq=freq_estimate, phase=1.0, amp=0.01, off=-0.12, plot=False)

rpm_popt['wo']['mag'], rpm_pcov['wo']['mag'] = fit_Rabi(x=rpm_res['wo']['amp'], y=rpm_res['wo']['mag'], 
                                                        freq=freq_estimate, phase=1.0, amp=1.0, off=0.018, plot=False)
rpm_popt['wo']['pha'], rpm_pcov['wo']['mag'] = fit_Rabi(x=rpm_res['wo']['amp'], y=rpm_res['wo']['pha'], 
                                                        freq=freq_estimate, phase=1.0, amp=0.01, off=-0.12, plot=False)

rpm_amp_plot = np.linspace(rpm_res['with']['amp'][0], rpm_res['with']['amp'][-1], int(rpm_res['with']['amp'].shape[0] * 20))

In [ ]:
rpm_fig, rpm_ax = plt.subplots(6, 1, figsize=(10, 18), sharex=True)
rpm_ax[0].scatter(rpm_res['with']['amp'], rpm_res['with']['mag'])
rpm_ax[0].set_ylabel('Amplitude with pre-pulse')
rpm_ax[0].plot(rpm_amp_plot, func_osc(rpm_amp_plot, *rpm_popt['with']['mag']), color='tab:red', linestyle='dashed')

rpm_ax[1].scatter(rpm_res['wo']['amp'], rpm_res['wo']['mag'])
rpm_ax[1].set_ylabel('Amplitude without pre-pulse')
rpm_ax[1].plot(rpm_amp_plot, func_osc(rpm_amp_plot, *rpm_popt['wo']['mag']), color='tab:red', linestyle='dashed')

rpm_ax[2].scatter(rpm_res['with']['amp'], rpm_res['with']['pha'])
rpm_ax[2].set_ylabel('Phase with pre-pulse')
rpm_ax[2].plot(rpm_amp_plot, func_osc(rpm_amp_plot, *rpm_popt['with']['pha']), color='tab:red', linestyle='dashed')

rpm_ax[3].scatter(rpm_res['wo']['amp'], rpm_res['wo']['pha'])
rpm_ax[3].set_ylabel('Phase without pre-pulse')
rpm_ax[3].plot(rpm_amp_plot, func_osc(rpm_amp_plot, *rpm_popt['wo']['pha']), color='tab:red', linestyle='dashed')

_, offset, norm = normalize_1d_osc(rpm_res['with']['mag'])

rpm_ax[4].scatter(rpm_res['with']['amp'], normalize_1d_osc(rpm_res['with']['mag'])[0], label='with', color='salmon')
rpm_ax[4].plot(rpm_amp_plot, normalize_1d_osc(func_osc(rpm_amp_plot, *rpm_popt['with']['mag']), offset, norm)[0], color='tab:red', linestyle='dashed')

rpm_ax[4].scatter(rpm_res['wo']['amp'], normalize_1d_osc(rpm_res['wo']['mag'], norm=norm)[0], label='w/o', color='skyblue')
rpm_ax[4].set_ylabel('Amplitude (normalized)')
rpm_ax[4].plot(rpm_amp_plot, normalize_1d_osc(func_osc(rpm_amp_plot, *rpm_popt['wo']['mag']), norm=norm)[0], color='tab:blue', linestyle='dashed')
rpm_ax[4].legend()

_, offset, norm = normalize_1d_osc(rpm_res['with']['pha'])

rpm_ax[5].scatter(rpm_res['with']['amp'], normalize_1d_osc(rpm_res['with']['pha'])[0], label='with', color='salmon')
rpm_ax[5].plot(rpm_amp_plot, normalize_1d_osc(func_osc(rpm_amp_plot, *rpm_popt['with']['pha']), offset, norm)[0], color='tab:red', linestyle='dashed')

rpm_ax[5].scatter(rpm_res['wo']['amp'], normalize_1d_osc(rpm_res['wo']['pha'], norm=norm)[0], label='w/o', color='skyblue')
rpm_ax[5].set_ylabel('Phase (normalized)')
rpm_ax[5].plot(rpm_amp_plot, normalize_1d_osc(func_osc(rpm_amp_plot, *rpm_popt['wo']['pha']), norm=norm)[0], color='tab:blue', linestyle='dashed')
rpm_ax[5].legend()

rpm_ax[-1].set_xlabel('ef-amplitude')

# rpm_fig.savefig(get_path_to_file('RPM_full_amp_sweep_loop_10', '.pdf'), bbox_inches='tight')



In [ ]:
qrpm_dict = get_qrpm_dict_from_list_of_results([rpm_res])

#### Pi Pulse Check

In [ ]:
# extract min max pulses
signal_in_mag = True

if signal_in_mag:
    fit_params_with = rpm_popt['with']['mag']
    fit_params_wo = rpm_popt['wo']['mag']
    with_dat = rpm_res['with']['mag']
    wo_dat = rpm_res['wo']['mag']
    ylabel = 'Amplitude'
    
else:
    fit_params_with = rpm_popt['with']['pha']
    fit_params_wo = rpm_popt['wo']['pha']
    with_dat = rpm_res['with']['pha']
    wo_dat = rpm_res['wo']['pha']
    ylabel = 'Phase'

# amp_with = (fit_params_with[1]) / fit_params_with[0]
# amp_wo = (fit_params_wo[1]) / fit_params_wo[0]

# amp_with = rpm_amp_plot[np.argmax(func_osc(rpm_amp_plot, *fit_params_with))]

amp_with = 0.22
amp_wo = amp_with

amp_avg = (amp_with + amp_wo) / 2

cal_fig, cal_axs = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

cal_axs[0].scatter(rpm_res['wo']['amp'], with_dat)
cal_axs[0].plot(rpm_amp_plot, func_osc(rpm_amp_plot, *fit_params_with), color='tab:red', linestyle='dashed')
cal_axs[0].vlines([amp_with], with_dat.min(), with_dat.max(), color='black', linestyle='dashed', label='with')
cal_axs[0].vlines([amp_avg], with_dat.min(), with_dat.max(), color='tab:green', linestyle='dashed', label='avg')
cal_axs[0].vlines([amp_wo], with_dat.min(), with_dat.max(), color='tab:purple', linestyle='dashed', label='wo')
cal_axs[0].legend()

cal_axs[0].set_ylabel(ylabel)
cal_axs[0].set_title('with ge pre pulse')

cal_axs[1].scatter(rpm_res['wo']['amp'], wo_dat)
cal_axs[1].plot(rpm_amp_plot, func_osc(rpm_amp_plot, *fit_params_wo), color='tab:red', linestyle='dashed')
cal_axs[1].vlines([amp_with], wo_dat.min(), wo_dat.max(), color='black', linestyle='dashed', label='with')
cal_axs[1].vlines([amp_avg], wo_dat.min(), wo_dat.max(), color='tab:green', linestyle='dashed', label='avg')
cal_axs[1].vlines([amp_wo], wo_dat.min(), wo_dat.max(), color='tab:purple', linestyle='dashed', label='wo')
cal_axs[1].legend()

cal_axs[1].set_xlabel('ef amplitude')
cal_axs[1].set_ylabel(ylabel)
cal_axs[1].set_title('without ge pre pulse')

# cal_fig.savefig(get_path_to_file('RPM_x180_ef_calibration', '.pdf'), bbox_inches='tight')

In [ ]:
#set a an rpm pi_ef pulse
x180_ef_rpm = pulse_library.gaussian(
    uid="x180_ef", length=qubit_parameters["qb_ef_len_qrpm"], amplitude=amp_avg
)

qubit_parameters["pi_ef_amp_qrpm"] = amp_avg

### Population Measurement

In [ ]:
rpm_quick_exp = rpm_exp = get_quick_rabi_population_measurement(rpm_x180_ef_pulse=x180_ef_rpm, rep_count = 1,
                                                                gaussian_pulse=gaussian_pulse_qrpm, 
                                          readout_pulse=readout_pulse, 
                                          readout_weighting_function=readout_weighting_function,
                                          relax_time=qubit_parameters["relax"], n_average=17, x180=x180)

In [ ]:
# signal map for qubit 0
rpm_map = {
    "drive": "/logical_signal_groups/q0/drive_line",
    "drive_ef": "/logical_signal_groups/q0/drive_ef_line",
    "measure": "/logical_signal_groups/q0/measure_line",
    "acquire": "/logical_signal_groups/q0/acquire_line",
}

In [ ]:
rpm_exp.set_signal_map(rpm_map)

In [ ]:
rpm_quick_exp_comp = my_session.compile(rpm_quick_exp)

In [ ]:
my_results = my_session.run(rpm_quick_exp_comp)

In [ ]:
# get measurement data returned by the instruments
rpm_res = {}
for key in ['with', 'wo']:
    rpm_res[key] = {'mag_min': np.abs(my_results.get_data(f"rpm_{key}_min")),
                    'pha_min': np.angle(my_results.get_data(f"rpm_{key}_min")),
                    'mag_max': np.abs(my_results.get_data(f"rpm_{key}_max")),
                    'pha_max': np.angle(my_results.get_data(f"rpm_{key}_max")),
                    'I_max': np.real(my_results.get_data(f"rpm_{key}_max")),
                    'Q_max': np.imag(my_results.get_data(f"rpm_{key}_max")),
                    'I_min': np.real(my_results.get_data(f"rpm_{key}_min")),
                    'Q_min': np.imag(my_results.get_data(f"rpm_{key}_min"))}

In [ ]:
quick_rpm_fig_iq , quick_rpm_axs_iq = plt.subplots(2, 1, figsize=(10, 12))

quick_rpm_axs_iq[0].scatter(rpm_res['with']['I_max'], rpm_res['with']['Q_max'])
quick_rpm_axs_iq[0].scatter(rpm_res['with']['I_min'], rpm_res['with']['Q_min'])
quick_rpm_axs_iq[0].set_xlabel('I')
quick_rpm_axs_iq[0].set_ylabel('Q')
quick_rpm_axs_iq[0].set_title('Quick RPM Max')

quick_rpm_axs_iq[1].scatter(rpm_res['wo']['I_max'], rpm_res['wo']['Q_max'])
quick_rpm_axs_iq[1].scatter(rpm_res['wo']['I_min'], rpm_res['wo']['Q_min'])
quick_rpm_axs_iq[1].set_xlabel('I')
quick_rpm_axs_iq[1].set_ylabel('Q')
quick_rpm_axs_iq[1].set_title('Quick RPM Min')

In [ ]:
quick_rpm_fig, quick_rpm_axs = plt.subplots(4, 1, figsize=(10, 12), sharex=True)

y_min = np.repeat(0, rpm_res['with']['mag_min'].size)
y_max = np.repeat(x180_ef_rpm.amplitude, rpm_res['with']['mag_min'].size)

quick_rpm_axs[0].scatter(y_min, rpm_res['with']['mag_min'])
quick_rpm_axs[0].scatter(y_max, rpm_res['with']['mag_max'])
quick_rpm_axs[0].scatter(np.mean(y_min), np.mean(rpm_res['with']['mag_min']), marker='x', color='black', s=200)
quick_rpm_axs[0].scatter(np.mean(y_max), np.mean(rpm_res['with']['mag_max']), marker='x', color='black', s=200)
quick_rpm_axs[0].plot(rpm_amp_plot, func_osc(rpm_amp_plot, *rpm_popt['with']['mag']), color='tab:red', linestyle='dashed')

ti = 9
quick_rpm_axs[1].scatter(y_min, rpm_res['wo']['mag_min'])
quick_rpm_axs[1].scatter(y_max, rpm_res['wo']['mag_max'])
quick_rpm_axs[1].scatter(np.mean(y_min), np.mean(rpm_res['wo']['mag_min']), marker='x', color='black', s=200)
quick_rpm_axs[1].scatter(np.mean(y_max), np.mean(rpm_res['wo']['mag_max']), marker='x', color='black', s=200)
quick_rpm_axs[1].plot(rpm_amp_plot, func_osc(rpm_amp_plot, *rpm_popt['wo']['mag']), color='tab:red', linestyle='dashed')

quick_rpm_axs[2].scatter(y_min, rpm_res['with']['pha_min'])
quick_rpm_axs[2].scatter(y_max, rpm_res['with']['pha_max'])
quick_rpm_axs[2].scatter(np.mean(y_min), np.mean(rpm_res['with']['pha_min']), marker='x', color='black', s=200)
quick_rpm_axs[2].scatter(np.mean(y_max), np.mean(rpm_res['with']['pha_max']), marker='x', color='black', s=200)
quick_rpm_axs[2].plot(rpm_amp_plot, func_osc(rpm_amp_plot, *rpm_popt['with']['pha']), color='tab:red', linestyle='dashed')

quick_rpm_axs[3].scatter(y_min, rpm_res['wo']['pha_min'])
quick_rpm_axs[3].scatter(y_max, rpm_res['wo']['pha_max'])
quick_rpm_axs[3].scatter(np.mean(y_min), np.mean(rpm_res['wo']['pha_min']), marker='x', color='black', s=200)
quick_rpm_axs[3].scatter(np.mean(y_max), np.mean(rpm_res['wo']['pha_max']), marker='x', color='black', s=200)
quick_rpm_axs[3].plot(rpm_amp_plot, func_osc(rpm_amp_plot, *rpm_popt['wo']['pha']), color='tab:red', linestyle='dashed')

for ax, ylabel, title in zip(quick_rpm_axs, 
                             ['Amplitude', 'Amplitude', 'Phase', 'Phase'], 
                             ['with ge pre-pulse', 'without ge pre-pulse', 'with ge pre-pulse', 'without ge pre-pulse']):
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    
    
amps_with_mag = np.abs(rpm_res['with']['mag_min'] - rpm_res['with']['mag_max'])
amps_with_pha = np.abs(rpm_res['with']['pha_min'] - rpm_res['with']['pha_max'])

amps_wo_mag = np.abs(rpm_res['wo']['mag_min'] - rpm_res['wo']['mag_max'])
amps_wo_pha = np.abs(rpm_res['wo']['pha_min'] - rpm_res['wo']['pha_max'])

mean_amps_with_mag = np.mean(amps_with_mag)
mean_amps_with_pha = np.mean(amps_with_pha)

mean_amps_wo_mag = np.mean(amps_wo_mag)
mean_amps_wo_pha = np.mean(amps_wo_pha)

In [ ]:
max_iq_with = np.column_stack((rpm_res['with']['I_max'], rpm_res['with']['Q_max']))
min_iq_with = np.column_stack((rpm_res['with']['I_min'], rpm_res['with']['Q_min']))
iq_amp_with = np.linalg.norm(max_iq_with-min_iq_with, axis=1)

max_iq_wo = np.column_stack((rpm_res['wo']['I_max'], rpm_res['wo']['Q_max']))
min_iq_wo = np.column_stack((rpm_res['wo']['I_min'], rpm_res['wo']['Q_min']))
iq_amp_wo = np.linalg.norm(max_iq_wo-min_iq_wo, axis=1)


iq_amp_fig, iq_amp_axs = plt.subplots(2, 1, figsize=(10, 8))
iq_amp_axs[0].plot(iq_amp_with, marker='o')
iq_amp_axs[1].plot(iq_amp_wo, marker='o')

### Temperature 

In [ ]:
# rpm_pe = amps_wo_mag / (amps_with_mag + amps_wo_mag)
rpm_pe = iq_amp_wo / (iq_amp_wo + iq_amp_with)

rpm_temp = T_calc(C=rpm_pe / (1-rpm_pe), f_q = (qubit_parameters["qb_freq"] + lo_settings['qb_lo']) * 1e-9)

quick_rpm_temps_fig, quick_rpm_temps_axs = plt.subplots(figsize=(15, 5))
quick_rpm_temps_axs.plot(np.arange(len(rpm_pe)), rpm_temp, marker='o')
quick_rpm_temps_axs.hlines([np.mean(rpm_temp)], 0, len(rpm_pe), color='black', linestyle='dashed', label='average')
quick_rpm_temps_axs.set_ylabel('temp (K)')
quick_rpm_temps_axs.set_xlabel('Measurement')
quick_rpm_temps_axs.set_title('Quick Rabi Population Measurement')

quick_rpm_temps_hist_fig, quick_rpm_temps_hist_ax = plt.subplots()
quick_rpm_temps_hist_ax.hist(rpm_temp, bins=75)
quick_rpm_temps_hist_ax.set_xlabel('Temperature')
quick_rpm_temps_hist_ax.set_ylabel('Counts')
quick_rpm_temps_hist_ax.set_title('Quick Rabi Population Measurement')

## Three level population measurements

### Single-point experiment

In [ ]:
n_average = 15

exp_population_full = create_exp_population_full(x180, 
                                                 x180_ef, 
                                                 readout_opt,# readout_low,
                                                 n_average)

#compile experiment
exp_population_full_comp = my_session.compile(exp_population_full)

#show_pulse_sheet("Population", exp_population_full_comp)

In [ ]:
#run experiment
results_population_full = my_session.run(exp_population_full_comp)

In [ ]:
print(results_population_full.get_data('x0_readout'))
print(results_population_full.get_data('x1_readout'))

### Population statistic

In [ ]:
#function for recolculation of population to effective temperature

f_q = (lo_settings["qb_lo"] + qubit_parameters["qb_freq"])*1e-9

def T_calc(C, f_q):
	h = 6.628*1E-2
	k = 1.381
	return -(h*f_q/k)/np.log(C)

In [ ]:
f_q

In [ ]:
T_calc(0.082, f_q)

In [ ]:
N = 3 #Number of measurements

#lists for data storage
pop_full_results = {}
labels = ['x0', 'x1', 'x2', 'y0', 'y1', 'y2']

for i in range(N):
    print('Iteration: ', i)
    results_population_full = my_session.run(exp_population_full_comp)
    for k in labels:
        if i == 0:
            pop_full_results[k] = [results_population_full.get_data(handle=f'{k}_readout')]
        else:
            pop_full_results[k].append(results_population_full.get_data(handle=f'{k}_readout'))
            

for k in labels:
    pop_full_results[k] = np.array(pop_full_results[k])

In [ ]:
#fast temperature extraction
TABC_I, TABC_Q = make_all_temperatures(pop_full_results, f_q, qubit_parameters['qb_anharm']*1e-9)


#plot the results
#skip = ['TA3', 'TB3', 'TC3']
skip = ['TC2']

plot_temperatures(TABC_I, TABC_Q, skip = skip)

# #Save figure
# figname = 'Population_stat_read_test_'
# file_path = get_path_to_file(figname, '.png', sample_parameters)
# plt.savefig(file_path, dpi=600, format='png', metadata=None,
#         bbox_inches=None, pad_inches=0.1,
#         facecolor='auto', edgecolor='auto',
#         backend=None,
#        )

In [ ]:
STAT_I = get_stat(TABC_I)
STAT_Q = get_stat(TABC_Q)

plot_stat(STAT_I, STAT_Q)

In [ ]:
#Population 3lvl method, proper rotation

#Tmxc_str = stretch_by_N(Tmix_N, 5)

ABC = get_ABC_parallel(pop_full_results)
TABC = get_temperature(ABC, f_q, qubit_parameters['qb_anharm']*1e-9, three_levels = True)

#skip = ['TA3', 'TB3', 'TC3']
# skip = ['TC2', 'TC1']

plot_temp_single(TABC, skip = skip)

In [ ]:
STAT = get_stat(TABC)

plot_stat(STAT, STAT)

In [ ]:
for k in pop_full_results.keys():
    plt.plot(pop_full_results[k].real, pop_full_results[k].imag, label = k)

plt.legend()

In [ ]:
qubit_parameters

In [ ]:
#Save the data!

Data_T_stat = {}

Data_T_stat.update(pop_full_results)
# Data_T_stat.update(TABC_I)
# Data_T_stat.update(TABC_Q)
Data_T_stat.update(qubit_parameters._params)
Data_T_stat.update(lo_settings._params)

file_name = 'Teff_stat_10p_all_unsh_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_T_stat)

In [ ]:
plt.plot(T_A_real, '-o', label = 'A_Q', color = 'red')
plt.plot(T_A_imag, '-D', mfc = 'w', label = 'A_I', color = 'red')

plt.plot(T_B_real, '-o', label = 'B_Q', color = 'blue')
plt.plot(T_B_imag, '-D', mfc = 'w', label = 'B_I', color = 'blue')

plt.plot(T_C_real, '-o', label = 'C_Q', color = 'green')
plt.plot(T_C_imag, '-D', mfc = 'w', label = 'C_I', color = 'green')

plt.xlabel('Experiment number')
plt.ylabel('T_eff, mK')
#plt.ylim(38,185)
plt.axhline(100, linestyle = 'dashed', color = 'grey')
plt.axhspan(90, 110, color = 'lightgrey')

plt.legend()

#Save figure
figname = 'Population_stat10p_'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

### Rotation and measurement way optimization

In [ ]:
#Cut the data

# pop_slice = {}
# point = 10
# start = (point-1)*32+1
# stop = point*32+1
# for k in pop_full_results:
#     pop_slice[k] = pop_full_results[k][start:stop]

In [ ]:
phase_arr = np.linspace(-1, 1, 65)*np.pi/2

STAT_I, STAT_Q = make_rotation_temperature(pop_full_results, phase_arr, f_q, qubit_parameters['qb_anharm']*1e-9, three_levels = True)

In [ ]:
plot_rotation_stat(phase_arr, STAT_I, STAT_Q, info_type = 'rel_err')

In [ ]:
STAT_I, STAT_Q = make_rotation_temperature(pop_full_results, [0], f_q, qubit_parameters['qb_anharm']*1e-9, three_levels = True)

In [ ]:
STAT_I[k]

In [ ]:
plot_stat(STAT_I, STAT_Q, element = 0)

In [ ]:
anharm = qubit_parameters['qb_anharm']*1e-9
skip = ['TA1', 'TA3']
optimal_method = make_optimal_temperature(pop_full_results, phase_arr, f_q, anharm, skip = skip)

In [ ]:
optimal_method

In [ ]:
Teff = find_optimal_temperature(pop_full_results, optimal_method, f_q, anharm, three_levels = True)

In [ ]:
plt.plot(Teff)

In [ ]:
STAT_I = {}
STAT_Q = {}

phase_arr = np.linspace(-1, 1, 65)*np.pi/2

for i in range(len(phase_arr)):
    proj_I, proj_Q = make_projection(pop_full_results, phase_arr[i])
    
    ABC_I = get_ABC(proj_I)
    ABC_Q = get_ABC(proj_Q)

    TABC_I = get_temperature(ABC_I, f_q, qubit_parameters['qb_anharm'])
    TABC_Q = get_temperature(ABC_Q, f_q, qubit_parameters['qb_anharm'])

#     stat_ABC_I = get_stat(ABC_I)
#     stat_ABC_Q = get_stat(ABC_Q)

#     for k in stat_ABC_I.keys():
#         if i == 0:
#             STAT_I[k] = [stat_ABC_I[k]]
#             STAT_Q[k] = [stat_ABC_Q[k]]
#         else:
#             STAT_I[k].append(stat_ABC_I[k])
#             STAT_Q[k].append(stat_ABC_Q[k])

# for k in stat_ABC_I.keys():
#     STAT_I[k] = np.array(STAT_I[k])
#     STAT_Q[k] = np.array(STAT_Q[k])

    stat_TABC_I = get_stat(TABC_I)
    stat_TABC_Q = get_stat(TABC_Q)
    for k in stat_TABC_I.keys():
        if i == 0:
            STAT_I[k] = [stat_TABC_I[k]]
            STAT_Q[k] = [stat_TABC_Q[k]]
        else:
            STAT_I[k].append(stat_TABC_I[k])
            STAT_Q[k].append(stat_TABC_Q[k])

for k in stat_TABC_I.keys():
    STAT_I[k] = np.array(STAT_I[k])
    STAT_Q[k] = np.array(STAT_Q[k])
    
# phase = np.pi/4
# proj_I, proj_Q = make_projection(pop_slice, phase)
# ABC_I = get_ABC(proj_I)
# ABC_Q = get_ABC(proj_Q)

In [ ]:
fig, ax = plt.subplots(3, 3, sharex = True, figsize = (10, 8))

fig.suptitle('Relative errors for effective temperature vs. phase rotation', fontsize=16)

fig.supxlabel('Relative error')
fig.supylabel('Phase')

# ax[0,0].plot(phase_arr, STAT_I['A1'][:,2], label = 'I')
# ax[0,0].plot(phase_arr, STAT_Q['A1'][:,2], label = 'Q')
# ax[0,1].plot(phase_arr, STAT_I['A2'][:,2], label = 'I')
# ax[0,1].plot(phase_arr, STAT_Q['A2'][:,2], label = 'Q')
# ax[0,2].plot(phase_arr, STAT_I['A3'][:,2], label = 'I')
# ax[0,2].plot(phase_arr, STAT_Q['A3'][:,2], label = 'Q')

# ax[1,0].plot(phase_arr, STAT_I['B1'][:,2], label = 'I')
# ax[1,0].plot(phase_arr, STAT_Q['B1'][:,2], label = 'Q')
# ax[1,1].plot(phase_arr, STAT_I['B2'][:,2], label = 'I')
# ax[1,1].plot(phase_arr, STAT_Q['B2'][:,2], label = 'Q')
# ax[1,2].plot(phase_arr, STAT_I['B3'][:,2], label = 'I')
# ax[1,2].plot(phase_arr, STAT_Q['B3'][:,2], label = 'Q')

# ax[2,0].plot(phase_arr, STAT_I['C1'][:,2], label = 'I')
# ax[2,0].plot(phase_arr, STAT_Q['C1'][:,2], label = 'Q')
# ax[2,1].plot(phase_arr, STAT_I['C2'][:,2], label = 'I')
# ax[2,1].plot(phase_arr, STAT_Q['C2'][:,2], label = 'Q')
# ax[2,2].plot(phase_arr, STAT_I['C3'][:,2], label = 'I')
# ax[2,2].plot(phase_arr, STAT_Q['C3'][:,2], label = 'Q')

ax[0,0].plot(phase_arr, STAT_I['TA1'][:,2], label = 'I')
ax[0,0].plot(phase_arr, STAT_Q['TA1'][:,2], label = 'Q')
ax[0,1].plot(phase_arr, STAT_I['TA2'][:,2], label = 'I')
ax[0,1].plot(phase_arr, STAT_Q['TA2'][:,2], label = 'Q')
ax[0,2].plot(phase_arr, STAT_I['TA3'][:,2], label = 'I')
ax[0,2].plot(phase_arr, STAT_Q['TA3'][:,2], label = 'Q')

ax[1,0].plot(phase_arr, STAT_I['TB1'][:,2], label = 'I')
ax[1,0].plot(phase_arr, STAT_Q['TB1'][:,2], label = 'Q')
ax[1,1].plot(phase_arr, STAT_I['TB2'][:,2], label = 'I')
ax[1,1].plot(phase_arr, STAT_Q['TB2'][:,2], label = 'Q')
ax[1,2].plot(phase_arr, STAT_I['TB3'][:,2], label = 'I')
ax[1,2].plot(phase_arr, STAT_Q['TB3'][:,2], label = 'Q')

ax[2,0].plot(phase_arr, STAT_I['TC1'][:,2], label = 'I')
ax[2,0].plot(phase_arr, STAT_Q['TC1'][:,2], label = 'Q')
ax[2,1].plot(phase_arr, STAT_I['TC2'][:,2], label = 'I')
ax[2,1].plot(phase_arr, STAT_Q['TC2'][:,2], label = 'Q')
ax[2,2].plot(phase_arr, STAT_I['TC3'][:,2], label = 'I')
ax[2,2].plot(phase_arr, STAT_Q['TC3'][:,2], label = 'Q')

ax[0,0].set_title('A1')
ax[1,0].set_title('B1')
ax[2,0].set_title('C1')
ax[0,1].set_title('A2')
ax[1,1].set_title('B2')
ax[2,1].set_title('C2')
ax[0,2].set_title('A3')
ax[1,2].set_title('B3')
ax[2,2].set_title('C3')

fig.legend()

In [ ]:
fig, ax = plt.subplots(3, 3, sharex = True, figsize = (10, 8))

fig.suptitle('Relative errors for effective temperature vs. phase rotation', fontsize=16)

fig.supxlabel('Mean temperature')
fig.supylabel('Phase')

# ax[0,0].plot(phase_arr, STAT_I['A1'][:,2], label = 'I')
# ax[0,0].plot(phase_arr, STAT_Q['A1'][:,2], label = 'Q')
# ax[0,1].plot(phase_arr, STAT_I['A2'][:,2], label = 'I')
# ax[0,1].plot(phase_arr, STAT_Q['A2'][:,2], label = 'Q')
# ax[0,2].plot(phase_arr, STAT_I['A3'][:,2], label = 'I')
# ax[0,2].plot(phase_arr, STAT_Q['A3'][:,2], label = 'Q')

# ax[1,0].plot(phase_arr, STAT_I['B1'][:,2], label = 'I')
# ax[1,0].plot(phase_arr, STAT_Q['B1'][:,2], label = 'Q')
# ax[1,1].plot(phase_arr, STAT_I['B2'][:,2], label = 'I')
# ax[1,1].plot(phase_arr, STAT_Q['B2'][:,2], label = 'Q')
# ax[1,2].plot(phase_arr, STAT_I['B3'][:,2], label = 'I')
# ax[1,2].plot(phase_arr, STAT_Q['B3'][:,2], label = 'Q')

# ax[2,0].plot(phase_arr, STAT_I['C1'][:,2], label = 'I')
# ax[2,0].plot(phase_arr, STAT_Q['C1'][:,2], label = 'Q')
# ax[2,1].plot(phase_arr, STAT_I['C2'][:,2], label = 'I')
# ax[2,1].plot(phase_arr, STAT_Q['C2'][:,2], label = 'Q')
# ax[2,2].plot(phase_arr, STAT_I['C3'][:,2], label = 'I')
# ax[2,2].plot(phase_arr, STAT_Q['C3'][:,2], label = 'Q')

ax[0,0].plot(phase_arr, STAT_I['TA1'][:,0], label = 'I')
ax[0,0].plot(phase_arr, STAT_Q['TA1'][:,0], label = 'Q')
ax[0,1].plot(phase_arr, STAT_I['TA2'][:,0], label = 'I')
ax[0,1].plot(phase_arr, STAT_Q['TA2'][:,0], label = 'Q')
ax[0,2].plot(phase_arr, STAT_I['TA3'][:,0], label = 'I')
ax[0,2].plot(phase_arr, STAT_Q['TA3'][:,0], label = 'Q')

ax[1,0].plot(phase_arr, STAT_I['TB1'][:,0], label = 'I')
ax[1,0].plot(phase_arr, STAT_Q['TB1'][:,0], label = 'Q')
ax[1,1].plot(phase_arr, STAT_I['TB2'][:,0], label = 'I')
ax[1,1].plot(phase_arr, STAT_Q['TB2'][:,0], label = 'Q')
ax[1,2].plot(phase_arr, STAT_I['TB3'][:,0], label = 'I')
ax[1,2].plot(phase_arr, STAT_Q['TB3'][:,0], label = 'Q')

ax[2,0].plot(phase_arr, STAT_I['TC1'][:,0], label = 'I')
ax[2,0].plot(phase_arr, STAT_Q['TC1'][:,0], label = 'Q')
ax[2,1].plot(phase_arr, STAT_I['TC2'][:,0], label = 'I')
ax[2,1].plot(phase_arr, STAT_Q['TC2'][:,0], label = 'Q')
ax[2,2].plot(phase_arr, STAT_I['TC3'][:,0], label = 'I')
ax[2,2].plot(phase_arr, STAT_Q['TC3'][:,0], label = 'Q')

ax[0,0].set_title('A1')
ax[1,0].set_title('B1')
ax[2,0].set_title('C1')
ax[0,1].set_title('A2')
ax[1,1].set_title('B2')
ax[2,1].set_title('C2')
ax[0,2].set_title('A3')
ax[1,2].set_title('B3')
ax[2,2].set_title('C3')

fig.legend()

In [ ]:
fig, ax = plt.subplots(3, 3, sharex = True, figsize = (10, 8))

fig.suptitle('Relative errors for effective temperature vs. phase rotation', fontsize=16)

fig.supxlabel('Mean temperature')
fig.supylabel('Phase')

# ax[0,0].plot(phase_arr, STAT_I['A1'][:,2], label = 'I')
# ax[0,0].plot(phase_arr, STAT_Q['A1'][:,2], label = 'Q')
# ax[0,1].plot(phase_arr, STAT_I['A2'][:,2], label = 'I')
# ax[0,1].plot(phase_arr, STAT_Q['A2'][:,2], label = 'Q')
# ax[0,2].plot(phase_arr, STAT_I['A3'][:,2], label = 'I')
# ax[0,2].plot(phase_arr, STAT_Q['A3'][:,2], label = 'Q')

# ax[1,0].plot(phase_arr, STAT_I['B1'][:,2], label = 'I')
# ax[1,0].plot(phase_arr, STAT_Q['B1'][:,2], label = 'Q')
# ax[1,1].plot(phase_arr, STAT_I['B2'][:,2], label = 'I')
# ax[1,1].plot(phase_arr, STAT_Q['B2'][:,2], label = 'Q')
# ax[1,2].plot(phase_arr, STAT_I['B3'][:,2], label = 'I')
# ax[1,2].plot(phase_arr, STAT_Q['B3'][:,2], label = 'Q')

# ax[2,0].plot(phase_arr, STAT_I['C1'][:,2], label = 'I')
# ax[2,0].plot(phase_arr, STAT_Q['C1'][:,2], label = 'Q')
# ax[2,1].plot(phase_arr, STAT_I['C2'][:,2], label = 'I')
# ax[2,1].plot(phase_arr, STAT_Q['C2'][:,2], label = 'Q')
# ax[2,2].plot(phase_arr, STAT_I['C3'][:,2], label = 'I')
# ax[2,2].plot(phase_arr, STAT_Q['C3'][:,2], label = 'Q')

ax[0,0].plot(phase_arr, STAT_I['TA1'][:,1], label = 'I')
ax[0,0].plot(phase_arr, STAT_Q['TA1'][:,1], label = 'Q')
ax[0,1].plot(phase_arr, STAT_I['TA2'][:,1], label = 'I')
ax[0,1].plot(phase_arr, STAT_Q['TA2'][:,1], label = 'Q')
ax[0,2].plot(phase_arr, STAT_I['TA3'][:,1], label = 'I')
ax[0,2].plot(phase_arr, STAT_Q['TA3'][:,1], label = 'Q')

ax[1,0].plot(phase_arr, STAT_I['TB1'][:,1], label = 'I')
ax[1,0].plot(phase_arr, STAT_Q['TB1'][:,1], label = 'Q')
ax[1,1].plot(phase_arr, STAT_I['TB2'][:,1], label = 'I')
ax[1,1].plot(phase_arr, STAT_Q['TB2'][:,1], label = 'Q')
ax[1,2].plot(phase_arr, STAT_I['TB3'][:,1], label = 'I')
ax[1,2].plot(phase_arr, STAT_Q['TB3'][:,1], label = 'Q')

ax[2,0].plot(phase_arr, STAT_I['TC1'][:,1], label = 'I')
ax[2,0].plot(phase_arr, STAT_Q['TC1'][:,1], label = 'Q')
ax[2,1].plot(phase_arr, STAT_I['TC2'][:,1], label = 'I')
ax[2,1].plot(phase_arr, STAT_Q['TC2'][:,1], label = 'Q')
ax[2,2].plot(phase_arr, STAT_I['TC3'][:,1], label = 'I')
ax[2,2].plot(phase_arr, STAT_Q['TC3'][:,1], label = 'Q')

ax[0,0].set_title('A1')
ax[1,0].set_title('B1')
ax[2,0].set_title('C1')
ax[0,1].set_title('A2')
ax[1,1].set_title('B2')
ax[2,1].set_title('C2')
ax[0,2].set_title('A3')
ax[1,2].set_title('B3')
ax[2,2].set_title('C3')

fig.legend()

In [ ]:
optimal_method = {'rel_err': 1000,
                 'phase': 0,
                  'ph_p': 0,
                 'method': '0',
                 'axes': '0'
                 }

skip = ['TA3']

for k in stat_TABC_I.keys():
    if k in skip:
        continue
    i = 2
    rel_Q = np.nanmin(STAT_Q[k][:,i])
    ph_p_Q = np.nanargmin(STAT_Q[k][:,i])
    rel_I = np.nanmin(STAT_I[k][:,i])
    ph_p_I = np.nanargmin(STAT_I[k][:,i])
    print('For', k)
    print('For I:')
    print('Min rel err:', rel_I, 'phase point number', ph_p_I)
    print('For Q:')
    print('Min rel err:', rel_Q, 'phase point number', ph_p_Q)

    if (rel_I or ph_p_Q) < optimal_method['rel_err']:
        optimal_method['method'] = k
        if abs(phase_arr[ph_p_Q]) < abs(phase_arr[ph_p_I]):
            optimal_method['rel_err'] = rel_Q
            optimal_method['phase'] = phase_arr[ph_p_Q]
            optimal_method['ph_p'] = ph_p_Q
            optimal_method['axes'] = 'Q'
        else:
            optimal_method['rel_err'] = rel_I
            optimal_method['phase'] = phase_arr[ph_p_I]
            optimal_method['ph_p'] = ph_p_I
            optimal_method['axes'] = 'I'
            
    
    #STAT_Q[k] = np.array(STAT_Q[k])

In [ ]:
optimal_method

In [ ]:
STAT_Q[optimal_method['method']][optimal_method['ph_p'],:]

In [ ]:
#test_var = None
test_var = optimal_method
if test_var:
    print('Hey!')

In [ ]:
for k in stat_TABC_I.keys():
    print('For', k)
    print('For I:')
    print('Min rel err:', np.nanmin(STAT_I[k]), 'phase point number', np.nanargmin(STAT_I[k]))
    print('For Q:')
    print('Min rel err:', np.nanmin(STAT_Q[k]), 'phase point number', np.nanargmin(STAT_Q[k]))

In [ ]:
m = np.nanmin(STAT_I['TB1'])
print(np.argwhere(np.nanmin(STAT_I['TB1'])==m))

In [ ]:
STAT_I['TC3'][:,2]

In [ ]:
ABC_Q['A3']

In [ ]:
phase_arr = np.linspace(-1, 1, 17)
print(phase_arr)

In [ ]:
TABC_I = get_temperature(ABC_I, f_q)
stat_TABC_I = get_stat(TABC_I) 
for k in TABC_I.keys():
    print(stat_TABC_I[k][2])


In [ ]:
pop_slice = {}
for k in pop_full_results:
    pop_slice[k] = pop_full_results[k][1:33]

In [ ]:
phase = np.pi/4
proj_I, proj_Q = make_projection(pop_slice, phase)
ABC_I = get_ABC(proj_I)
ABC_Q = get_ABC(proj_Q)

In [ ]:
for k in ABC_I.keys():
    M = np.mean(ABC_I[k])
    V = np.sqrt(np.var(ABC_I[k], ddof=1))
    print('For', k, 'mean', M, 'relative error', V/M)

In [ ]:
plt.plot(T_calc(1-ABC_Q['A1'], f_q)*1e3, '-o', label = 'C1')
plt.ylim(0,300)

In [ ]:
#Get ABC
A_real = (pop_full_results['x0'].real -pop_full_results['x1'].real)/(pop_full_results['y0'].real-pop_full_results['y1'].real)
A_imag = (pop_full_results['x0'].imag -pop_full_results['x1'].imag)/(pop_full_results['y0'].imag-pop_full_results['y1'].imag)

B_real = (pop_full_results['x1'].real -pop_full_results['y1'].real)/(pop_full_results['y0'].real-pop_full_results['x2'].real)
B_imag = (pop_full_results['x1'].imag -pop_full_results['y1'].imag)/(pop_full_results['y0'].imag-pop_full_results['x2'].imag)

C_real = (pop_full_results['x0'].real -pop_full_results['y0'].real)/(pop_full_results['x1'].real-pop_full_results['x2'].real)
C_imag = (pop_full_results['x0'].imag -pop_full_results['y0'].imag)/(pop_full_results['x1'].imag-pop_full_results['x2'].imag)

In [ ]:
#Plot pe/pg

plt.plot(C_real, '-o', label = 'C_Q')
plt.plot(C_imag, '-o', label = 'C_I')

plt.plot(1 - A_real, '-o', label = 'A_Q')
plt.plot(1- A_imag, '-o', label = 'A_I')

plt.plot(B_real/(B_real+1), '-o', label = 'B_Q')
plt.plot(B_imag/(B_imag+1), '-o', label = 'B_I')

plt.xlabel('Experiment number')
plt.ylabel('$p_e/p_g$')
plt.legend()

In [ ]:
print('Mean pe/pg C_real:', np.mean(C_real)*1e2)
print('Mean pe/pg C_imag:', np.mean(C_imag)*1e2)

print('Mean pe/pg A_real:', np.mean(1 - A_real)*1e2)
print('Mean pe/pg A_imag:', np.mean(1- A_imag)*1e2)

print('Mean pe/pg B_real:', np.mean(B_real/(B_real+1))*1e2)
print('Mean pe/pg B_imag:', np.mean(B_imag/(B_imag+1))*1e2)

In [ ]:
#Get temperatures

T_A_real = T_calc(1 - A_real, f_q)*1e3
T_A_imag = T_calc(1 - A_imag, f_q)*1e3

T_B_real = T_calc(B_real/(B_real+1), f_q)*1e3
T_B_imag = T_calc(B_imag/(B_imag+1), f_q)*1e3

T_C_real = T_calc(C_real, f_q)*1e3
T_C_imag = T_calc(C_imag, f_q)*1e3

In [ ]:
Data_T_stat = {#'pop_full_results': pop_full_results,
                'T_A_real': T_A_real,
                'T_A_imag': T_A_imag,
                'T_B_real': T_B_real,
                'T_B_imag': T_B_imag,
                'T_C_real': T_C_real,
               'T_C_imag': T_C_imag
                }

Data_T_stat.update(qubit_parameters)
Data_T_stat.update(lo_settings)
Data_T_stat.update(pop_full_results)

file_name = 'Teff_stat_full_10p_corr_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_T_stat)

### Population with prepulse

In [ ]:
# range of pulse amplitude scan
amp_min = 0
amp_max = qubit_parameters["pi_amp"] #min([qubit_parameters["pi_amp"] * 2.2, 1.0])
# how many amplitude points to measure
amp_num = 51

# set up sweep parameter - qubit drive pulse amplitude
pulse_sweep = LinearSweepParameter(
    uid="pulse_amp", start=amp_min, stop=amp_max, count=amp_num
)

# how many averages per point: 2^n_average
n_average = 17

# Rabi excitation pulse - gaussian of unit amplitude - amplitude will be scaled with sweep parameter in experiment
gaussian_pulse = pulse_library.gaussian(
    uid="gaussian_pulse", length=qubit_parameters["qb_len"], amplitude=1.0
)

In [ ]:
exp_population_prepulse = make_exp_population_prepulse(pulse_sweep, 
                                                   x180, 
                                                   x180_ef, 
                                                   gaussian_pulse, 
                                                   readout_pulse, 
                                                   readout_weighting_function, 
                                                   qubit_parameters["relax"], 
                                                   n_average)

exp_population_prepulse.set_signal_map(qubit_ef_map)

exp_population_prepulse_comp = my_session.compile(exp_population_prepulse)

#show_pulse_sheet("Population_prep", exp_population_prepulse_comp)

In [ ]:
results_population_prepulse = my_session.run(exp_population_prepulse_comp)

In [ ]:
pulse_amp = results_population_prepulse.get_axis("x0_readout")[0]

In [ ]:
x0 = results_population_prepulse.get_data('x0_readout')
x1 = results_population_prepulse.get_data('x1_readout')
x2 = results_population_prepulse.get_data('x2_readout')
y0 = results_population_prepulse.get_data('y0_readout')
y1 = results_population_prepulse.get_data('y1_readout')
y2 = results_population_prepulse.get_data('y2_readout')

In [ ]:
plt.plot(pulse_amp, (abs(x0)-np.min(abs(x0)))/(np.max(abs(x0))-np.min(abs(x0))), 'o-')

In [ ]:
Data_pop_prep = {'x0': x0,
               'x1': x1,
               'x2': x2,
               'y0': y0,
               'y1': y1,
               'y2': y2,
               'pulse_amp':pulse_amp
                }

Data_pop_prep.update(qubit_parameters)
Data_pop_prep.update(lo_settings)

file_name = 'R4_population_prep_19mK_'
file_path = get_path_to_file(file_name, '.mat')
savemat(file_path, Data_pop_prep)

In [ ]:
A_real = (x0.real -x1.real)/(y0.real-y1.real)
A_imag = (x0.imag -x1.imag)/(y0.imag-y1.imag)

B_real = (x1.real -y1.real)/(y0.real-x2.real)
B_imag = (x1.imag -y1.imag)/(y0.imag-x2.imag)

C_real = (x0.real -y0.real)/(x1.real-x2.real)
C_imag = (x0.imag -y0.imag)/(x1.imag-x2.imag)

In [ ]:
pe0 = 1 - A_real[0]

In [ ]:
#plt.plot(pulse_amp, C_real, '-o', label = 'C_Q')
#plt.plot(pulse_amp, C_imag, '-o', label = 'C_I')

plt.plot(pulse_amp, 1 - A_real, '-o', label = 'A_Q')
#plt.plot(pulse_amp, 1- A_imag, '-o', label = 'A_I')

plt.plot(pulse_amp, B_real/(B_real+1), '-o', label = 'B_Q')
#plt.plot(pulse_amp, B_imag/(B_imag+1), '-o', label = 'B_I')

pe = (abs(x0)-np.min(abs(x0)))/(np.max(abs(x0))-np.min(abs(x0)))
pe_sh = (1-2*pe0)*pe+pe0
plt.plot(pulse_amp, pe_sh/(1-pe_sh), '-')

plt.xlabel('Pulse_amp')
plt.ylabel('$p_e/p_g$')
plt.legend()
plt.ylim(0.0,1.0)

## Population measurements for different readout frequencies (slow!)

In [ ]:
# frequency range of spectroscopy scan - around expected centre frequency as defined in qubit parameters
spec_range = 10e6
# how many frequency points to measure
spec_num = 25

ro_freq_arr = np.linspace(qubit_parameters["ro_freq"] - spec_range / 2, qubit_parameters["ro_freq"] + spec_range / 2, spec_num)

# take how many averages per point: 2^n_average
n_average = 12

In [ ]:
readout_test

In [ ]:
pop_i_results = {}
labels = ['x0', 'x1', 'x2', 'y0', 'y1', 'y2']

for i in range(spec_num):
    readout_i = {'readout_type': 'pulse',
                    'readout_pulse': readout_test['readout_pulse'],
                    'readout_weighting_function': readout_test['readout_weighting_function'],
                    'relax_time': qubit_parameters["relax"],
                    'measure_freq': ro_freq_arr[i],
                    'acquire_freq': ro_freq_arr[i],
                    'readout_range': -25,
                    'readout_delay': qubit_parameters['ro_int_delay']
                   }
    
    exp_population_i = create_exp_population_full(x180, 
                                                  x180_ef, 
                                                  readout_i, 
                                                  n_average)

    #compile experiment
    exp_population_i_comp = my_session.compile(exp_population_i)
    results_population_i = my_session.run(exp_population_i_comp)

    for k in labels:
        if i == 0:
            pop_i_results[k] = [results_population_i.get_data(handle=f'{k}_readout')]
        else:
            pop_i_results[k].append(results_population_i.get_data(handle=f'{k}_readout'))
            

for k in labels:
    pop_i_results[k] = np.array(pop_i_results[k])


In [ ]:
plt.plot(pop_i_results['x0'].real, pop_i_results['x0'].imag)
plt.plot(pop_i_results['x1'].real, pop_i_results['x1'].imag)
A = 0.135
T = np.linspace(-2*np.pi, 2*np.pi, 201)
#plt.plot(A*np.sin(T), A*np.cos(T))

In [ ]:
readout_pulse

In [ ]:
plt.plot((ro_freq_arr+lo_settings['ro_lo'])*1e-9, np.abs(pop_i_results['x0']-pop_i_results['x1']))
plt.plot(res_1_freq*1e-9, np.abs(res_1_res-res_0_res)*1600, 'r', label = 'first exited')
#plt.plot(np.abs(pop_i_results['x0']-pop_i_results['y0']))
#plt.plot(np.abs(pop_i_results['x1']-pop_i_results['y0']))
#plt.yscale('log')

In [ ]:
plt.plot(np.unwrap(np.angle(pop_i_results['x0']))-np.unwrap(np.angle(pop_i_results['x1'])))
plt.plot(np.unwrap(np.angle(pop_i_results['x0']))-np.unwrap(np.angle(pop_i_results['y0'])))
#plt.plot(np.unwrap(np.angle(pop_i_results['y0']))-np.unwrap(np.angle(pop_i_results['x1'])))
#plt.yscale('log')

In [ ]:
#fast temperature extraction
TABC_I, TABC_Q = make_all_temperatures(pop_i_results, f_q, qubit_parameters['qb_anharm']*1e-9)


#plot the results
#skip = ['TA1', 'TB1', 'TC1', 'TB3', 'TC3']
skip = ['TC3']
#skip = []

plot_temperatures(TABC_I, TABC_Q, skip = skip)

In [ ]:
STAT_I = {}
STAT_Q = {}

#phase_arr = np.linspace(-1, 1, 65)*np.pi/2
phase_arr = np.array([0])

for i in range(len(phase_arr)):
    proj_I, proj_Q = make_projection(pop_i_results, phase_arr[i])
    
    ABC_I = get_ABC(proj_I)
    ABC_Q = get_ABC(proj_Q)

    TABC_I = get_temperature(ABC_I, f_q)
    TABC_Q = get_temperature(ABC_Q, f_q)

#     stat_ABC_I = get_stat(ABC_I)
#     stat_ABC_Q = get_stat(ABC_Q)

#     for k in stat_ABC_I.keys():
#         if i == 0:
#             STAT_I[k] = [stat_ABC_I[k]]
#             STAT_Q[k] = [stat_ABC_Q[k]]
#         else:
#             STAT_I[k].append(stat_ABC_I[k])
#             STAT_Q[k].append(stat_ABC_Q[k])

# for k in stat_ABC_I.keys():
#     STAT_I[k] = np.array(STAT_I[k])
#     STAT_Q[k] = np.array(STAT_Q[k])

    stat_TABC_I = get_stat(TABC_I)
    stat_TABC_Q = get_stat(TABC_Q)
    for k in stat_TABC_I.keys():
        if i == 0:
            STAT_I[k] = [stat_TABC_I[k]]
            STAT_Q[k] = [stat_TABC_Q[k]]
        else:
            STAT_I[k].append(stat_TABC_I[k])
            STAT_Q[k].append(stat_TABC_Q[k])

for k in stat_TABC_I.keys():
    STAT_I[k] = np.array(STAT_I[k])
    STAT_Q[k] = np.array(STAT_Q[k])
    
# phase = np.pi/4
# proj_I, proj_Q = make_projection(pop_slice, phase)
# ABC_I = get_ABC(proj_I)
# ABC_Q = get_ABC(proj_Q)

In [ ]:
np.mean(TABC_I['TA1'])

In [ ]:
for k in TABC_I.keys():
    plt.plot(ro_freq_arr+lo_settings['ro_lo'], TABC_I[k], label = k)
plt.ylim(0, 200)
plt.legend()

In [ ]:
for k in TABC_Q.keys():
    plt.plot(ro_freq_arr+lo_settings['ro_lo'], TABC_Q[k], label = k)
plt.ylim(0, 200)
plt.legend()

In [ ]:
for k in TABC_Q.keys():
    plt.plot(TABC_I[k], TABC_Q[k], label = k)
plt.ylim(0, 200)
plt.xlim(0, 200)
plt.legend()

# Fast flux drive

## Flux amplitude calibration

In [ ]:
lsg_q0['th_res_line'].physical_channel.calibration.port_mode = PortMode.LF
lsg_q0['th_res_line'].local_oscillator.frequency = 0

In [ ]:
#lsg_q0['th_res_line'].port_mode = lsg_q0['drive_line'].port_mode

In [ ]:
#lsg_q0['th_res_line'].physical_channel.calibration.port_mode = None

In [ ]:
#Set the thermal bath resonator relative frequency
#qubit_parameters["th_res_freq"] = 0 # 675*1e6

res_amp_min = 0.0
res_amp_max = 1.0
res_amp_num = 101

# set up delay sweep parameter
res_amp_sweep = LinearSweepParameter(
    uid="res_amp", start=res_amp_min, stop=res_amp_max, count=res_amp_num
)

# how many averages per point: 2^n_average
n_average = 12

#delay after pi-pulse
delay_time = 1e-7

#qubit detuning
detune_ss = 0

flux_pulse = pulse_library.gaussian_square(
    uid="flux_pulse", length=2.0e-6, width = 1.8e-6, amplitude=1, can_compress = True
)

In [ ]:
test_0_pulse = pulse_library.gaussian(
    uid="test_0", length=qubit_parameters["qb_len"], amplitude=0.0
)

In [ ]:
qubit_parameters["qb_len"]

In [ ]:
readout_weighting_function

In [ ]:
flux_pulse

In [ ]:
qubit_parameters["th_res_freq"]

In [ ]:
#make experiment
exp_tr_amp = make_th_res_amp(res_amp_sweep, 
                             flux_pulse, 
                             x180,
                             readout_pulse, 
                             readout_weighting_function, 
                             qubit_parameters["relax"], 
                             n_average, 
                             delay_time = delay_time)

#make calibration
th_res_cal = make_th_res_calib(qubit_parameters["th_res_freq"], qubit_parameters["qb_freq"], detune_ss)

#set calibration
exp_tr_amp.set_calibration(th_res_cal)

#set signal map
exp_tr_amp.set_signal_map(qubit_resonator_map)

#compile experiment
exp_tr_amp_comp = my_session.compile(exp_tr_amp)

#show pulse sequense
#show_pulse_sheet("Thermal_resonator_exitation", my_session.exp_tr_amp_comp)

In [ ]:
show_pulse_sheet("Thermal_resonator_exitation", exp_tr_amp_comp)

In [ ]:
#run experiment
tr_amp_results = my_session.run(exp_tr_amp_comp)

In [ ]:
# get measurement data returned by the instruments
tr_amp_res = tr_amp_results.get_data("q0_tr_amp")

# define time axis from qubit parameters
tr_amp_pump = tr_amp_results.get_axis("q0_tr_amp")[0]

In [ ]:
# plot measurement results
fig = plt.figure()
plt.plot(tr_amp_pump, tr_amp_res.imag)
#plt.plot(tr_amp_pump, tr_amp_res_0.imag)
#plt.plot(tr_amp_pump, np.unwrap(np.angle(tr_amp_res)), ".k")
plt.ylabel("A (a.u.)")
plt.xlabel("ampl")

In [ ]:
plt.plot(tr_amp_res.real, tr_amp_res.imag)
#plt.plot(tr_amp_res_0.real, tr_amp_res_0.imag)

In [ ]:
lo_settings['qb_lo']

In [ ]:
detuning_sweep = np.linspace(0, 2000, 321)*1e6

lo_shift_step = 0.5e9

flux_freq_results_list = []

for i in range(len(detuning_sweep)):
    print('Iteration:', i)
    N_shift = detuning_sweep[i]//lo_shift_step
    print('LO shift number:', N_shift)
    lo_freq = lo_settings['qb_lo'] - N_shift*lo_shift_step    
    
    th_res_cal_sw = make_fast_flux_calib(qubit_parameters["th_res_freq"], qubit_parameters["qb_freq"], -(detuning_sweep[i]-N_shift*lo_shift_step), lo_freq)

    exp_tr_amp.set_calibration(th_res_cal_sw)
    #exp_tr_amp.set_signal_map(qubit_resonator_map)

    exp_tr_amp_comp = my_session.compile(exp_tr_amp)

    tr_amp_results = my_session.run(exp_tr_amp_comp)

    tr_amp_res = tr_amp_results.get_data("q0_tr_amp")

    flux_freq_results_list.append(tr_amp_res)

flux_freq_results_arr = np.array(flux_freq_results_list)

In [ ]:
#drive_Oscillator_q1.frequency = lo_settings['qb_lo']
#drive_ef_Oscillator_q1.frequency = lo_settings['qb_lo']

In [ ]:
#N = 3

for N in range(11):
    plt.plot(tr_amp_pump, flux_freq_results_arr[N,:].imag, ".k")

In [ ]:
X = tr_amp_pump
Y = detuning_sweep*1e-6
#Z = np.unwrap(np.angle(rabi_shev_arr))
Z = flux_freq_results_arr.real
calc.plot_2d(Y, X,Z, flip = False, cmap = 'Reds')

plt.ylabel('Flux drive amplitude, a.u.')
plt.xlabel('Detuning, MHz')

#save figure
figname = 'Flux_drive_calibration_real_'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

In [ ]:
Data_freq_flux_sweep = {'detuning_sweep': detuning_sweep,
                        'fl_amp_sweep': tr_amp_pump,
                        'flux_freq_results': flux_freq_results_arr,
                        'delay_time': delay_time,
                        'pulse_comment': 'flux gaussian square length=2.0e-6, width = 1.8e-6',
                        'readout_comment': 'opt_readout'
                       }

Data_freq_flux_sweep.update(qubit_parameters)
Data_freq_flux_sweep.update(lo_settings)

file_name = 'Flux_amp_sweep_and_freq_sweep_full_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_freq_flux_sweep)

In [ ]:
#correction

flux_freq_results_arr_corr = np.zeros_like(flux_freq_results_arr)

for i in range(len(detuning_sweep)):
    flux_freq_results_arr_corr[i,:] = flux_freq_results_arr[i,:]-flux_freq_results_arr[-1,:]


In [ ]:
plt.plot(flux_freq_results_arr_corr.imag[100,:])

In [ ]:
X = tr_amp_pump
Y = detuning_sweep*1e-6
#Z = np.unwrap(np.angle(rabi_shev_arr))
Z = flux_freq_results_arr_corr.imag
calc.plot_2d(Y, X,Z, flip = False, cmap = 'Reds')

plt.ylabel('Flux drive amplitude, a.u.')
plt.xlabel('Detuning, MHz')

In [ ]:
amp_max = np.argmax(abs(flux_freq_results_arr_corr), axis = 0)

In [ ]:
f_max = detuning_sweep[amp_max]

In [ ]:
plt.plot(tr_amp_pump, (lo_settings['qb_lo']+qubit_parameters["qb_freq"]-f_max)*1e-9)

x = tr_amp_pump
y = (lo_settings['qb_lo']+qubit_parameters["qb_freq"]-f_max)*1e-9

popt, pcov = opt.curve_fit(func_osc, x, y, p0 = [1, 0, 2, 5.5])
plt.plot(x, func_osc(x, *popt))

## Flux pulse check

In [ ]:
res_amp_min = 0.0
res_amp_max = 1.0
res_amp_num = 201

# set up delay sweep parameter
res_amp_sweep = LinearSweepParameter(
    uid="res_amp", start=res_amp_min, stop=res_amp_max, count=res_amp_num
)

# how many averages per point: 2^n_average
n_average = 11

#delay after pi-pulse
delay_time = 1e-7

#qubit detuning
detune_ss = 0

flux_pulse = pulse_library.gaussian_square(
    uid="flux_pulse", length=2.0e-6, width = 1.8e-6, amplitude=1, can_compress = True
)

In [ ]:
print(np.linspace(100, 200, 21))

In [ ]:
#Delay time sweep
n_average = 11

delay_sweep = np.linspace(10, 2000, 21)*1e-9
#flux_width_sweep = np.linspace(100, 2000, 21)*1e-9

flux_width = 2e-6

#delay_time = 10e-9

flux_delay_results_list = []

detune_ss = 250e6

for i in range(len(delay_sweep)):
    print('Iteration:', i)

    flux_pulse = pulse_library.gaussian_square(
        uid="flux_pulse", length=flux_width+2*delay_sweep[i], width = flux_width, amplitude=1, can_compress = True
    )

    exp_tr_amp = make_th_res_amp(res_amp_sweep, 
                                 flux_pulse, 
                                 x180,
                                 readout_pulse, 
                                 readout_weighting_function, 
                                 qubit_parameters["relax"], 
                                 n_average, 
                                 delay_time = delay_sweep[i])
    
    th_res_cal_sw = make_fast_flux_calib(qubit_parameters["th_res_freq"], 
                                         qubit_parameters["qb_freq"], 
                                         -detune_ss, 
                                         lo_settings['qb_lo']
                                        )

    exp_tr_amp.set_calibration(th_res_cal_sw)
    exp_tr_amp.set_signal_map(qubit_resonator_map)

    exp_tr_amp_comp = my_session.compile(exp_tr_amp)

    tr_amp_results = my_session.run(exp_tr_amp_comp)

    tr_amp_res = tr_amp_results.get_data("q0_tr_amp")

    flux_delay_results_list.append(tr_amp_res)

flux_delay_results_arr = np.array(flux_delay_results_list)

In [ ]:
tr_amp_pump = tr_amp_results.get_axis("q0_tr_amp")[0]

In [ ]:
X = tr_amp_pump
Y = delay_sweep*1e9
#Z = np.unwrap(np.angle(rabi_shev_arr))
Z = flux_delay_results_arr.real
calc.plot_2d(Y, X,Z, flip = False, cmap = 'Reds')

In [ ]:
for i in range(len(flux_width_sweep)):
    plt.plot(tr_amp_pump, flux_delay_results_arr[i,:].imag)

In [ ]:
Data_flux_len_sweep = {'flux_width_sweep': flux_width_sweep,
                        'fl_amp_sweep': tr_amp_pump,
                        'flux_freq_results': flux_freq_results_arr,
                        'delay_time': delay_time,
                        'detuning': detune_ss,
                        'pulse_comment': 'flux gaussian square length=width+2*delay_time, width = flux_width_sweep',
                        'readout_comment': 'opt_readout'
                       }

Data_flux_len_sweep.update(qubit_parameters)
Data_flux_len_sweep.update(lo_settings)

file_name = 'Flux_amp_sweep_and_width_sweep_short_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_flux_len_sweep)

## Decay with different flux detuning

In [ ]:
#prepare t1 sweep

# delay range for ramsey pulses
t1_min = 0.0
t1_max = 30e-6 #5 * qubit_parameters["qb_t1"]
# how many delay points to sweep
t1_num = 31

t1_ff_sweep = log_sweep_help(t1_min, t1_max, t1_num)

n_average = 12

In [ ]:
#preset pulses and parameters

edge = 10e-9

#qubit detuning
#detune_ss = 0
flux_amp = 0.2

# flux_pulse = pulse_library.gaussian_square(
#     uid="flux_pulse", length=2.0e-6, width = 1.98e-6, amplitude=flux_amp, can_compress = True
# )

flux_pulse = pulse_library.const(
    uid="flux_pulse", length=30.0e-6, amplitude=flux_amp, can_compress = True
)

In [ ]:
#make experiment and compile
exp_FF_decay = make_fast_flux_decay(t1_ff_sweep, 
                                    flux_pulse, 
                                    edge,
                                    x180, 
                                    readout_pulse, 
                                    readout_weighting_function, 
                                    qubit_parameters["relax"], 
                                    n_average)
#set calibration and signal map

ff_decay_cal = make_fast_flux_calib(qubit_parameters["th_res_freq"], 
                                    qubit_parameters["qb_freq"], 
                                    0, 
                                    lo_settings['qb_lo']
                                    )

exp_FF_decay.set_calibration(ff_decay_cal) #calibration from previous experiment
exp_FF_decay.set_signal_map(qubit_resonator_map)


# compile
exp_FF_decay_comp = my_session.compile(exp_FF_decay)

#show_pulse_sheet("Thermal_resonator_decay", my_session.exp_tr_decay_comp)

In [ ]:
show_pulse_sheet("FF_decay", exp_FF_decay_comp)

In [ ]:
#run the experiment
FF_decay_results = my_session.run(exp_FF_decay_comp)

In [ ]:
# get measurement data returned by the instruments
FF_decay_res = FF_decay_results.get_data("q0_ff_decay")

# define time axis from qubit parameters
FF_decay_delay = FF_decay_results.get_axis("q0_ff_decay")[0]

In [ ]:
#fit the data
popt_sl, pcov_sl = auto_T1_fit(FF_decay_delay, FF_decay_res, data_type = 'rot', plot = True)

#Transform fit parameters to proper formatting for one slice measurements
popt = popt_sl[0,:]
pcov = pcov_sl[0,:,:]

plt.title('Decay, rotated data')
plt.figtext(0.6, 0.5, r'$T_1$ = '+'{:.2f}'.format(1/popt[0]*1e6)+r'$\mu$s')

# update qubit parameters - here relaxation time / qubit lifetime
qubit_parameters["qb_t1"] = 1 / popt[0]

# T1 error in us
err = np.sqrt(np.diag(pcov))
print(f'T1 = {qubit_parameters["qb_t1"]*1e6:.3f} +- {(err[0]/(popt[0]*popt[0])*1e6):.3f} us')

#save figure
figname = 'T1_FF_'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

In [ ]:
#Sweeep for different amplitudes
n_average = 12

flux_amp_sweep = np.linspace(0.0, 0.3, 41)

FF_decay_res_list = []

for i in range(len(flux_amp_sweep)):
    print('Flux amplitude:', flux_amp_sweep[i])
    flux_pulse_sw = pulse_library.const(
        uid="flux_pulse", length=2.0e-6, amplitude=flux_amp_sweep[i], can_compress = True
    )

    exp_FF_decay = make_fast_flux_decay(t1_ff_sweep, 
                                        flux_pulse_sw, 
                                        edge,
                                        x180, 
                                        readout_pulse, 
                                        readout_weighting_function, 
                                        qubit_parameters["relax"], 
                                        n_average)

    exp_FF_decay.set_calibration(ff_decay_cal) #calibration from previous experiment
    exp_FF_decay.set_signal_map(qubit_resonator_map)

    exp_FF_decay_comp = my_session.compile(exp_FF_decay)

    FF_decay_results = my_session.run(exp_FF_decay_comp)

    FF_decay_res_list.append(FF_decay_results.get_data("q0_ff_decay"))
    FF_decay_delay = FF_decay_results.get_axis("q0_ff_decay")[0]

FF_decay_res_arr = np.array(FF_decay_res_list)

In [ ]:
FF_decay_res_arr = np.array(FF_decay_res_list)

In [ ]:
show_pulse_sheet("Decay_FF", exp_FF_decay_comp)

In [ ]:
popt_t1_FF_arr, pcov_t1_FF_arr = auto_T1_fit(FF_decay_delay, FF_decay_res_arr, data_type = 'rot', plot = True)

In [ ]:
plt.plot(flux_amp_sweep, 1e6/popt_t1_FF_arr[:,0])
#plt.ylabel('gamma, 1/mks')
plt.ylim(0,10)
plt.ylabel('T1, mks')
plt.xlabel('Flux_amp, a.u.')

In [ ]:
popt_t1_FF_arr.shape

In [ ]:
X = flux_amp_sweep[:1018]
Y = FF_decay_delay*1e6
#Z = np.unwrap(np.angle(rabi_shev_arr))
Z = FF_decay_res_arr.imag
calc.plot_2d(Y, X,Z, flip = False, cmap = 'Reds')

In [ ]:
N = len(flux_amp_sweep[:1018])


for i in range(N):
    #print('Flux amplitude up to ', flux_amp_sweep[i])
    plt.plot(FF_decay_res_arr[i,:].real, FF_decay_res_arr[i,:].imag)

In [ ]:
i = 30
plt.plot(FF_decay_res_arr[i,:].real, FF_decay_res_arr[i,:].imag)

In [ ]:
Data_ff_decay = {'FF_decay_delay': FF_decay_delay,
                 'fl_amp_sweep': flux_amp_sweep[:1018],
                 'FF_decay_res_arr': FF_decay_res_arr,
                 'delay_time': delay_time,
                 'pulse_comment': 'flux const',
                 'readout_comment': 'opt_readout'
                }

Data_ff_decay.update(qubit_parameters)
Data_ff_decay.update(lo_settings)

file_name = 'Fast_flux_decay_long_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_ff_decay)

## Population with flux drive

### Single experiment

In [ ]:
n_average = 15

flux_amp = 0.2
edge = 1e-8

flux_pulse = pulse_library.gaussian_square(
    uid="flux_pulse", length=qubit_parameters['relax'], width = qubit_parameters['relax']-2*edge, amplitude=flux_amp, can_compress = True
)

exp_population_flux = create_exp_population_flux(x180, 
                                                 x180_ef, 
                                                 flux_pulse,
                                                 readout_opt,# readout_low,
                                                 n_average)

#compile experiment
exp_population_flux_comp = my_session.compile(exp_population_flux)

In [ ]:
#show_pulse_sheet("Population_FF", exp_population_flux_comp)

In [ ]:
#run experiment
results_population_flux = my_session.run(exp_population_flux_comp)

### Flux sweeep experiment

In [ ]:
n_average = 15

flux_amp_sw = np.linspace(0, 0.7, 41)

edge = 1e-8

pop_flux_results = {}
labels = ['x0', 'x1', 'x2', 'y0', 'y1', 'y2']

In [ ]:
for i in range(len(flux_amp_sw)):
    print('Iteration number:', i)
    print('Flux pulse amplitude:', flux_amp_sw[i])
    flux_pulse_sw = pulse_library.gaussian_square(
        uid="flux_pulse", length=qubit_parameters['relax'], width = qubit_parameters['relax']-2*edge, amplitude=flux_amp_sw[i], can_compress = True
    )

    exp_population_flux = create_exp_population_flux(x180, 
                                                     x180_ef, 
                                                     flux_pulse_sw,
                                                     readout_opt,# readout_low,
                                                     n_average)

    #compile experiment
    exp_population_flux_comp = my_session.compile(exp_population_flux)
    results_population_flux = my_session.run(exp_population_flux_comp)

    
    for k in labels:
        if i == 0:
            pop_flux_results[k] = [results_population_flux.get_data(handle=f'{k}_readout')]
        else:
            pop_flux_results[k].append(results_population_flux.get_data(handle=f'{k}_readout'))

for k in labels:
    pop_flux_results[k] = np.array(pop_flux_results[k])

In [ ]:
#Population 3lvl method, proper rotation

ABC = get_ABC_parallel(pop_flux_results)
TABC = get_temperature(ABC, f_q, qubit_parameters['qb_anharm']*1e-9, three_levels = True)

skip = ['TA3', 'TB3', 'TC3']
#skip = []

plot_temp_single(TABC, skip = skip)

In [ ]:
for k in TABC.keys():
    plt.plot(flux_amp_sw, TABC[k], label = k)

plt.ylabel('Effective temperature, mK')
plt.xlabel('Flux amplitude, a.u.')
plt.legend()

In [ ]:
plt.plot(pop_flux_results['x0'].real, pop_flux_results['x0'].imag)

In [ ]:
Data_T_FF = {'n_av_teff': 15}

Data_T_FF.update(pop_flux_results)
Data_T_FF.update(TABC)
Data_T_FF.update(qubit_parameters)
Data_T_FF.update(lo_settings)

file_name = 'Teff_vs_FF_fast_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_T_FF)

## T1 and population vs. flux simultaniously

In [ ]:
n_average_pop = 15

flux_amp_sw = np.linspace(0.4, 0.7, 151)

edge = 1e-8

pop_flux_results = {}
labels = ['x0', 'x1', 'x2', 'y0', 'y1', 'y2']

n_average_t1 = 12

FF_decay_res_list = []

In [ ]:
#sweep settings
t1_min = 0.0
t1_max = 30e-6 #5 * qubit_parameters["qb_t1"]
# how many delay points to sweep
t1_num = 31

t1_ff_sweep = log_sweep_help(t1_min, t1_max, t1_num)

In [ ]:
#Measurement loop
for i in range(len(flux_amp_sw)):
    print('Iteration number:', i)
    print('Flux pulse amplitude:', flux_amp_sw[i])
    #population
    flux_pulse_sw = pulse_library.gaussian_square(
        uid="flux_pulse", length=qubit_parameters['relax'], width = qubit_parameters['relax']-2*edge, amplitude=flux_amp_sw[i], can_compress = True
    )

    exp_population_flux = create_exp_population_flux(x180, 
                                                     x180_ef, 
                                                     flux_pulse_sw,
                                                     readout_opt,# readout_low,
                                                     n_average_pop)

    #compile experiment
    exp_population_flux_comp = my_session.compile(exp_population_flux)
    results_population_flux = my_session.run(exp_population_flux_comp)

    
    for k in labels:
        if i == 0:
            pop_flux_results[k] = [results_population_flux.get_data(handle=f'{k}_readout')]
        else:
            pop_flux_results[k].append(results_population_flux.get_data(handle=f'{k}_readout'))

    #T1
    print('T1 measurements')
    flux_pulse_t1 = pulse_library.const(
        uid="flux_pulse", length=2.0e-6, amplitude=flux_amp_sw[i], can_compress = True
    )

    exp_FF_decay = make_fast_flux_decay(t1_ff_sweep, 
                                        flux_pulse_t1, 
                                        edge,
                                        x180, 
                                        readout_pulse, 
                                        readout_weighting_function, 
                                        qubit_parameters["relax"], 
                                        n_average_t1)

    exp_FF_decay.set_calibration(ff_decay_cal) #calibration from previous experiment
    exp_FF_decay.set_signal_map(qubit_resonator_map)

    exp_FF_decay_comp = my_session.compile(exp_FF_decay)

    FF_decay_results = my_session.run(exp_FF_decay_comp)

    FF_decay_res_list.append(FF_decay_results.get_data("q0_ff_decay"))
    FF_decay_delay = FF_decay_results.get_axis("q0_ff_decay")[0]

FF_decay_res_arr = np.array(FF_decay_res_list)

for k in labels:
    pop_flux_results[k] = np.array(pop_flux_results[k])

In [ ]:
len(FF_decay_res_list)

In [ ]:
len(flux_amp_sw[:629])

In [ ]:
FF_decay_res_arr = np.array(FF_decay_res_list)

for k in labels:
    pop_flux_results[k] = np.array(pop_flux_results[k])

In [ ]:
Data_ff_decay = {'FF_decay_delay': FF_decay_delay,
                 'fl_amp_sweep_pop': flux_amp_sw,
                 'fl_amp_sweep_T1': flux_amp_sw,
                 'FF_decay_res_arr': FF_decay_res_arr,
                 'delay_time': delay_time,
                 'n_av_pop': 15,
                 'n_av_T1': 12,
                 'pulse_T1_comment': 'flux const',
                 'pulse_pop_comment': 'flux gaussian squre, edge 10ns',
                 'readout_comment': 'opt_readout'
                }
Data_ff_decay.update(pop_flux_results)
Data_ff_decay.update(qubit_parameters)
Data_ff_decay.update(lo_settings)

file_name = 'Decay_and_population_allparam_0p4-0p7_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_ff_decay)

In [ ]:
#population
ABC = get_ABC_parallel(pop_flux_results)
TABC = get_temperature(ABC, f_q, qubit_parameters['qb_anharm']*1e-9, three_levels = True)

for k in TABC.keys():
    plt.plot(flux_amp_sw, TABC[k], label = k)

plt.ylabel('Effective temperature, mK')
plt.xlabel('Flux amplitude, a.u.')
plt.legend()

In [ ]:
popt_t1_FF_arr, pcov_t1_FF_arr = auto_T1_fit(FF_decay_delay, FF_decay_res_arr, data_type = 'rot', plot = False)

In [ ]:
plt.plot(flux_amp_sw, 1e6/popt_t1_FF_arr[:,0])
#plt.ylabel('gamma, 1/mks')
#plt.ylim(0,10)
plt.ylabel('T1, mks')
plt.xlabel('Flux_amp, a.u.')

In [ ]:
plt.plot(FF_decay_res_arr[100,:].real,FF_decay_res_arr[100,:].imag)

# SINIS calibration, statistic and temperature sweep measurements

## Setup instruments for DC-measurements

In [ ]:
#setup voltmeter

DMM = KeysightDMM34465A('TCPIP0::192.168.1.44::hislip0::INSTR')
DMM.set_nplc(10)

In [ ]:
#setup SIM source
V_off = -1.335

frame_addr = 'GPIB1::3::INSTR'
frame = SIM900(address=frame_addr)


sim = frame.get_sim(2)
sim.get_level(only_number = True)
sim.ramp(0.0, N_steps = 1e2)

In [ ]:
sim.get_level(only_number = True)

In [ ]:
sim.ramp(11.556, N_steps = 2e2)

In [ ]:
sim.ramp(0, N_steps = 1e2)

In [ ]:
sim.open_sim()
output = sim.frame.query_inst('NOUT? 1')
print(f"instrument output state: {output}")
sim.close_sim()

In [ ]:
sim.set_output(1)

In [ ]:
sim.get_output()

In [ ]:
print(DMM.scan())

In [ ]:
#SINIS presets
#A12-A11 - measurements pair
I_probe = 50e-12 #current sourse 50pA
#A9 - non-symmetric heating
V_div = 1000
#VSIM = np.linspace(-4.0, 4.0, 41)
#Vsampl = Vappl/2e3
#Vappl = VSIM/V_div

In [ ]:
popt_0  = np.array([0, 0, 0])
pcov_0  = np.array([0, 0, 0])

## Statistics

In [ ]:
#lists for the data storage

#timestamps
timestamp = []

#temperature data
temp_info_list = []

#Rabi experiment
#rabi_list = []

#T1 experiment
t1_list = []

#T1 for ef and ge
#t1_ef_list = []

#Single Shots
#SS_list = []

#Ramsey
ramsey_list = []

#Echo
#echo_list = []

#Three levels population
pop_full_results = {}
labels = ['x0', 'x1', 'x2', 'y0', 'y1', 'y2']

#Rabi population
#qrpm_full_results = []

In [ ]:
N_points = 250

for i in range(N_points):
    print('Measurement number:', i)

    #make timestamp
    timestamp.append(time.time())
        
    #measure temperature
    response = BFTCont.get_mxc_temperature()
    temp_info_list.append(response)
    print('MXC temperature:', response*1e3, 'mK')

    #Rabi measurements
#    my_results_rabi = my_session.run(exp_rabi_comp)
#    rabi_res = my_results_rabi.get_data("q0_rabi")
#    rabi_amp = my_results_rabi.get_axis("q0_rabi")[0]
            
#    rabi_list.append(rabi_res)

    # Quick Rabi Population Measurement
#    print('QRPM')
#    results_qrpm = my_session.run(rpm_quick_exp_comp)
#    rpm_res = {}
#    for key in ['with', 'wo']:
#        rpm_res[key] = {'mag_min': np.abs(results_qrpm.get_data(f"rpm_{key}_min")),
#                        'pha_min': np.angle(results_qrpm.get_data(f"rpm_{key}_min")),
#                        'mag_max': np.abs(results_qrpm.get_data(f"rpm_{key}_max")),
#                        'pha_max': np.angle(results_qrpm.get_data(f"rpm_{key}_max")),
#                        'I_max': np.real(results_qrpm.get_data(f"rpm_{key}_max")),
#                        'Q_max': np.imag(results_qrpm.get_data(f"rpm_{key}_max")),
#                        'I_min': np.real(results_qrpm.get_data(f"rpm_{key}_min")),
#                        'Q_min': np.imag(results_qrpm.get_data(f"rpm_{key}_min"))}

#    qrpm_full_results.append(rpm_res)
#    qrpm_dict = get_qrpm_dict_from_list_of_results(qrpm_full_results)
        
    #Population measurements
    results_population_full = my_session.run(exp_population_full_comp)
    for p in labels:
        if i== 0 :
            pop_full_results[p] = [results_population_full.get_data(handle=f'{p}_readout')]
        else:
            pop_full_results[p].append(results_population_full.get_data(handle=f'{p}_readout'))
    
    #T1 measurements
    my_results_t1 = my_session.run(exp_t1_comp)
    t1_res = my_results_t1.get_data("q0_t1")
    t1_delay = my_results_t1.get_axis("q0_t1")[0]
    
    t1_list.append(t1_res)

    #T1 on ef measurements
    # my_results_t1_ef = my_session.run(exp_T1_ef_comp)
    # t1_ef_res = my_results_t1_ef.get_data("q0_t1_ef")
    # t1_ef_delay = my_results_t1_ef.get_axis("q0_t1_ef")[0]
    
    # t1_ef_list.append(t1_ef_res)
        
    #Ramsey measurements and fit
    my_results_ramsey = my_session.run(exp_ramsey_comp)
    ramsey_res = my_results_ramsey.get_data("q0_ramsey")
    ramsey_delay = my_results_ramsey.get_axis("q0_ramsey")[0]
    
    ramsey_list.append(ramsey_res)

    #Echo measurements
    # my_results_echo = my_session.run(exp_echo_comp)
    # echo_res = my_results_echo.get_data("q0_echo")
    # echo_delay = my_results_echo.get_axis("q0_echo")[0]
       
    # echo_list.append(echo_res)
    
    #Single shots
#    my_results_SS = my_session.run(exp_rabi_SS_comp)
#    SS_res = my_results_SS.get_data("SS_rabi")
#    SS_it = my_results_SS.get_axis("SS_rabi")[0]
#    SS_amp = my_results_SS.get_axis("SS_rabi")[1]
    
#    SS_list.append(SS_res)
        
#sim.get_level(only_number = True)
#sim.ramp(0.0, N_steps = 1e2)

In [ ]:
#convert list to arrays
temp_info_array = np.array(temp_info_list)

t1_arr = np.array(t1_list)
t1_time_arr = t1_delay

# rabi_arr = np.array(rabi_list)

# t1_ef_arr = np.array(t1_ef_list)
# t1_ef_time_arr = t1_ef_delay

ramsey_arr = np.array(ramsey_list)
ramsey_time_arr = ramsey_delay 

# SS_res_arr = np.array(SS_list)

# echo_arr = np.array(echo_list)

timestamp_arr = np.array(timestamp)

for k in labels:
    pop_full_results[k] = np.array(pop_full_results[k])

In [ ]:
#Save the data
Data_statistics = {'timestamp': timestamp_arr,
                   # 'rabi': rabi_arr,
                   #  'rabi_amp': rabi_amp,
                   'T1': t1_arr,  
                   't1_time_arr': t1_time_arr,
                   # 'T1_ef': t1_ef_arr,  
                   # 't1_ef_time_arr': t1_ef_time_arr,
                   'Ramsey': ramsey_arr, 
                   'ramsey_time': ramsey_time_arr,
                   # 'echo_time': echo_delay,
                   # 'Echo': echo_arr,
                   # 'Single_Shot_res': SS_res_arr,
                   # 'Single_Shot_iterations': SS_it,
                   # 'Single_Shot_amp': SS_amp,
                   'MXC_temp': temp_info_array
                  }

Data_statistics.update(pop_full_results)
#Data_statistics.update(qrpm_dict)
Data_statistics.update(qubit_parameters._params)
Data_statistics.update(lo_settings._params)

file_name = 'Statistic_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_statistics)

In [ ]:
#Quick data processing

popt_ramsey_arr, pcov_ramsey_arr = auto_ramsey_fit(ramsey_delay, ramsey_arr, data_type = 'rot', plot = True)

popt_t1_arr, pcov_t1_arr = auto_T1_fit(t1_delay, t1_arr, data_type = 'rot', plot = False)

#popt_echo_arr, pcov_echo_arr = auto_echo_fit(echo_delay, echo_arr, data_type = 'phase', plot = False)

In [ ]:
# Plot Ramsey detuning and T2

fig, ax = plt.subplots(2, 1, sharex = True, figsize = (10, 8))
fig.suptitle('Ramsey measurements')


fig.supxlabel('Point number')

ax[0].set_ylabel('Detuning, MHz')
ax[1].set_ylabel('T2 Ramsey, mks')

ax[0].plot(popt_ramsey_arr[:,0]/np.pi/2*1e-6, 'o-')
ax[1].plot(1/popt_ramsey_arr[:,2]*1e6, 'o-')

file_name = 'Statistic_RamseyT2_'
file_path = get_path_to_file(file_name, '.pdf', sample_parameters)
# fig.savefig(file_path)

In [ ]:
plt.title('T1 measurements')
plt.plot(1/popt_t1_arr[:,0]*1e6, 'o-')
#plt.plot(popt_ramsey_arr[:,0]/np.pi/2*1e-6)
plt.ylabel('T1, mks')
plt.xlabel('Point number')

file_name = 'Statistic_T1_'
file_path = get_path_to_file(file_name, '.pdf', sample_parameters)
# plt.savefig(file_path)

In [ ]:
n_bins = 30

T1_stat = 1/popt_t1_arr[:,0]*1e6

fig, axs = plt.subplots(1, 1, sharey=True, tight_layout=True)

# We can set the number of bins with the *bins* keyword argument.
#axs.hist(T1_real, bins=n_bins, alpha=0.5, label = 'Real')
#axs.hist(T1_imag, bins=n_bins, alpha=0.5, label = 'Imag
axs.hist(T1_stat, bins=n_bins, alpha=0.5, label = 'Rot')
axs.legend()

print('Average T1:', round(np.mean(T1_stat), 3), 'with sigma', round(np.std(T1_stat), 3), 'relative error', round(np.std(T1_stat)/np.mean(T1_stat), 3))

file_name = 'Statistic_T1_histogram_'
file_path = get_path_to_file(file_name, '.pdf', sample_parameters)
# fig.savefig(file_path)

In [ ]:
np.median(T1_stat)

In [ ]:
#fast temperature extraction
TABC_I, TABC_Q = make_all_temperatures(pop_full_results, f_q, qubit_parameters['qb_anharm']*1e-9)


#plot the results
#skip = ['TA3', 'TB3', 'TC3']
skip = []

plot_temperatures(TABC_I, TABC_Q, skip = skip)

In [ ]:
#Population 3lvl method, proper rotation

#Tmxc_str = stretch_by_N(Tmix_N, 5)

ABC = get_ABC_parallel(pop_full_results)
TABC = get_temperature(ABC, f_q, qubit_parameters['qb_anharm']*1e-9, three_levels = True)

skip = ['TA3', 'TB3', 'TC3']
# skip = []

plot_temp_single(TABC, skip = skip)

file_name = 'Statistic_Eff_temp_'
file_path = get_path_to_file(file_name, '.pdf', sample_parameters)
plt.savefig(file_path)

## Flux sweep

In [ ]:
ramsey_list = []

# t1_list = []

temp_info_list = []

#Echo
#echo_list = []

timestamp = []

#pop_full_results = {}
#labels = ['x0', 'x1', 'x2', 'y0', 'y1', 'y2']

In [ ]:
#qubit_parameters['flux_upper_ss'] = 0.775*1e-3

In [ ]:
#qubit_parameters['flux_point'] = 0.19*1e-3

In [ ]:
qubit_parameters['flux_point']

In [ ]:
#volt_source.ramp(qubit_parameters['flux_lower_ss']*R, 10)

In [ ]:
flux_sweep = np.linspace(-0.002, 0.002, 11)*1e-3
flux_points = qubit_parameters['flux_point']+flux_sweep
print(flux_points)

In [ ]:
for i in range(len(flux_points)):
    print('Measurement number:', i)
    
    #votlt-source in current mode
    #volt_source.ramp(flux_points[i], 10)
    
    #votlt-source in volage mode
    volt_source.ramp(flux_points[i]*R, 10)
    print('Flux:', flux_points[i])
    
    time.sleep(1)
    
    #make timestamp
    timestamp.append(time.time())
        
#        Vappl_list.append(VSIM[i]/V_div)
        
    #measure temperature
    response = BFTCont.get_mxc_temperature()
    temp_info_list.append(response)
    print('MXC temperature:', response*1e3, 'mK')
    
    # #Rabi measurements
    # my_results_rabi = my_session.run(exp_rabi_comp)
    # rabi_res = my_results_rabi.get_data("q0_rabi")
    # rabi_amp = my_results_rabi.get_axis("q0_rabi")[0]
            
    # rabi_list.append(rabi_res)
        
    #Population measurements
    #results_population_full = my_session.run(exp_population_full_comp)
    #for p in labels:
    #    if i== 0 :
    #        pop_full_results[p] = [results_population_full.get_data(handle=f'{p}_readout')]
    #    else:
    #        pop_full_results[p].append(results_population_full.get_data(handle=f'{p}_readout'))
    
    # # #T1 measurements and fit
    #my_results_t1 = my_session.run(exp_t1_comp)
    #t1_res = my_results_t1.get_data("q0_t1")
    #t1_delay = my_results_t1.get_axis("q0_t1")[0]

    #t1_list.append(t1_res)
        
    #Ramsey measurements and fit
    #my_results_ramsey = my_session.run(exp_ramsey_comp)
    my_results_ramsey = my_session.run(exp_ramsey_comp)
    ramsey_res = my_results_ramsey.get_data("q0_ramsey")
    ramsey_delay = my_results_ramsey.get_axis("q0_ramsey")[0]
    
    ramsey_list.append(ramsey_res)


    #     #Echo measurements
    #my_results_echo = my_session.run(exp_echo_comp)
    #echo_res = my_results_echo.get_data("q0_echo")
    #echo_delay = my_results_echo.get_axis("q0_echo")[0]
       
    #echo_list.append(echo_res)

#set coil current back to the sweet spot
volt_source.ramp(qubit_parameters['flux_point']*R, 10)

In [ ]:
# #Convert lists with data to arrays
temp_info_array = np.array(temp_info_list)

# rabi_arr = np.array(rabi_list)

#t1_arr = np.array(t1_list)
#t1_time_arr = t1_delay

#for k in labels:
#    pop_full_results[k] = np.array(pop_full_results[k])

#t1_time_arr = t1_delay

ramsey_arr = np.array(ramsey_list)
ramsey_time_arr = ramsey_delay 

#echo_arr = np.array(echo_list)

timestamp_arr = np.array(timestamp)

In [ ]:
popt_ramsey_arr, pcov_ramsey_arr = auto_ramsey_fit(ramsey_delay, ramsey_arr, data_type = 'rot', plot = True)

# popt_t1_arr, pcov_t1_arr = auto_T1_fit(t1_delay, t1_arr, data_type = 'phase', plot = False)

#popt_echo_arr, pcov_echo_arr = auto_echo_fit(echo_delay, echo_arr, data_type = 'phase', plot = False)

#popt_echo_arr, pcov_echo_arr = auto_echo_fit(echo_delay, echo_arr_fliped, data_type = 'phase', plot = False)

In [ ]:
plt.plot(flux_points*1e3, 1/popt_echo_arr[:,0]*1e6, 'o-')
#plt.plot(popt_ramsey_arr[:,0]/np.pi/2*1e-6)
plt.ylabel('T2 Echo, mks')
plt.xlabel('Coil current, mA')
plt.ylim(0, 6)

In [ ]:
plt.plot(flux_points*1e3, 1/popt_t1_arr[:,0]*1e6, 'o-')
#plt.plot(popt_ramsey_arr[:,0]/np.pi/2*1e-6)
plt.ylabel('T1, mks')
plt.xlabel('Coil current, mA')

In [ ]:
#Detuning vs. flux point
popt, pcov = opt.curve_fit(func_parabola, flux_points[:]*1e3, popt_ramsey_arr[:,0]/np.pi/2*1e-6)
plt.plot(flux_points[:]*1e3, popt_ramsey_arr[:,0]/np.pi/2*1e-6, 'o-')
plt.plot(flux_points*1e3, func_parabola(flux_points*1e3, *popt))
#plt.plot(popt_ramsey_arr[:,0]/np.pi/2*1e-6)
plt.ylabel('Detuning, MHz')
plt.xlabel('Coil current, mA')

#save figure
figname = 'Flux_sweep_detuning_'
file_path = get_path_to_file(figname, '.png', sample_parameters)
plt.savefig(file_path, dpi=600, format='png', metadata=None,
        bbox_inches=None, pad_inches=0.1,
        facecolor='auto', edgecolor='auto',
        backend=None,
       )

In [ ]:
plt.plot(flux_points*1e3, 1/popt_ramsey_arr[:,2]*1e6, 'o-')
plt.ylabel('T2 Ramsey, mks')
plt.xlabel('Coil current, mA')

In [ ]:
proj_I, proj_Q = make_projection(pop_full_results, 0)
    
ABC_I = get_ABC(proj_I)
ABC_Q = get_ABC(proj_Q)

TABC_I = get_temperature(ABC_I, f_q, qubit_parameters['qb_anharm'], three_levels = True)
TABC_Q = get_temperature(ABC_Q, f_q, qubit_parameters['qb_anharm'], three_levels = True)

In [ ]:
fig, ax = plt.subplots(2, 1, sharex = True, figsize = (10, 8))

# fig.suptitle('Relative errors for effective temperature vs. phase rotation', fontsize=16)

fig.supxlabel('Coil_current, mA')
fig.supylabel('Effective temperature, mK')

for k in TABC_I.keys():
    if k[2] != '3':
        ax[0].plot(flux_points*1e3, TABC_I[k], label = k)
ax[0].legend()

for k in TABC_Q.keys():
    if k[2] != '3':
        ax[1].plot(flux_points*1e3, TABC_Q[k], label = k)

# for k in TABC_I.keys():
#     if k[2] != '3':
#         plt.plot(TABC_I[k], label = k)
ax[1].legend()
# #plt.ylim(0,200)

In [ ]:
fig, ax = plt.subplots(2, 1, sharex = True, figsize = (10, 8))

# fig.suptitle('Relative errors for effective temperature vs. phase rotation', fontsize=16)

fig.supxlabel('Detuning, MHz')
fig.supylabel('Effective temperature, mK')

for k in TABC_I.keys():
    if k[2] != '3':
        ax[0].plot(popt_ramsey_arr[:,0]/np.pi/2*1e-6, TABC_I[k], label = k)
ax[0].legend()

for k in TABC_Q.keys():
    if k[2] != '3':
        ax[1].plot(popt_ramsey_arr[:,0]/np.pi/2*1e-6, TABC_Q[k], label = k)

# for k in TABC_I.keys():
#     if k[2] != '3':
#         plt.plot(TABC_I[k], label = k)
ax[1].legend()
# #plt.ylim(0,200)

In [ ]:
flux_max = -popt[1]/popt[0]/2
#flux_max = -0.075
print('Minimal detuning at flux: ', flux_max , 'mA')
print('Voltage on flux source: ', round(flux_max*R*1e-3, 4))

In [ ]:
qubit_parameters.update_parameter('flux_point', flux_max*1e-3)

In [ ]:
#Save the flux sweep data
Data_flux_sweep = {'timestamp': timestamp_arr,
                   # 'rabi': rabi_arr,
                   # 'rabi_amp': rabi_amp,
                   # 'T1': t1_arr,  
                   # 't1_time_arr': t1_time_arr, 
                   'Ramsey': ramsey_arr, 
                   'ramsey_time': ramsey_time_arr,
                   #  'echo_time': echo_delay,
                   # 'Echo': echo_arr,
                   'MXC_temp': temp_info_array,
                   'flux_points': flux_points
                  }

#Data_flux_sweep.update(pop_full_results)
Data_flux_sweep.update(qubit_parameters)
Data_flux_sweep.update(lo_settings)

file_name = 'Flux_sweep_20mK_'#+'_switch'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_flux_sweep)

In [ ]:
#qubit_parameters['flux_upper_ss'] = flux_max*1e-3
#qubit_parameters['flux_lower_ss'] = 0.54e-3
volt_source.ramp(qubit_parameters['flux_point']*R, 10)

In [ ]:
qubit_parameters['flux_point']

In [ ]:
#volt_source.ramp(0.78*4, 10)

## Flux sweep and recalibration

In [ ]:
qb_param_flux = {}

#Set flux for lower and upper ss
qb_param_flux['flux_period'] = 0.068*1e-3
qb_param_flux['flux_lower_ss'] = qubit_parameters['flux_lower_ss']
qb_param_flux['flux_upper_ss'] = qubit_parameters['flux_lower_ss']+qb_param_flux['flux_period']/2

#Set qubit freq for lower and upper ss
qb_param_flux['freq_lower_ss'] = qubit_parameters['qb_freq']
qb_param_flux['lo_lower_ss'] = lo_settings['qb_lo']

qb_param_flux['freq_upper_ss'] = 350*1e6
qb_param_flux['lo_upper_ss'] = 6*1e9

qb_param_flux['lo_step'] = 0.5*1e9

#Set readout freq for lower and upper ss
qb_param_flux['ro_lower_ss'] = qubit_parameters['ro_freq']
qb_param_flux['ro_upper_ss'] = qubit_parameters['ro_freq']

qb_param_flux['ro_lo'] = lo_settings['ro_lo']

In [ ]:
qb_param_flux['flux_lower_ss']

In [ ]:
qb_param_flux['flux_upper_ss']

In [ ]:
qb_param_flux['global_flux_sweep']

In [ ]:
array([-1.86607223e-05, -1.55698132e-05, -1.24789041e-05, -9.38799502e-06,
       -6.29708593e-06, -3.20617684e-06, -1.15267746e-07,  2.97564135e-06,
        6.06655044e-06,  9.15745953e-06,  1.22483686e-05,  1.53392777e-05])

In [ ]:
#Set params for the global flux sweep

qb_param_flux['flux_sweep_points'] = 32
qb_param_flux['global_flux_sweep'] = np.linspace(qb_param_flux['flux_lower_ss'], qb_param_flux['flux_upper_ss'], qb_param_flux['flux_sweep_points'])

In [ ]:
def get_freq_preset(qb_param_flux):
    fmax =  qb_param_flux['freq_upper_ss'] + qb_param_flux['lo_upper_ss']
    fmin =  qb_param_flux['freq_lower_ss'] + qb_param_flux['lo_lower_ss']
    amp = (fmax-fmin)/2
    off = (fmax+fmin)/2

    omega = 2*np.pi/qb_param_flux['flux_period']
    phase = (qb_param_flux['flux_upper_ss']-qb_param_flux['flux_lower_ss'])/2

    x = qb_param_flux['global_flux_sweep']

    return amp*np.cos(omega*(x-phase))+off

def split_freq(freq):
    base = (freq*1e-8)//5
    lo = base*0.5*1e9
    fr = freq-lo
    return lo, fr

In [ ]:
#qb_param_flux['freq_est'] = get_freq_preset(qb_param_flux)

qb_param_flux['freq_est'] = func_osc(qb_param_flux['global_flux_sweep']*1e3, *popt_flux)*1e9
#qb_param_flux['freq_est'] = qb_freq

plt.plot(qb_param_flux['global_flux_sweep'], qb_param_flux['freq_est'])

In [ ]:
qb_param_flux['lo_est'], qb_param_flux['qb_freq_est'] = split_freq(qb_param_flux['freq_est'])
print(qb_param_flux['lo_est']*1e-9)

In [ ]:
qb_param_flux['ro_freq'] = []
qb_param_flux['qb_freq'] = []
qb_param_flux['rabi_popt'] = []
qb_param_flux['rabi_pcov'] = []
#corr = [0.0]

init_flux = volt_source.get_level(only_number = True)
init_lo = lsg_q0["drive_line"].local_oscillator.frequency

for i in range(qb_param_flux['flux_sweep_points']):
    print('Iteration:', i)
    #Set flux
    volt_source.ramp(qb_param_flux['global_flux_sweep'][i]*R, 10)
    print('Flux:', qb_param_flux['global_flux_sweep'][i])

    #Readout resonator spectroscopy
    ro_freq = recalibration_STS(qb_param_flux['ro_lower_ss'])
    qb_param_flux['ro_freq'].append(ro_freq)

    #Qubit pulsed spectroscopy
    lsg_q0["drive_line"].local_oscillator.frequency = qb_param_flux['lo_est'][i]

    readout_spec_sweep = {'readout_type': 'spec_sweep',
                        'readout_pulse': readout_spec['readout_pulse'],
                        'readout_weighting_function': readout_spec['readout_weighting_function'],
                        'relax_time': qubit_parameters["relax"],
                        'measure_freq': ro_freq,
                        'acquire_freq': ro_freq,
                        'readout_range': -25,
                        'readout_delay': qubit_parameters['ro_int_delay']
                       }
    #With correction
    #qfreq, qspec_res, qspec_freq = recalibration_TTS(qb_param_flux['qb_freq_est'][i]+corr[i], readout_spec_sweep)

    #No correction
    qfreq, qspec_res, qspec_freq = recalibration_TTS(qb_param_flux['qb_freq_est'][i], readout_spec_sweep)

    #corr.append(qfreq-qb_param_flux['qb_freq_est'][i])

    qb_param_flux['qb_freq'].append(qfreq)
    print('Qubit frequency:', (qfreq+qb_param_flux['lo_est'][i])*1e-9, 'GHz')

    readout_low_sw = {'readout_type': 'pulse',
                      'readout_pulse': readout_pulse,
                      'readout_weighting_function': readout_weighting_function,
                      'relax_time': qubit_parameters["relax"],
                      'measure_freq': ro_freq,
                      'acquire_freq': ro_freq,
                      'readout_range': -25,
                      'readout_delay': qubit_parameters['ro_int_delay']
                      }

    popt_norm, pcov_norm, signal = recalibration_Rabi(qb_param_flux['qb_freq'][i], readout_low_sw)
    qb_param_flux['rabi_popt'].append(popt_norm)
    qb_param_flux['rabi_pcov'].append(pcov_norm)

#Return to the initial settings
volt_source.ramp(init_flux , 10)
lsg_q0["drive_line"].local_oscillator.frequency = init_lo

In [ ]:
file_name = 'Global_flux_sweep_20mK_'#+'_switch'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, qb_param_flux)

In [ ]:
qb_param_flux['global_flux_sweep'].shape

In [ ]:
np.array(corr)*1e-6

In [ ]:
qb_freq_arr = np.array(qb_param_flux['qb_freq']) + qb_param_flux['lo_est']

In [ ]:
qb_param_flux['qb_freq']

In [ ]:
plt.plot(qb_param_flux['global_flux_sweep']*1e3, qb_freq_arr*1e-9, ".k")

In [ ]:
rabi_popt_arr = np.array(qb_param_flux['rabi_popt'])
#np.pi/popt_amp[0]

In [ ]:
rabi_popt_arr.shape

In [ ]:
plt.plot(qb_param_flux['global_flux_sweep']*1e3, np.pi/rabi_popt_arr[:,0], ".k")
plt.ylim(0,1.0)

In [ ]:
# x = qb_param_flux['global_flux_sweep'][0:5]*1e3
# y = qb_freq[0:5]*1e-9

x = qb_param_flux['global_flux_sweep']*1e3
y = qb_freq_arr*1e-9

x_ext = np.linspace(x[0], qb_param_flux['global_flux_sweep'][-1]*1e3, 50)

#plt.plot(qb_param_flux['global_flux_sweep'][0:5], qb_freq[0:5])

popt_flux2, pcov_flux2 = fit_Rabi(x, y, 0.01/qb_param_flux['flux_period'], -np.pi/2, amp = 1.0, off = 5.0)
plt.plot(qb_param_flux['global_flux_sweep']*1e3, qb_freq_arr*1e-9, ".k")
plt.plot(x_ext, func_osc(x_ext, *popt_flux2), "-r")
print(popt_flux)

#plt.plot(qb_param_flux['global_flux_sweep']*1e3, qb_param_flux['freq_est']*1e-9, 'o')

In [ ]:
qb_param_flux['qb_freq_est']*1e-6

In [ ]:
qb_param_flux['freq_est']*1e-6

In [ ]:
qb_param_flux['qb_freq']+qb_param_flux['lo_est'][0:5]

In [ ]:
lsg_q0["drive_line"].local_oscillator.frequency = lo_settings['qb_lo']

In [ ]:
readout_spec_sweep = {'readout_type': 'spec_sweep',
                      'readout_pulse': readout_spec['readout_pulse'],
                      'readout_weighting_function': readout_spec['readout_weighting_function'],
                      'relax_time': qubit_parameters["relax"],
                      'measure_freq': qubit_parameters['ro_freq'],
                      'acquire_freq': qubit_parameters['ro_freq'],
                      'readout_range': -25,
                      'readout_delay': qubit_parameters['ro_int_delay']
                     }

In [ ]:
def recalibration_STS(ro_freq_init):
    #STS
    spec_range = 6e6
    spec_num = 201
    n_average = 10

    # define sweep parameters for two qubits
    freq_sweep_ST = LinearSweepParameter(
        uid="res_freq",
        start=ro_freq_init - spec_range / 2,
        stop=ro_freq_init + spec_range / 2,
        count=spec_num,
    )

    # define the experiment with the frequency sweep relevant for qubit 0
    exp_spec_rec = res_spectroscopy(freq_sweep_ST, readout_pulse_spec, n_average)

    # set signal calibration and signal map for experiment to qubit 0
    exp_spec_rec.set_calibration(res_spec_calib(freq_sweep_ST))
    exp_spec_rec.set_signal_map(MA_map)

    exp_spec_rec_comp = my_session.compile(exp_spec_rec)
    ST_results = my_session.run(exp_spec_rec_comp)

    spec_res = ST_results.get_data("res_spec")
    spec_freq = lo_settings["ro_lo"] + ST_results.get_axis("res_spec")[0]

    # fit to asymmetric Lorentzian model 
    full_result, quick_result, sweep_manager = fit_single_STS_wrapper(freqs=spec_freq, y_data=spec_res)

    ro_freq_rec = quick_result['f0'] - lo_settings["ro_lo"]
    print('RO Frequency:', ro_freq_rec*1e-6, 'MHz')

    return ro_freq_rec

In [ ]:
def recalibration_TTS(q_freq_init, readout_spec):
    qspec_num = 401
    n_average = 12

    lo_freq = lsg_q0["drive_line"].local_oscillator.frequency

    freq_sweep_TT = LinearSweepParameter(
        uid="freq-qubit",
        start=q_freq_init - 200e6,
        stop=q_freq_init +  200e6,
        count=qspec_num,
    )

    square_pulse = pulse_library.gaussian(
        uid="const_gauss",
        length=qubit_parameters["qb_len_spec"],
        amplitude=qubit_parameters["qb_amp_spec"]/2,
    )

    #DONT FORGET TO CHANGE RO FREQ!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    exp_qspec_rec = create_qubit_spec(freq_sweep_TT, square_pulse, readout_spec, n_average)
    exp_qspec_rec_comp = my_session.compile(exp_qspec_rec)
    qspec_rec_results = my_session.run(exp_qspec_rec_comp)

    # get measurement data returned by the instruments
    qspec_res = qspec_rec_results.get_data("q0_spec")
    qspec_freq = lo_freq + qspec_rec_results.get_axis("q0_spec")[0]

    #fit the data
    try:
        pha = np.unwrap(np.angle(qspec_res))
        PM = np.argmax(abs(pha-np.mean(pha)))
        
        popt_pha, pcov_pha = fit_3DSpec(
            qspec_freq,
            #abs(qspec_res),
            pha,
            5e5,
            qspec_freq[PM],
            5.0e3,
            off=np.mean(pha),
            plot=True,
        )
        qfreq_pha = popt_pha[1] - lo_freq
    except:
        print('TTS phase fit didnt converged!')

    try:
        amp = abs(qspec_res)
        AM = np.argmax(abs(amp-np.mean(amp)))
        
        popt_amp, pcov_amp = fit_3DSpec(
            qspec_freq,
            amp,
            #np.unwrap(np.angle(qspec_res)),
            5e5,
            qspec_freq[AM],
            5.0e3,
            off=np.mean(amp),
            plot=True,
        )

        qfreq_amp = popt_amp[1] - lo_freq
    except:
        print('TTS amplitude fit didnt converged!')

    if 'qfreq_pha' in locals():
        if 'qfreq_amp' in locals():
            if abs(qfreq_pha-qfreq_amp)<10e6:
                qfreq = qfreq_pha
                print('Qubit Frequency:', qfreq*1e-6, 'MHz')
            else:
                qfreq = qfreq_pha
                print('Qubit Frequency is unreliabl!:', qfreq*1e-6, 'MHz')
        else:
            qfreq = qfreq_pha
            print('Qubit Frequency from pha:', qfreq*1e-6, 'MHz')
    else:
        if 'qfreq_amp' in locals():
            qfreq = qfreq_amp
            print('Qubit Frequency from amp:', qfreq*1e-6, 'MHz')
        else:
            qfreq = q_freq_init
            print('Cannot get frequency!')
    

    return qfreq, qspec_res, qspec_freq

In [ ]:
def recalibration_Rabi(q_freq, readout_low_sw):
    drive_Oscillator_q0.frequency = q_freq

    n_average = 13

    # Rabi excitation pulse - gaussian of unit amplitude - amplitude will be scaled with sweep parameter in experiment
    gaussian_pulse = pulse_library.gaussian(
        uid="gaussian_pulse", length=qubit_parameters["qb_len"], amplitude=1.0
    )
    
    exp_rabi_sw = create_rabi(rabi_sweep, 
                              gaussian_pulse, 
                              readout_low_sw, 
                              n_average)

    exp_rabi_sw_comp = my_session.compile(exp_rabi_sw)
    my_results = my_session.run(exp_rabi_sw_comp)

    # get measurement data returned by the instruments
    rabi_res = my_results.get_data("q0_rabi")

    # define amplitude axis from qubit parameters
    rabi_amp = my_results.get_axis("q0_rabi")[0]

    signal = rabi_res

    I_zero = np.real(signal)[0]
    Q_zero = np.imag(signal)[0]

    new_signal = np.sqrt((np.real(signal) - I_zero)**2 + (np.imag(signal) - Q_zero)**2)

    fx, fy = FFT_analize(rabi_amp, new_signal, plot = False)
    opt_f = np.argmax(abs(fy))
    try:
        popt_real, pcov_real = fit_Rabi(rabi_amp, np.real(signal), 2*np.pi*fx[opt_f], 0, 1, np.mean(np.real(signal)), plot=False)
        popt_imag, pcov_imag = fit_Rabi(rabi_amp, np.imag(signal), 2*np.pi*fx[opt_f], 0, 1, np.mean(np.imag(signal)), plot=False)
        popt_norm, pcov_norm = fit_Rabi(rabi_amp, new_signal, 2*np.pi*fx[opt_f], 0, 1, np.mean(new_signal), plot=True)
    except:
        print('Rabi fit didnt converged!')
        drive_Oscillator_q0.frequency = qubit_parameters["qb_freq"]
        return np.zeros((4,)), np.zeros((4,4)), signal
    
    drive_Oscillator_q0.frequency = qubit_parameters["qb_freq"]

    return popt_norm, pcov_norm, signal

In [ ]:
def recalibration_T1(q_freq, x180, readout_low_sw, proj = 'pha'):
    drive_Oscillator_q0.frequency = q_freq
    # delay range for ramsey pulses
    t1_min = 0.0
    t1_max = 60e-6 #5 * qubit_parameters["qb_t1"]
    # how many delay points to sweep
    t1_num = 201

    # set up sweep parameter
    t1_sweep = LinearSweepParameter(uid="t1_delay", start=t1_min, stop=t1_max, count=t1_num)

    # how many averages per point: 2^n_average
    n_average = 12

    exp_t1_sw = create_t1(t1_sweep, 
                   x180, 
                   readout_low_sw,
                   n_average)

    exp_t1_sw_comp = my_session.compile(exp_t1_sw)

    t1_results = my_session.run(exp_t1_sw_comp)

    # get measurement data returned by the instruments
    t1_res = t1_results.get_data("q0_t1")
    t1_res_amp = abs(t1_res)
    t1_res_pha = np.unwrap(np.angle(t1_res))

    t1_delay = t1_results.get_axis("q0_t1")[0]
    # fit measurement results to decaying exponential curve
    if proj == 'pha':
        popt_t1, pcov_t1 = fit_T1(x = t1_delay, y = t1_res_pha, rate = 2e6, off = np.mean(t1_res_pha), amp = abs(np.max(t1_res_pha)-np.min(t1_res_pha)), plot=False)
    elif proj=='amp':
        popt_t1, pcov_t1 = fit_T1(x = t1_delay, y = t1_res_amp, rate = 2e6, off = np.mean(t1_res_amp), amp = abs(np.max(t1_res_amp)-np.min(t1_res_amp)), plot=False)

    #qubit_parameters["qb_t1"] = 1 / popt[0]
    drive_Oscillator_q0.frequency = qubit_parameters["qb_freq"]
    
    return popt_t1, pcov_t1, t1_res

In [ ]:
def check_errors(popt, pcov):
    err = np.sqrt(np.diag(pcov))
    rel_err = err/popt

    if np.max(rel_err)<1:
        return True
    else:
        return False

In [ ]:
if check_errors(popt, pcov):
    print('Good fit')
else:
    print('Bad fit')

In [ ]:
np.zeros((4,))

In [ ]:
#Test for Rabi recalibration
i = 1

init_lo = lsg_q0["drive_line"].local_oscillator.frequency
init_flux = volt_source.get_level(only_number = True)



volt_source.ramp(qb_param_flux['global_flux_sweep'][i]*R, 10)
print('Flux:', qb_param_flux['global_flux_sweep'][i])
lsg_q0["drive_line"].local_oscillator.frequency = qb_param_flux['lo_est'][i]

readout_low_sw = {'readout_type': 'pulse',
                  'readout_pulse': readout_pulse,
                  'readout_weighting_function': readout_weighting_function,
                  'relax_time': qubit_parameters["relax"],
                  'measure_freq': qb_param_flux['ro_freq'][i],
                  'acquire_freq': qb_param_flux['ro_freq'][i],
                  'readout_range': -25,
                  'readout_delay': qubit_parameters['ro_int_delay']
                  }

popt_norm, pcov_norm, signal = recalibration_Rabi(qb_param_flux['qb_freq'][i], readout_low_sw)

volt_source.ramp(init_flux, 10)
lsg_q0["drive_line"].local_oscillator.frequency = init_lo

In [ ]:
I_zero = np.real(signal)[0]
Q_zero = np.imag(signal)[0]

new_signal = np.sqrt((np.real(signal) - I_zero)**2 + (np.imag(signal) - Q_zero)**2)

fx, fy = FFT_analize(rabi_amp, new_signal, plot = True)
opt_f = np.argmax(abs(fy))

In [ ]:
plt.plot(rabi_amp, func_osc(rabi_amp, 2*np.pi*fx[opt_f], 0, 1, 0))

In [ ]:
np.pi/popt_norm[0]

In [ ]:
def testfunc(val):
    if val == 0:
        return 1
    else:
        return val

In [ ]:
testfunc(1)

In [ ]:
readout_pulse

In [ ]:
qubit_parameters["qb_freq"]*1e-6

In [ ]:
test, qspec_res, qspec_freq = recalibration_TTS(qubit_parameters["qb_freq"])

In [ ]:
pha = np.unwrap(np.angle(qspec_res))
PM = np.argmax(abs(pha-np.mean(pha)))

popt_pha, pcov_pha = fit_3DSpec(
            qspec_freq,
            #abs(qspec_res),
            pha,
            10e5,
            qspec_freq[PM],
            5.0e3,
            off=np.mean(pha),
            plot=True,
        )

In [ ]:
qspec_freq[PM]*1e-9

In [ ]:
amp = abs(qspec_res)
AM = np.argmax(abs(amp-np.mean(amp)))

popt_amp, pcov_amp = fit_3DSpec(
            qspec_freq,
            amp,
            10e5,
            qspec_freq[AM],
            5.0e3,
            off=np.mean(amp),
            plot=True,
        )

In [ ]:
print(popt_amp)
print(popt_pha)

In [ ]:
pha = np.unwrap(np.angle(qspec_res))
AM = np.argmax(abs(pha-np.mean(pha)))
print(qspec_freq[AM])

## Local heating

In [ ]:
V_off = -1.335

a = np.linspace(0, 0.8, 20)
b = np.linspace(0.85, 2.1, 25)


V_sh_arr = np.round(np.concatenate([a,b]),3)

print(V_sh_arr)
print(len(V_sh_arr))

In [ ]:
VSIM = V_off+V_sh_arr
print(VSIM)

In [ ]:
#Simple linear spased voltage sweep
VSIM = np.round(np.linspace(-2.2, -1.8, 251),3)
print(VSIM)

In [ ]:
a = np.linspace(-3.2, -2.1, 10)
b = np.linspace(-2.0, -0.6, 47)
c = np.linspace(-0.5, 0.8, 10)

#print(a)
d = np.concatenate([a,b,c])
#e = np.concatenate([d,b])

res = np.round(d,3)

print(res)
#d = np.linspace(-1.0, 0, 3)

In [ ]:
#More complicated voltage sweep
a = np.linspace(0, 1.0, 3)
b = np.linspace(1.5, 4.0, 10)
c = np.concatenate([a,b])

d = np.linspace(-1.0, 0, 3)
e = np.linspace(-4.0, -1.5, 10)
f = np.concatenate([e,d])

f = np.delete(f, -1)

res = np.round(np.concatenate([f,c]),3)

print(res)
print(len(res))

In [ ]:
VSIM = res
print(VSIM)

In [ ]:
#lists for data storage

pop_full_results = {}
labels = ['x0', 'x1', 'x2', 'y0', 'y1', 'y2']

# qspec_res_list = []

t1_list = []

ramsey_list = []


#echo_list = []

temp_info_list = []
dmm_list = []

timestamp = []

Vappl_list = [] #VSIM/V_div

wait_time_2 = 10

In [ ]:
sim.set_output(1)

In [ ]:
sim.get_level(only_number = True)

In [ ]:
sim.ramp(-1.335, N_steps = 10)

In [ ]:
print(DMM.scan())

In [ ]:
#Main sweep loop

#V_off+V_sh_arr

V_sh_arr = VSIM

inner_loop = 1

for i in range(len(V_sh_arr)):
    print('Point',i+1,'out of',len(V_sh_arr))
    
    for j in range(inner_loop):
        print(f'Inner Loop: Point {j+1} out of {inner_loop}')

        #if j == 0:
        #    V_set = V_off-V_sh_arr[i]
        #elif j == 1:
        #    V_set = V_off
        #elif j == 2:
        #    V_set = V_off+V_sh_arr[i]

        V_set = VSIM[i]
        print('Voltage on SIM:', V_set)
        
        k=True
        
        while k:
            try:
                sim.get_level(only_number = True)
                sim.ramp(V_set, N_steps = 1e2)
                k=False
            except:
                print('Could not set voltage!')
                time.sleep(wait_time_2)
                
        time.sleep(wait_time_2)
        
        #make timestamp
        timestamp.append(time.time())
        
        Vappl_list.append(V_set/V_div)
        
        #measure temperature
        response = BFTCont.get_mxc_temperature()
        temp_info_list.append(response)
        print('MXC temperature:', response*1e3, 'mK')
        
        #Population measurements
        for n in range(1):
            results_population_full = my_session.run(exp_population_full_comp)
            for p in labels:
                if i== 0 and j == 0 and n==0:
                    pop_full_results[p] = [results_population_full.get_data(handle=f'{p}_readout')]
                else:
                    pop_full_results[p].append(results_population_full.get_data(handle=f'{p}_readout'))
        
        #Make the TTS
        #my_results_TTS = my_session.run(exp_qspec_comp)
        #TTS_res = my_results_TTS.get_data("q0_spec")
        
        #qspec_res = my_results_TTS.get_data("q0_spec")
        #qspec_freq = lo_settings["qb_lo"] + my_results_TTS.get_axis("q0_spec")[0]
        
        #qspec_res_list.append(qspec_res)
    
        #T1 measurements and fit
        my_results_t1 = my_session.run(exp_t1_comp)
        t1_res = my_results_t1.get_data("q0_t1")
        t1_delay = my_results_t1.get_axis("q0_t1")[0]

        t1_list.append(t1_res)
        
        # # #Ramsey measurements and fit
        my_results_ramsey = my_session.run(exp_ramsey_comp)
        ramsey_res = my_results_ramsey.get_data("q0_ramsey")
        ramsey_delay = my_results_ramsey.get_axis("q0_ramsey")[0]
    
        ramsey_list.append(ramsey_res)

        # # #Echo measurements
        #my_results_echo = my_session.run(exp_echo_comp)
        #echo_res = my_results_echo.get_data("q0_echo")
        #echo_delay = my_results_echo.get_axis("q0_echo")[0]
       
        #echo_list.append(echo_res)

        #NIS measurements
        V_meas = DMM.scan()
        dmm_list.append(float(V_meas))
        print(f'Measured voltage: {V_meas}')
        
sim.get_level(only_number = True)
sim.ramp(V_off, N_steps = 1e2)

In [ ]:
temp_info_array = np.array(temp_info_list)

for k in labels:
    pop_full_results[k] = np.array(pop_full_results[k])

t1_arr = np.array(t1_list)
t1_time_arr = t1_delay

ramsey_arr = np.array(ramsey_list)
ramsey_time_arr = ramsey_delay 

#echo_arr = np.array(echo_list)

# #qspec_res_arr = np.array(qspec_res_list)
# #qspec_freq_arr = qspec_freq

# rabi_arr = np.array(rabi_list)

Vappl = np.array(Vappl_list)

timestamp_arr = np.array(timestamp)

SINIS_volt = np.array(dmm_list)

In [ ]:
Data_local_heat = {#'Redout_1': pop_r1_arr,
                #'Redout_2': pop_r2_arr,
                'timestamp': timestamp_arr,
                'T1': t1_arr, 
                't1_time_arr': t1_time_arr,
                'Ramsey': ramsey_arr, 
                'ramsey_time': ramsey_time_arr,
                #'echo_time': echo_delay,
                #'Echo': echo_arr,
                # 'rabi': rabi_arr,
                # 'rabi_amp': rabi_amp,
                #'qspec_res': qspec_res_arr,
                #'qspec_freq': qspec_freq_arr,
                'V_sinis': SINIS_volt,
                'Vheat_apppl': Vappl,
                'Vheat_SIM': VSIM,
                'MXC_temp': temp_info_array,
                'comment': 'Fast temperature sweep 1000 div 50pA current source, SINIS1'
               }
Data_local_heat.update(pop_full_results)
Data_local_heat.update(qubit_parameters._params)
Data_local_heat.update(lo_settings._params)

#file_name = 'R4_temp_sweep_and_calib_NIS2_'
file_name = 'remote_heating_3points_t1_pop_ramsey_SINIS1_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_local_heat)

In [ ]:
#SINIS voltage vs. heating

SINIS_volt = np.array(dmm_list)
#T_mxc = np.array(temp_info_list)[:,0]
plt.plot(Vappl*1e3, SINIS_volt, 'o-')
plt.xlabel('VSIM')
plt.ylabel('V_SINIS')

In [ ]:
len(dmm_list)

In [ ]:
Vappl

In [ ]:
#SINIS_volt = np.array(dmm_list)
#T_mxc = np.array(temp_info_list)[:,0]
plt.plot(temp_info_array, SINIS_volt, 'o-')
plt.xlabel('T_mxc')
plt.ylabel('V_SINIS')

In [ ]:
ABC = get_ABC_parallel(pop_full_results)
TABC = get_temperature(ABC, f_q, qubit_parameters['qb_anharm']*1e-9, three_levels = True)

#skip = ['TA3', 'TB3', 'TC3']
skip = []

fig, ax = plot_temp_single(TABC, skip = skip, xdata=Vappl*1e3, xlabel='V_Heater, corrected (mV)')

ax.vlines([-2.2, -1], 50, 100)

In [ ]:
plt.plot(Vappl[:]*1e3, TABC['TC1'])

In [ ]:
plt.plot(SINIS_volt, TABC['TC1'])

In [ ]:
popt_t1_arr, pcov_t1_arr = auto_T1_fit(t1_delay, t1_arr, data_type = 'rot', plot = False)

In [ ]:
plt.title('T1 measurements')
plt.plot(Vappl*1e3, 1/popt_t1_arr[:,0]*1e6, 'o-')
#plt.plot(popt_ramsey_arr[:,0]/np.pi/2*1e-6)
plt.ylabel('T1, mks')
plt.xlabel('V_Heater (mV)')
plt.vlines([-2.2], (1/popt_t1_arr[:,0]*1e6).min(), (1/popt_t1_arr[:,0]*1e6).max())

In [ ]:
plt.plot(SINIS_volt, popt_t1_arr[:,0]*1e-6, 'o')
plt.xlabel('V_SINIS')
plt.ylabel('gamma1, 1/mks')

In [ ]:
popt_ramsey_arr, pcov_ramsey_arr = auto_ramsey_fit(ramsey_delay, ramsey_arr, data_type = 'rot', plot = True)

# popt_t1_arr, pcov_t1_arr = auto_T1_fit(t1_delay, t1_arr, data_type = 'phase', plot = False)

#popt_echo_arr, pcov_echo_arr = auto_echo_fit(echo_delay, echo_arr, data_type = 'phase', plot = False)

#popt_echo_arr, pcov_echo_arr = auto_echo_fit(echo_delay, echo_arr_fliped, data_type = 'phase', plot = False)

In [ ]:
#Detuning vs. flux point
plt.plot(Vappl*1e3, popt_ramsey_arr[:,0]/np.pi/2*1e-6, 'o-')
#plt.plot(popt_ramsey_arr[:,0]/np.pi/2*1e-6)
plt.ylabel('Detuning, MHz')
plt.xlabel('V_heater, mV')
plt.vlines([-2.2, -2], (popt_ramsey_arr[:,0]/np.pi/2*1e-6).min(), (popt_ramsey_arr[:,0]/np.pi/2*1e-6).max())

## IV-curve

In [ ]:
sim.get_level(only_number = True)

In [ ]:
VSIM = np.linspace(-4.0, 4.0, 201)
print(VSIM)

In [ ]:
#temp_info_list = []
dmm_list = []
timestamp = []
Vappl_list = [] #VSIM/V_div

In [ ]:
for i in range(len(VSIM)):
        #set voltage on SIM
    #print('Voltage on SIM:', VSIM[i])
    
    sim.get_level(only_number = True)
    sim.ramp(VSIM[i], N_steps = 1e2)
    
    time.sleep(1)
    
    #measure temperature
    #response = BFTCont.get_latest_channel_temp(6)
    #temp_info_list.append(response)
    #print('MXC temperature:', response[0]*1e3, 'mK')
        
    #make timestamp
    timestamp.append(time.time())
        
    Vappl_list.append(VSIM[i]/V_div)
        
    V_meas = DMM.scan()
    dmm_list.append(float(V_meas))
    #print(f'Measured voltage: {V_meas}')
        
sim.get_level(only_number = True)
sim.ramp(0.0, N_steps = 1e2)

In [ ]:
Vappl = np.array(Vappl_list)
timestamp_arr = np.array(timestamp)

SINIS_volt = np.array(dmm_list)

In [ ]:
#SINIS_volt = np.array(dmm_list)
#T_mxc = np.array(temp_info_list)[:,0]
plt.plot(Vappl*1e3, SINIS_volt, 'o-')
plt.xlabel('Vappl, mV')
plt.ylabel('V_SINIS')

In [ ]:
#temp_info_array = np.array(temp_info_list)

Vappl = np.array(Vappl_list)

timestamp_arr = np.array(timestamp)

SINIS_volt = np.array(dmm_list)

In [ ]:
Data_IV = {'timestamp': timestamp_arr,
            'V_sinis': SINIS_volt,
            'Vheat_apppl': Vappl,
            'Vheat_SIM': VSIM
            #'MXC_temp': temp_info_array[:,0]
            }

Data_IV.update(qubit_parameters)
Data_IV.update(lo_settings)

file_name = 'IV_battary_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_local_heat)

## Temperature sweep

In [ ]:
#lists for data storage

#timestamps
timestamp = []

#temperature data
temp_info_list = []

#SINIS measurements
#dmm_list = []

#Three levels population
pop_full_results = {}
labels = ['x0', 'x1', 'x2', 'y0', 'y1', 'y2']

#Rabi population
#qrpm_full_results = []

#Qubit spectroscopy
#qspec_res_list = []

#T1 experiment
t1_list = []

#T1 for ef and ge
#t1_ef_list = []

#Single Shots
#SS_list = []

#Ramsey
ramsey_list = []

#Echo
echo_list = []

#Rabi
rabi_list = []

In [ ]:
from XLD_Server_Client import XLDMeasClient
from time import sleep

from XLD_Server_Passkey import xld_ip

client = XLDMeasClient(user='Elias', group='PICO', server_ip=xld_ip, update_interval=2)
n_sweep, _ = client.open_session()

print(n_sweep)

for i in range(n_sweep):
    if client.listen():
        print("Starting measurement")
        
        for j in range(1):
            print('Iteration:', j)
               
            #make timestamp
            timestamp.append(time.time())
        
            #measure temperature
            response = BFTCont.get_latest_channel_temp(6)
            temp_info_list.append(response)
            print('MXC temperature:', response[0]*1e3, 'mK')
            
            #Rabi measurements
            my_results_rabi = my_session.run(exp_rabi_comp)
            rabi_res = my_results_rabi.get_data("q0_rabi")
            rabi_amp = my_results_rabi.get_axis("q0_rabi")[0]
            
            rabi_list.append(rabi_res)

            # #Population measurements
            # for k in range(32):
            #     print('Population Iteration:', k)

            #     results_population_full = my_session.run(exp_population_full_comp)
            #     for p in labels:
            #         if i== 0 and j==0 :
            #             pop_full_results[p] = [results_population_full.get_data(handle=f'{p}_readout')]
            #         else:
            #             pop_full_results[p].append(results_population_full.get_data(handle=f'{p}_readout'))
                    
            # # Quick Rabi Population Measurement
            # print('QRPM')
            # results_qrpm = my_session.run(rpm_quick_exp_comp)
            # rpm_res = {}
            # for key in ['with', 'wo']:
            #     rpm_res[key] = {'mag_min': np.abs(results_qrpm.get_data(f"rpm_{key}_min")),
            #                     'pha_min': np.angle(results_qrpm.get_data(f"rpm_{key}_min")),
            #                     'mag_max': np.abs(results_qrpm.get_data(f"rpm_{key}_max")),
            #                     'pha_max': np.angle(results_qrpm.get_data(f"rpm_{key}_max")),
            #                     'I_max': np.real(results_qrpm.get_data(f"rpm_{key}_max")),
            #                     'Q_max': np.imag(results_qrpm.get_data(f"rpm_{key}_max")),
            #                     'I_min': np.real(results_qrpm.get_data(f"rpm_{key}_min")),
            #                     'Q_min': np.imag(results_qrpm.get_data(f"rpm_{key}_min"))}

            # qrpm_full_results.append(rpm_res)
            # qrpm_dict = get_qrpm_dict_from_list_of_results(qrpm_full_results)
            
            # #Make the TTS
            # my_results_TTS = my_session.run(exp_qspec_comp)
        
            # qspec_res = my_results_TTS.get_data("q0_spec")
            # qspec_freq = lo_settings["qb_lo"] + my_results_TTS.get_axis("q0_spec")[0]
        
            # qspec_res_list.append(qspec_res)
        
            #T1 measurements
            my_results_t1 = my_session.run(exp_t1_comp)
            t1_res = my_results_t1.get_data("q0_t1")
            t1_delay = my_results_t1.get_axis("q0_t1")[0]
    
            t1_list.append(t1_res)

            #T1 on ef measurements
            #my_results_t1_ef = my_session.run(exp_T1_ef_comp)
            #t1_ef_res = my_results_t1_ef.get_data("q0_t1_ef")
            #t1_ef_delay = my_results_t1_ef.get_axis("q0_t1_ef")[0]
    
            #t1_ef_list.append(t1_ef_res)
        
            #Ramsey measurements
            my_results_ramsey = my_session.run(exp_ramsey_short_comp)
            ramsey_res = my_results_ramsey.get_data("q0_ramsey")
            ramsey_delay = my_results_ramsey.get_axis("q0_ramsey")[0]
            
            #if response[0]*1e3<130:
            #    print('Long Ramsey')
            #    my_results_ramsey = my_session.run(exp_ramsey_comp)
            #    ramsey_res = my_results_ramsey.get_data("q0_ramsey")
            #    ramsey_delay = my_results_ramsey.get_axis("q0_ramsey")[0]
            #else:
            #    print('Short Ramsey')
            #    my_results_ramsey = my_session.run(exp_ramsey_short_comp)
            #    ramsey_res = my_results_ramsey.get_data("q0_ramsey")
            #    ramsey_delay_short = my_results_ramsey.get_axis("q0_ramsey")[0]
    
            ramsey_list.append(ramsey_res)

            #Echo measurements
            my_results_echo = my_session.run(exp_echo_comp)
            echo_res = my_results_echo.get_data("q0_echo")
            echo_delay = my_results_echo.get_axis("q0_echo")[0]
       
            echo_list.append(echo_res)
    
            #Single shots
            #my_results_SS = my_session.run(exp_rabi_SS_comp)
            #SS_res = my_results_SS.get_data("SS_rabi")
            #SS_it = my_results_SS.get_axis("SS_rabi")[0]
            #SS_amp = my_results_SS.get_axis("SS_rabi")[1]
    
            #SS_list.append(SS_res)
        
            # #NIS measurements
            # V_meas = DMM.scan()
            # dmm_list.append(float(V_meas))
            # print(f'Measured voltage: {V_meas}')
            #Population measurements
            for k in range(32):
                print('Population Iteration:', k)

                results_population_full = my_session.run(exp_population_full_comp)
                for p in labels:
                    if i== 0 and j==0 :
                        pop_full_results[p] = [results_population_full.get_data(handle=f'{p}_readout')]
                    else:
                        pop_full_results[p].append(results_population_full.get_data(handle=f'{p}_readout'))
            
    sleep(5)
    print(f"done with step {i+1}")
    client.stopped()

print("client stopped fully")
sleep(10)
client.close_session()

In [ ]:
#timestamps
timestamp = []

#temperature data
temp_info_list = []

#T1 experiment
t1_all_list = []

#Ramsey
ramsey_list = []


n_average_pop = 15

flux_amp_sw = np.linspace(0.0, 0.7, 101)

edge = 1e-8

pop_flux_list = []

n_average_t1 = 12

In [ ]:
from XLD_Server_Client import XLDMeasClient
from time import sleep

from XLD_Server_Passkey import xld_ip

client = XLDMeasClient(user='Elias', group='PICO', server_ip=xld_ip, update_interval=2)
n_sweep, _ = client.open_session()

print(n_sweep)

for i in range(n_sweep):
    if client.listen():
        print("Starting measurement")
        
              
        #make timestamp
        timestamp.append(time.time())
        
        #1st measure temperature
        response = BFTCont.get_latest_channel_temp(6)
        temp_info_list.append(response)
        print('MXC temperature:', response[0]*1e3, 'mK')

        FF_decay_res_list = []
        pop_flux_results = {}

        for j in range(len(flux_amp_sw)):
            print('Iteration number:', j)
            print('Flux pulse amplitude:', flux_amp_sw[j])
            #population
            flux_pulse_sw = pulse_library.gaussian_square(
                uid="flux_pulse", 
                length=qubit_parameters['relax'], 
                width = qubit_parameters['relax']-2*edge, 
                amplitude=flux_amp_sw[j], 
                can_compress = True
            )

            exp_population_flux = create_exp_population_flux(x180, 
                                                             x180_ef, 
                                                             flux_pulse_sw,
                                                             readout_opt,# readout_low,
                                                             n_average_pop)

            #compile experiment
            exp_population_flux_comp = my_session.compile(exp_population_flux)
            results_population_flux = my_session.run(exp_population_flux_comp)

    
            for k in labels:
                if j == 0:
                    pop_flux_results[k] = [results_population_flux.get_data(handle=f'{k}_readout')]
                else:
                    pop_flux_results[k].append(results_population_flux.get_data(handle=f'{k}_readout'))

            #T1
            print('T1 measurements')
            flux_pulse_t1 = pulse_library.const(
                uid="flux_pulse", length=2.0e-6, amplitude=flux_amp_sw[j], can_compress = True
            )

            exp_FF_decay = make_fast_flux_decay(t1_ff_sweep, 
                                                flux_pulse_t1, 
                                                edge,
                                                x180, 
                                                readout_pulse, 
                                                readout_weighting_function, 
                                                qubit_parameters["relax"], 
                                                n_average_t1)

            exp_FF_decay.set_calibration(ff_decay_cal) #calibration from previous experiment
            exp_FF_decay.set_signal_map(qubit_resonator_map)

            exp_FF_decay_comp = my_session.compile(exp_FF_decay)

            FF_decay_results = my_session.run(exp_FF_decay_comp)

            FF_decay_res_list.append(FF_decay_results.get_data("q0_ff_decay"))
            FF_decay_delay = FF_decay_results.get_axis("q0_ff_decay")[0]

        FF_decay_res_arr = np.array(FF_decay_res_list)
        t1_all_list.append(FF_decay_res_arr)#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

        
        for k in labels:
            pop_flux_results[k] = np.array(pop_flux_results[k])

        pop_flux_list.append(pop_flux_results)#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        
        #Ramsey measurements
        my_results_ramsey = my_session.run(exp_ramsey_comp)
        ramsey_res = my_results_ramsey.get_data("q0_ramsey")
        ramsey_delay = my_results_ramsey.get_axis("q0_ramsey")[0]
               
        ramsey_list.append(ramsey_res)

        #2nd measure temperature
        response = BFTCont.get_latest_channel_temp(6)
        temp_info_list.append(response)
        print('MXC temperature:', response[0]*1e3, 'mK')
            
    sleep(5)
    print(f"done with step {i+1}")
    client.stopped()

print("client stopped fully")
sleep(10)
client.close_session()

In [ ]:
len(t1_all_list)

In [ ]:
t1_all_arr = np.array(t1_all_list)
print(t1_all_arr.shape)

In [ ]:
len(pop_flux_list)

In [ ]:
pop_flux_all_res = {}

for i in range(len(pop_flux_list)):
    if i == 0:
        for k in labels:
            pop_flux_all_res[k] = [pop_flux_list[i][k]]
    else:
        for k in labels:
            pop_flux_all_res[k].append(pop_flux_list[i][k])

for k in labels:
    pop_flux_all_res[k] = np.array(pop_flux_all_res[k])

In [ ]:
pop_flux_all_res['x0']

In [ ]:
temp_info_array = np.array(temp_info_list)

timestamp_arr = np.array(timestamp)

#t1_arr = np.array(t1_list)
#t1_time_arr = t1_delay

#t1_ef_arr = np.array(t1_ef_list)
#t1_ef_time_arr = t1_ef_delay

#rabi_arr = np.array(rabi_list)

ramsey_arr = np.array(ramsey_list)
ramsey_time_arr = ramsey_delay 

#SS_res_arr = np.array(SS_list)

#echo_arr = np.array(echo_list)

#for k in labels:
#    pop_full_results[k] = np.array(pop_full_results[k])


#qspec_res_arr = np.array(qspec_res_list)
#qspec_freq_arr = qspec_freq

#rabi_arr = np.array(rabi_list)

#SINIS_volt = np.array(dmm_list)

In [ ]:
Data_temp_sweep = {'timestamp': timestamp_arr,
                'T1_double_sweep': t1_all_arr, 
                't1_time_arr': FF_decay_delay,
                'flux_amp_sweep': flux_amp_sw,
                # 'T1_ef': t1_ef_arr,  
                # 't1_ef_time_arr': t1_ef_time_arr,
                'Ramsey': ramsey_arr, 
                'ramsey_time': ramsey_time_arr,
                # 'ramsey_time_short': ramsey_delay_short,
                #'echo_time': echo_delay,
                #'Echo': echo_arr,
                #'rabi': rabi_arr,
                #'rabi_amp': rabi_amp,
                # 'Single_Shot_res': SS_res_arr,
                # 'Single_Shot_iterations': SS_it,
                # 'Single_Shot_amp': SS_amp,
                'MXC_temp': temp_info_array[:,0],
                'comment': 'Temperature sweep'
               }
Data_temp_sweep.update(pop_flux_all_res)
#Data_temp_sweep.update(qrpm_dict)
Data_temp_sweep.update(qubit_parameters)
Data_temp_sweep.update(lo_settings)

file_name = 'Temperature_sweep_part1_'
file_path = get_path_to_file(file_name, '.mat', sample_parameters)
savemat(file_path, Data_temp_sweep)

In [ ]:
popt_ramsey_list = []
pcov_ramsey_list = []

for i in range(len(timestamp_arr)):
    print('Iteration:', i)
    ramsey_res = ramsey_list[i]
    try:
        popt_ramsey, pcov_ramsey = fit_Ramsey(
            ramsey_delay,
            np.unwrap(np.angle(ramsey_res)),
            #abs(ramsey_res),
            4e6, #qubit_parameters["ramsey_det"],
            0,
            2 / 6e-7,
            amp=0.1,
            off=1.9,
            plot=True,
            # bounds=[
            #     [0.1e6, -np.pi, 0.05 / 6e-7, 0.0001, -2],
            #     [50e6, np.pi, 100 / 6e-7, 2, 2],
            # ],
        )
        print('T2 = ', 1 / popt_ramsey[2])
        print('Detuning = ', popt_ramsey[0]/np.pi/2*1e-6, 'MHz')
    except:
        print('Pump T2 Fit didnt converged!')
        popt_ramsey = popt_0
        pcov_ramsey = pcov_0
    
    popt_ramsey_list.append(popt_ramsey)
    pcov_ramsey_list.append(pcov_ramsey)

popt_ramsey_arr = np.array(popt_ramsey_list)
pcov_ramsey_arr = np.array(pcov_ramsey_list)

In [ ]:
popt_t1_list = []
pcov_t1_list = []

timestamp = []

#Measurement loop
for i in range(len(timestamp_arr)):
    print('Iteration:', i)

    t1_res = t1_list[i]
    try:
        popt_t1, pcov_t1 = fit_T1(x = t1_delay, y = np.unwrap(np.angle(t1_res)), rate = 5e6, off = -1.5, amp = 0.1, plot=True)
        print('Pump T1 = ', 1/popt_t1[0])
    except:
        print('Pump T1 Fit didnt converged!')
        tr_popt_t1 = popt_0
        tr_pcov_t1 = pcov_0
    
    popt_t1_list.append(popt_t1)
    pcov_t1_list.append(pcov_t1)

popt_t1_arr = np.array(popt_t1_list)
pcov_t1_arr = np.array(pcov_t1_list)

In [ ]:
qubit_parameters['ro_freq']*1e-6